# Pix2Pix CMP Facades: dari nol sampai training

Proyek kelompok **Vibe Coding & Generative AI from Scratch**: Pix2Pix (Generator U-Net + Discriminator PatchGAN, loss BCE + L1) untuk menerjemahkan **peta label fasad → foto gedung** (dataset CMP Facades base, 378 pasangan). Semua kode ditulis bersama LLM dari spesifikasi kelompok, tanpa bobot pretrained.

**Cara pakai:** *Runtime → Change runtime type → T4 GPU*, atur sel **Pengaturan** di bawah, lalu *Runtime → Run all*. Di akhir, `hasil_pix2pix.zip` terunduh otomatis.

**Penting:**
- Jangan menambah atau mengubah sel kode sumber (bagian 1). Sel verifikasi akan menghentikan *Run all* jika isi file tidak sama dengan repo.
- Jalankan sel secara berurutan (*Run all*, atau *Runtime → Jalankan setelahnya* dari sel tertentu). Menjalankan sel acak bisa membuat file tertulis setengah.
- Jika *Run all* diulang di runtime yang sama, training yang sudah selesai dilewati (`LANJUTKAN_RUN`).

| Bagian | Isi | Rubrik panduan | Job desk (usulan) |
|---|---|---|---|
| 1 | Kode sumber (sel bisa dibuka untuk dibaca) dan verifikasi isi file | Kualitas & orisinalitas kode | Semua |
| 2 | Tes otomatis (`pytest`) | Fungsionalitas | Anggota 4 |
| 3–5 | Dataset: unduh, grid pratinjau, smoke test | Fungsionalitas | Anggota 2 |
| 6 | Perbandingan `dataset.py` vs `dataset_baru.py` (jika file ada) | Log interaksi (penanganan versi) | Anggota 2 dan 4 |
| 7 | Pengukuran VRAM (dasar pemilihan batch, ngf, AMP) | Training & tuning | Anggota 3 |
| 8.1 | Training run utama, grafik loss, grid sampel per epoch | Fungsionalitas & pelatihan | Anggota 3 |
| 8.2 | Ablation dan tuning (λ, skip, n_layers, learning rate, batch size) | Fungsionalitas & pelatihan (tuning) | Anggota 3 |
| 8.3 | Evaluasi split test (L1, PSNR, SSIM) dan perbandingan antar-run | Fungsionalitas & pelatihan (metrik) | Anggota 3 dan 2 |
| 9 | Ringkasan hasil (`ringkasan.md`) | Dokumentasi | Anggota 4 |
| 10 | Zip semua hasil | Dokumentasi | Anggota 4 |

## Pengaturan

In [ ]:
#@title Pengaturan run (ubah sebelum Run all)
EPOCHS = 100  #@param {type:"integer"}
JALANKAN_TES = True  #@param {type:"boolean"}
UKUR_VRAM = False  #@param {type:"boolean"}
ABLASI = "lambda100, lambda10, noskip"  #@param {type:"string"}
EPOCHS_ABLASI = 30  #@param {type:"integer"}
JALANKAN_EVAL = True  #@param {type:"boolean"}
LANJUTKAN_RUN = True  #@param {type:"boolean"}
#@markdown - `EPOCHS`: epoch run utama. 5 = uji cepat (sekitar 2 menit di T4); 100 = training penuh (sekitar 30 menit).
#@markdown - `JALANKAN_TES`: sekitar 1–2 menit. Boleh dimatikan untuk run training berikutnya.
#@markdown - `UKUR_VRAM`: sekitar 5 menit. Cukup sekali; matikan untuk run training berikutnya.
#@markdown - `ABLASI`: daftar preset dipisah koma, kosong = tidak ada. Pilihan: `lambda10, lambda100, lambda200, noskip, nlayers1, lr1e-4, lr4e-4, batch1, batch4`. Setiap preset 30 epoch sekitar 8 menit di T4 (batch1 sekitar 2 kali lebih lama). Sesi Colab gratis bisa terputus, jadi jalankan beberapa preset per sesi.
#@markdown - `JALANKAN_EVAL`: evaluasi split test untuk semua run di `/content/runs`.
#@markdown - `LANJUTKAN_RUN`: **True (disarankan)** = run yang checkpoint-nya sudah mencapai jumlah epoch DILEWATI, yang terhenti DILANJUTKAN (resume). Jadi *Run all* aman diulang di runtime yang sama tanpa melatih ulang. False = semua run dihapus dan dilatih dari awal.
print(f"EPOCHS={EPOCHS}, JALANKAN_TES={JALANKAN_TES}, UKUR_VRAM={UKUR_VRAM}, "
      f"ABLASI={ABLASI!r}, EPOCHS_ABLASI={EPOCHS_ABLASI}, JALANKAN_EVAL={JALANKAN_EVAL}, "
      f"LANJUTKAN_RUN={LANJUTKAN_RUN}")


## 0. Lingkungan dan fungsi bantu

In [ ]:
import sys, subprocess
from pathlib import Path
import torch, PIL, numpy

print("Python", sys.version.split()[0], "| torch", torch.__version__, "| Pillow", PIL.__version__, "| numpy", numpy.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "tidak ada (bagian VRAM akan dilewati)")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=False)


In [ ]:
import os, shutil
REPO = Path("/content/pix2pix")
HASIL = Path("/content/hasil")
RUNS = Path("/content/runs")
# Folder hasil dikosongkan setiap Run all, supaya file dari run sebelumnya tidak ikut
# masuk zip. Hasil training (checkpoint, log) ada di RUNS dan TIDAK dihapus di sini.
shutil.rmtree(HASIL, ignore_errors=True)
for d in (REPO / "models", REPO / "scripts", REPO / "tests", HASIL, RUNS):
    d.mkdir(parents=True, exist_ok=True)

def sh(cmd, log=None, cwd=REPO):
    """Jalankan perintah shell. Output tampil LANGSUNG baris per baris dan ditulis
    ke HASIL/log saat itu juga (tetap tersimpan walaupun proses terhenti di tengah).
    Mengembalikan return code; tidak menghentikan Run all."""
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}   # tanpa ini, print di proses anak tertahan di buffer
    f = open(HASIL / log, "w", encoding="utf-8") if log else None
    if f:
        f.write(f"$ {cmd}\n\n"); f.flush()
    p = subprocess.Popen(cmd, shell=True, cwd=str(cwd), env=env, text=True, bufsize=1,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end="", flush=True)
        if f:
            f.write(line); f.flush()
    rc = p.wait()
    if f:
        f.close()
    return rc

def status_run(nama):
    """(status, epoch) sebuah run di RUNS/nama, dibaca dari checkpoint last.pt:
    'belum' = tidak ada checkpoint; selain itu epoch terakhir yang tersimpan."""
    import torch
    last = RUNS / nama / "last.pt"
    if not last.exists() or not (RUNS / nama / "best.pt").exists():
        return "belum", 0
    try:
        ep = torch.load(last, map_location="cpu", weights_only=True, mmap=True)["epoch"]
    except Exception as e:   # checkpoint rusak (misalnya proses mati saat menyimpan)
        print(f"[{nama}] last.pt tidak bisa dibaca ({type(e).__name__}); run diulang dari awal.")
        return "belum", 0
    return "ada", int(ep)

def latih(nama, args, epochs, log):
    """Training satu run. Dengan LANJUTKAN_RUN=True (Pengaturan):
    - checkpoint sudah >= epochs  -> DILEWATI (tidak dilatih ulang)
    - checkpoint < epochs         -> DILANJUTKAN dengan --resume
    - belum ada checkpoint        -> mulai dari awal
    Dengan LANJUTKAN_RUN=False: folder run dihapus dan dilatih dari awal."""
    status, ep = status_run(nama)
    if LANJUTKAN_RUN and status == "ada" and ep >= epochs:
        print(f"[{nama}] sudah selesai sampai epoch {ep} -> training DILEWATI.")
        if ep > epochs:
            print(f"   Catatan: run ini {ep} epoch, lebih banyak dari pengaturan ({epochs}).")
        return 0
    if LANJUTKAN_RUN and status == "ada":
        print(f"[{nama}] terhenti di epoch {ep} -> DILANJUTKAN (resume) sampai epoch {epochs}.")
        return sh(f"{sys.executable} train.py {args} --set epochs={epochs} --resume", log=log)
    shutil.rmtree(RUNS / nama, ignore_errors=True)
    print(f"[{nama}] mulai dari awal, {epochs} epoch.")
    return sh(f"{sys.executable} train.py {args} --set epochs={epochs}", log=log)

def show(*paths):
    """Tampilkan gambar jika ada, dan salin ke HASIL."""
    import shutil
    from IPython.display import Image, display
    for p in map(Path, paths):
        if p.exists():
            shutil.copy(p, HASIL / p.name)
            print(p.name); display(Image(str(p)))
        else:
            print(f"(gambar {p.name} tidak ada; lihat log di atas)")


def tulis(rel, isi, baru):
    """Tulis satu bagian file repo: baru=True membuat file, False menambah di akhir."""
    path = REPO / rel
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w" if baru else "a", encoding="utf-8", newline="\n") as f:
        f.write(isi)


## 1. Kode sumber

Setiap file dipecah per bagian kode (kelas, method, atau blok). Setiap sel hanya menampilkan **judul**; klik judulnya (atau *Show code*) untuk membaca kodenya. Bagian pertama membuat file, bagian berikutnya menambah ke file yang sama. Sel verifikasi di akhir memastikan isi setiap file identik dengan repo (SHA-256).

### `pytest.ini`

Konfigurasi pytest: root repo masuk sys.path. 1 sel, 5 baris. Isi sel → 1: `lanjutan`

In [ ]:
#@title pytest.ini · 1/1: lanjutan
tulis('pytest.ini', r'''[pytest]
# Tambahkan root repo ke sys.path agar `from models.generator import ...` bisa
# dipakai dari folder tests/ tanpa harus menginstal paket.
pythonpath = .
testpaths = tests
''', baru=True)


### `tests/__init__.py`

Menjadikan tests paket biasa (mencegah tertimpa paket `tests` lain di Colab). 1 sel, 6 baris. Isi sel → 1: `lanjutan`

In [ ]:
#@title tests/__init__.py · 1/1: lanjutan
tulis('tests/__init__.py', r'''"""Paket tes proyek.

File ini membuat `tests` menjadi paket biasa. Tanpa file ini, paket lain
bernama `tests` di site-packages (ada di Colab) menang saat import, sehingga
`from tests.helpers import ...` gagal dengan ModuleNotFoundError.
"""
''', baru=True)


### `tests/helpers.py`

Pembuat data sintetis format CMP yang dipakai beberapa file tes. 1 sel, 23 baris. Isi sel → 1: `make_pairs`

In [ ]:
#@title tests/helpers.py · 1/1: make_pairs
tulis('tests/helpers.py', r'''"""
tests/helpers.py

Fungsi bantu bersama untuk tes. Dipisah dari file tes supaya tidak ada file
tes yang meng-import file tes lain.
"""

import numpy as np
from PIL import Image


def make_pairs(root, n=12, size=64):
    """n pasangan sintetis format CMP: label .png mode palet (indeks kelas +
    peta warna) dan foto .jpg RGB dengan nama dasar sama."""
    rng = np.random.default_rng(0)
    root.mkdir(parents=True)
    palette = [0, 0, 0] + [c for k in range(1, 13) for c in (k * 20, 255 - k * 20, 128)]
    for i in range(n):
        idx = np.kron(rng.integers(1, 13, (4, 4)), np.ones((size // 4, size // 4))).astype(np.uint8)
        label = Image.fromarray(idx, mode="P")
        label.putpalette(palette)
        label.save(root / f"cmp_b{i:04d}.png")
        Image.fromarray(rng.integers(0, 256, (size, size, 3), dtype=np.uint8)).save(root / f"cmp_b{i:04d}.jpg")
''', baru=True)


### `scripts/__init__.py`

Menjadikan scripts paket biasa (alasan sama dengan tests). 1 sel, 6 baris. Isi sel → 1: `lanjutan`

In [ ]:
#@title scripts/__init__.py · 1/1: lanjutan
tulis('scripts/__init__.py', r'''"""Paket skrip proyek.

File ini membuat `scripts` menjadi paket biasa, supaya `from scripts import ...`
di tes tidak tertimpa paket lain bernama `scripts` di site-packages.
Skrip tetap dijalankan langsung, misalnya `python scripts/measure_vram.py`.
"""
''', baru=True)


### `models/__init__.py`

Helper bersama G dan D: `make_norm`, `init_weights`. 2 sel, 52 baris. Isi sel → 1: `make_norm` · 2: `init_weights`

In [ ]:
#@title models/__init__.py · 1/2: make_norm
tulis('models/__init__.py', r'''"""
models/__init__.py

Helper yang dipakai bersama oleh generator dan discriminator:
- make_norm   : membuat layer normalisasi berdasarkan nama ("batch"/"instance"/"none")
- init_weights: inisialisasi bobot Normal(0, std) ala Pix2Pix

Sengaja TIDAK meng-import generator/discriminator di sini supaya tidak terjadi
circular import (generator.py meng-import helper dari file ini).
"""

import torch.nn as nn

# Nama normalisasi yang didukung. Dipakai untuk validasi di generator & discriminator.
NORM_TYPES = ("batch", "instance", "none")


def make_norm(norm: str, num_channels: int) -> nn.Module:
    """Kembalikan layer normalisasi untuk `num_channels` channel.

    - "batch"   : BatchNorm2d. Statistik dihitung per channel dari seluruh batch
                  (B x H x W nilai). Punya parameter affine (gamma, beta).
    - "instance": InstanceNorm2d tanpa affine. Statistik dihitung per sampel per
                  channel (H x W nilai), jadi tidak bergantung ukuran batch.
                  Opsi ini berguna kalau batch harus 1-2 karena VRAM terbatas.
    - "none"    : nn.Identity, tidak melakukan apa-apa. Dipakai agar forward()
                  setiap blok tetap seragam (conv -> norm -> aktivasi).
    """
    if norm == "batch":
        return nn.BatchNorm2d(num_channels)
    if norm == "instance":
        return nn.InstanceNorm2d(num_channels, affine=False, track_running_stats=False)
    if norm == "none":
        return nn.Identity()
    raise ValueError(f"norm harus salah satu dari {NORM_TYPES}, didapat {norm!r}")
''', baru=True)


In [ ]:
#@title models/__init__.py · 2/2: init_weights
tulis('models/__init__.py', r'''

def init_weights(module: nn.Module, std: float = 0.02) -> None:
    """Inisialisasi bobot satu modul. Dipanggil lewat model.apply(...).

    - Conv2d / ConvTranspose2d: bobot ~ Normal(0, std), bias = 0 (jika ada).
    - BatchNorm2d: gamma ~ Normal(1, std), beta = 0. Gamma di sekitar 1 supaya
      di awal training BN hampir tidak mengubah skala aktivasi.
    - InstanceNorm2d tanpa affine tidak punya bobot, jadi dilewati.
    """
    if isinstance(module, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(module.weight, mean=0.0, std=std)
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, nn.BatchNorm2d):
        nn.init.normal_(module.weight, mean=1.0, std=std)
        nn.init.zeros_(module.bias)
''', baru=False)


### `models/generator.py`

Generator U-Net. 7 sel, 244 baris. Isi sel → 1: `import dan konstanta` · 2: `DownBlock` · 3: `forward` · 4: `__init__` · 5: `Validasi argumen: gagal cepat dengan pesan yang jelas` · 6: `Encoder` · 7: `forward`

In [ ]:
#@title models/generator.py · 1/7: import dan konstanta
tulis('models/generator.py', r'''"""
models/generator.py

Generator U-Net untuk Pix2Pix, ditulis from scratch dengan layer dasar torch.nn.

Ringkasan arsitektur (img_size=256, ngf=64, use_skip=True):

    Encoder (Conv 4x4 s2 p1 -> [Norm] -> LeakyReLU 0.2)   spasial   channel
      e1  tanpa norm                                       128      64
      e2                                                    64     128
      e3                                                    32     256
      e4                                                    16     512
      e5                                                     8     512
      e6                                                     4     512
      e7                                                     2     512
      e8  bottleneck, tanpa norm                             1     512

    Decoder (ConvT 4x4 s2 p1 -> Norm -> [Dropout] -> ReLU), lalu concat skip
      d1  in 512   -> out 512 (2x2),   dropout, concat e7 -> 1024
      d2  in 1024  -> out 512 (4x4),   dropout, concat e6 -> 1024
      d3  in 1024  -> out 512 (8x8),   dropout, concat e5 -> 1024
      d4  in 1024  -> out 512 (16x16),          concat e4 -> 1024
      d5  in 1024  -> out 256 (32x32),          concat e3 -> 512
      d6  in 512   -> out 128 (64x64),          concat e2 -> 256
      d7  in 256   -> out 64  (128x128),        concat e1 -> 128
      out in 128   -> out_ch (256x256), ConvT + Tanh

    Untuk img_size=128 jumlah blok encoder 7 (bottleneck tetap 1x1) dan decoder 6.
    Dengan use_skip=False tidak ada concat, sehingga channel input decoder
    (kecuali d1) menjadi setengahnya.
"""

import math
from functools import partial

import torch
import torch.nn as nn

from . import NORM_TYPES, init_weights, make_norm

# Ukuran gambar yang didukung. num_downs = log2(img_size) -> 7 untuk 128, 8 untuk 256,
# sehingga bottleneck selalu berukuran 1x1.
SUPPORTED_IMG_SIZES = (128, 256)
''', baru=True)


In [ ]:
#@title models/generator.py · 2/7: DownBlock
tulis('models/generator.py', r'''

class DownBlock(nn.Module):
    """Satu blok encoder: Conv 4x4 stride 2 -> Norm -> LeakyReLU.

    Conv dengan kernel 4, stride 2, padding 1 membagi dua tinggi dan lebar secara
    tepat: out = (in + 2*1 - 4) / 2 + 1 = in / 2 (untuk in genap).
    """

    def __init__(self, in_ch: int, out_ch: int, norm: str, leaky_slope: float):
        super().__init__()
        # Bias tidak dipakai jika ada normalisasi: BN/IN mengurangi mean per
        # channel, sehingga bias konstan dari conv akan terhapus (parameter mubazir).
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=4, stride=2, padding=1,
                              bias=(norm == "none"))
        self.norm = make_norm(norm, out_ch)
        # LeakyReLU membiarkan sedikit gradien lewat untuk nilai negatif,
        # sehingga neuron encoder tidak "mati" seperti pada ReLU biasa.
        self.act = nn.LeakyReLU(leaky_slope)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # [B, in_ch, H, W] -> [B, out_ch, H/2, W/2]
        return self.act(self.norm(self.conv(x)))


class UpBlock(nn.Module):
    """Satu blok decoder: ConvTranspose 4x4 stride 2 -> Norm -> [Dropout] -> ReLU.

    ConvTranspose dengan kernel 4, stride 2, padding 1 menggandakan ukuran spasial:
    out = (in - 1)*2 - 2*1 + 4 = 2*in.
    """

    def __init__(self, in_ch: int, out_ch: int, norm: str, dropout_p: float):
        super().__init__()
        self.deconv = nn.ConvTranspose2d(in_ch, out_ch, kernel_size=4, stride=2,
                                         padding=1, bias=(norm == "none"))
        self.norm = make_norm(norm, out_ch)
        # dropout_p = 0 berarti blok ini tidak memakai dropout (pakai Identity
        # supaya struktur forward tetap sama dan mudah dibaca).
        self.dropout = nn.Dropout(dropout_p) if dropout_p > 0 else nn.Identity()
        self.act = nn.ReLU()
''', baru=False)


In [ ]:
#@title models/generator.py · 3/7: forward
tulis('models/generator.py', r'''
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # [B, in_ch, H, W] -> [B, out_ch, 2H, 2W]
        return self.act(self.dropout(self.norm(self.deconv(x))))


class UNetGenerator(nn.Module):
    """Generator U-Net Pix2Pix.

    Args:
        in_ch:        jumlah channel gambar input (kondisi), default 3 (RGB).
        out_ch:       jumlah channel gambar output, default 3.
        ngf:          lebar channel dasar. Lebar encoder: ngf, 2ngf, 4ngf, 8ngf,
                      lalu tetap 8ngf. Kenop utama penghemat VRAM (64 atau 32).
        img_size:     ukuran sisi gambar persegi; hanya 128 atau 256.
        use_dropout:  pasang Dropout di `n_dropout` blok decoder pertama.
        use_skip:     True = U-Net (concat skip), False = encoder-decoder biasa.
        norm:         "batch", "instance", atau "none".

    Argumen keyword tambahan (default = nilai spesifikasi; disediakan agar bisa
    diatur dari config.py, bukan di-hardcode):
        n_dropout:    jumlah blok decoder pertama yang memakai dropout (default 3).
        dropout_p:    probabilitas dropout (default 0.5).
        max_mult:     batas pengali channel encoder (default 8 -> maksimum 8*ngf).
        leaky_slope:  kemiringan negatif LeakyReLU encoder (default 0.2).
        init_std:     std inisialisasi Normal untuk bobot conv (default 0.02).

    Shape:
        input  : [B, in_ch,  img_size, img_size], diharapkan bernilai di [-1, 1]
        output : [B, out_ch, img_size, img_size], nilai di [-1, 1] (Tanh)
    """
''', baru=False)


In [ ]:
#@title models/generator.py · 4/7: __init__
tulis('models/generator.py', r'''
    def __init__(
        self,
        in_ch: int = 3,
        out_ch: int = 3,
        ngf: int = 64,
        img_size: int = 256,
        use_dropout: bool = True,
        use_skip: bool = True,
        norm: str = "batch",
        *,
        n_dropout: int = 3,
        dropout_p: float = 0.5,
        max_mult: int = 8,
        leaky_slope: float = 0.2,
        init_std: float = 0.02,
    ):
        super().__init__()
''', baru=False)


In [ ]:
#@title models/generator.py · 5/7: lanjutan
tulis('models/generator.py', r'''
        # ---------- Validasi argumen: gagal cepat dengan pesan yang jelas ----------
        if img_size not in SUPPORTED_IMG_SIZES:
            raise ValueError(
                f"img_size harus salah satu dari {SUPPORTED_IMG_SIZES}, didapat {img_size}. "
                "Ukuran lain mengubah jumlah blok sehingga bottleneck tidak lagi 1x1."
            )
        if norm not in NORM_TYPES:
            raise ValueError(f"norm harus salah satu dari {NORM_TYPES}, didapat {norm!r}")
        for name, val in (("in_ch", in_ch), ("out_ch", out_ch), ("ngf", ngf),
                          ("max_mult", max_mult)):
            if not isinstance(val, int) or val < 1:
                raise ValueError(f"{name} harus bilangan bulat >= 1, didapat {val!r}")
        if not 0.0 <= dropout_p < 1.0:
            raise ValueError(f"dropout_p harus di [0, 1), didapat {dropout_p}")

        self.in_ch = in_ch
        self.out_ch = out_ch
        self.img_size = img_size
        self.use_skip = use_skip
        # log2(256) = 8, log2(128) = 7 -> spasial bottleneck = img_size / 2**num_downs = 1
        self.num_downs = int(math.log2(img_size))

        n_up = self.num_downs - 1  # jumlah UpBlock (layer output dihitung terpisah)
        if not 0 <= n_dropout <= n_up:
            raise ValueError(f"n_dropout harus di [0, {n_up}], didapat {n_dropout}")

        # Lebar channel tiap blok encoder: ngf * min(2**i, max_mult)
        # contoh ngf=64: [64, 128, 256, 512, 512, 512, 512, 512]
        enc_ch = [ngf * min(2 ** i, max_mult) for i in range(self.num_downs)]
        self.enc_channels = enc_ch
''', baru=False)


In [ ]:
#@title models/generator.py · 6/7: Decoder ke-j (j = 1..n_up) mengeluarkan channel yang sama dengan encoder
tulis('models/generator.py', r'''
        # ---------- Encoder ----------
        self.encoder = nn.ModuleList()
        for i in range(self.num_downs):
            block_in = in_ch if i == 0 else enc_ch[i - 1]
            # Blok pertama: input masih berupa gambar mentah, norm tidak diperlukan.
            # Bottleneck (blok terakhir): spasial 1x1, sehingga dengan batch 1
            # BatchNorm hanya melihat 1 nilai per channel dan akan error.
            is_first_or_bottleneck = i == 0 or i == self.num_downs - 1
            block_norm = "none" if is_first_or_bottleneck else norm
            self.encoder.append(DownBlock(block_in, enc_ch[i], block_norm, leaky_slope))

        # ---------- Decoder ----------
        # Decoder ke-j (j = 1..n_up) mengeluarkan channel yang sama dengan encoder
        # "pasangannya", yaitu enc_ch[num_downs - 1 - j], dengan ukuran spasial yang
        # sama pula. Karena itu concat skip di dimensi channel bisa dilakukan.
        self.decoder = nn.ModuleList()
        for j in range(1, n_up + 1):
            block_out = enc_ch[self.num_downs - 1 - j]
            if j == 1:
                # d1 hanya menerima bottleneck (belum ada skip yang di-concat).
                block_in = enc_ch[-1]
            else:
                # Output decoder sebelumnya = enc_ch[num_downs - j] channel.
                # Dengan skip, ditambah fitur encoder yang sama lebarnya -> 2x.
                prev_out = enc_ch[self.num_downs - j]
                block_in = 2 * prev_out if use_skip else prev_out
            p = dropout_p if (use_dropout and j <= n_dropout) else 0.0
            self.decoder.append(UpBlock(block_in, block_out, norm, p))

        # ---------- Layer output ----------
        # Tanpa norm dan tanpa ReLU: kita ingin nilai bebas yang lalu dipetakan
        # Tanh ke [-1, 1], sama dengan rentang gambar target setelah normalisasi.
        # Bias dipakai karena tidak ada normalisasi sesudahnya.
        final_in = 2 * enc_ch[0] if use_skip else enc_ch[0]
        self.final = nn.Sequential(
            nn.ConvTranspose2d(final_in, out_ch, kernel_size=4, stride=2, padding=1,
                               bias=True),
            nn.Tanh(),
        )
''', baru=False)


In [ ]:
#@title models/generator.py · 7/7: forward
tulis('models/generator.py', r'''
        # ---------- Inisialisasi bobot ----------
        # apply() memanggil init_weights untuk setiap sub-modul secara rekursif.
        self.apply(partial(init_weights, std=init_std))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Validasi shape input supaya kesalahan data terdeteksi di sini, bukan
        # sebagai error shape yang membingungkan di tengah decoder.
        expected = (self.in_ch, self.img_size, self.img_size)
        if x.dim() != 4 or tuple(x.shape[1:]) != expected:
            raise ValueError(
                f"Input generator harus [B, {self.in_ch}, {self.img_size}, "
                f"{self.img_size}], didapat {list(x.shape)}"
            )

        # ----- Encoder: simpan output tiap blok untuk skip connection -----
        # Fitur hanya disimpan jika use_skip=True agar tidak memakan memori sia-sia.
        skips = []
        h = x
        for block in self.encoder:
            h = block(h)
            if self.use_skip:
                skips.append(h)
        # Sekarang h = bottleneck, shape [B, 8*ngf, 1, 1]

        # ----- Decoder: upsample lalu (opsional) concat dengan fitur encoder -----
        for j, block in enumerate(self.decoder, start=1):
            h = block(h)
            if self.use_skip:
                # skips[num_downs-1-j] berukuran spasial sama dengan h.
                # Urutan concat: [decoder, encoder] di dimensi channel (dim=1).
                h = torch.cat([h, skips[self.num_downs - 1 - j]], dim=1)

        # ----- Output: [B, final_in, img/2, img/2] -> [B, out_ch, img, img] -----
        return self.final(h)

    def num_parameters(self, trainable_only: bool = True) -> int:
        """Jumlah parameter model (untuk laporan dan perbandingan ablation)."""
        return sum(p.numel() for p in self.parameters()
                   if p.requires_grad or not trainable_only)
''', baru=False)


### `models/discriminator.py`

PatchGAN discriminator + fungsi hitung ukuran patch dan receptive field. 6 sel, 196 baris. Isi sel → 1: `import dan konstanta` · 2: `layer_strides` · 3: `PatchDiscriminator` · 4: `width` · 5: `forward` · 6: `output_size`

In [ ]:
#@title models/discriminator.py · 1/6: import dan konstanta
tulis('models/discriminator.py', r'''"""
models/discriminator.py

PatchGAN discriminator conditional untuk Pix2Pix, ditulis from scratch.

Ide utama: alih-alih satu skor "asli/palsu" untuk seluruh gambar, D mengeluarkan
PETA skor [B, 1, h, w]. Setiap elemen peta menilai satu patch input berukuran
receptive field (RF) x RF. D jadi fokus pada struktur lokal/tekstur, sedangkan
struktur global sudah dijaga oleh loss L1 di generator.

Susunan layer (kernel 4, padding 1 untuk semua conv):

    blok   stride  norm   aktivasi      channel out (ndf=64, n_layers=3)
    c1     2       tidak  LeakyReLU 0.2  64
    c2     2       ya     LeakyReLU 0.2  128
    c3     2       ya     LeakyReLU 0.2  256        <- total n_layers conv stride 2
    c4     1       ya     LeakyReLU 0.2  512
    out    1       tidak  tidak ada      1  (LOGIT, tanpa sigmoid)

Untuk input 256 dan n_layers=3: 256 -> 128 -> 64 -> 32 -> 31 -> 30, RF = 70.
"""

from functools import partial

import torch
import torch.nn as nn

from . import NORM_TYPES, init_weights, make_norm

# Semua conv di D memakai kernel 4 dan padding 1 (bagian dari definisi arsitektur).
KERNEL = 4
PADDING = 1
''', baru=True)


In [ ]:
#@title models/discriminator.py · 2/6: layer_strides
tulis('models/discriminator.py', r'''

# ---------------------------------------------------------------------------
# Fungsi murni untuk perhitungan ukuran. Tidak membuat model, sehingga bisa
# dipakai untuk menjelaskan angka di laporan dan untuk membandingkan dengan
# shape tensor yang sebenarnya di tes.
# ---------------------------------------------------------------------------
def layer_strides(n_layers: int) -> list:
    """Daftar stride berurutan: n_layers kali stride 2, lalu dua kali stride 1
    (conv stride 1 dengan norm, lalu conv output 1 channel)."""
    return [2] * n_layers + [1, 1]


def conv_output_size(size: int, kernel: int, stride: int, padding: int) -> int:
    """Rumus ukuran output Conv2d: floor((size + 2p - k) / s) + 1."""
    return (size + 2 * padding - kernel) // stride + 1


def patch_output_size(img_size: int, n_layers: int) -> int:
    """Ukuran sisi peta logit D untuk input img_size x img_size."""
    size = img_size
    for s in layer_strides(n_layers):
        size = conv_output_size(size, KERNEL, s, PADDING)
    return size


def receptive_field(n_layers: int) -> int:
    """Receptive field satu elemen output, dihitung mundur dari layer terakhir.

    Mulai dari 1 piksel di output. Mundur melewati satu conv (kernel k, stride s),
    sebuah daerah selebar r di output conv itu berasal dari daerah selebar
    (r - 1) * s + k di inputnya: r - 1 langkah antar-piksel yang masing-masing
    bergeser s piksel di input, ditambah lebar satu kernel. Padding tidak
    mengubah lebar RF, hanya posisinya.
    """
    rf = 1
    for s in reversed(layer_strides(n_layers)):
        rf = (rf - 1) * s + KERNEL
    return rf
''', baru=False)


In [ ]:
#@title models/discriminator.py · 3/6: PatchDiscriminator
tulis('models/discriminator.py', r'''

class PatchDiscriminator(nn.Module):
    """PatchGAN conditional.

    Args:
        in_ch:    total channel setelah concat (cond + img), default 6 = 3 + 3.
        ndf:      lebar channel dasar. Lebar: ndf, 2ndf, 4ndf, ... maks max_mult*ndf.
        n_layers: jumlah conv stride 2. 1 -> RF 16, 3 -> RF 70.
        norm:     "batch", "instance", atau "none".

    Argumen keyword tambahan (default = spesifikasi; untuk diatur dari config.py):
        max_mult:    batas pengali channel (default 8).
        leaky_slope: kemiringan negatif LeakyReLU (default 0.2).
        init_std:    std inisialisasi bobot conv (default 0.02).

    Shape:
        cond, img : [B, c_cond, H, W] dan [B, c_img, H, W], c_cond + c_img = in_ch
        output    : [B, 1, h, w] LOGIT (belum di-sigmoid). Pasangkan dengan
                    nn.BCEWithLogitsLoss, jangan BCELoss.
    """

    def __init__(
        self,
        in_ch: int = 6,
        ndf: int = 64,
        n_layers: int = 3,
        norm: str = "batch",
        *,
        max_mult: int = 8,
        leaky_slope: float = 0.2,
        init_std: float = 0.02,
    ):
        super().__init__()
''', baru=False)


In [ ]:
#@title models/discriminator.py · 4/6: width
tulis('models/discriminator.py', r'''
        # ---------- Validasi argumen ----------
        if norm not in NORM_TYPES:
            raise ValueError(f"norm harus salah satu dari {NORM_TYPES}, didapat {norm!r}")
        for name, val in (("in_ch", in_ch), ("ndf", ndf), ("n_layers", n_layers),
                          ("max_mult", max_mult)):
            if not isinstance(val, int) or val < 1:
                raise ValueError(f"{name} harus bilangan bulat >= 1, didapat {val!r}")

        self.in_ch = in_ch
        self.n_layers = n_layers

        def width(i: int) -> int:
            # Lebar channel layer ke-i: ndf * 2^i, dibatasi max_mult * ndf
            return ndf * min(2 ** i, max_mult)

        layers = []

        # ---------- c1: conv stride 2 TANPA norm ----------
        # Input masih berupa gambar mentah (nilai [-1, 1]); norm di sini tidak
        # perlu. Bias dipakai karena tidak ada norm sesudahnya.
        layers += [
            nn.Conv2d(in_ch, width(0), KERNEL, stride=2, padding=PADDING, bias=True),
            nn.LeakyReLU(leaky_slope),
        ]

        # ---------- c2..c_n: conv stride 2 dengan norm ----------
        # Setiap layer membagi dua ukuran spasial dan menggandakan channel.
        for i in range(1, n_layers):
            layers += [
                nn.Conv2d(width(i - 1), width(i), KERNEL, stride=2, padding=PADDING,
                          bias=(norm == "none")),
                make_norm(norm, width(i)),
                nn.LeakyReLU(leaky_slope),
            ]
''', baru=False)


In [ ]:
#@title models/discriminator.py · 5/6: forward
tulis('models/discriminator.py', r'''
        # ---------- conv stride 1 dengan norm ----------
        # Menambah kedalaman dan RF tanpa memperkecil peta lagi
        # (ukuran hanya berkurang 1 karena kernel 4 dengan padding 1).
        layers += [
            nn.Conv2d(width(n_layers - 1), width(n_layers), KERNEL, stride=1,
                      padding=PADDING, bias=(norm == "none")),
            make_norm(norm, width(n_layers)),
            nn.LeakyReLU(leaky_slope),
        ]

        # ---------- output: conv stride 1 ke 1 channel ----------
        # Tanpa norm, tanpa aktivasi, TANPA Sigmoid: hasilnya logit.
        # Sigmoid sudah ada di dalam BCEWithLogitsLoss (lebih stabil secara
        # numerik dan aman dipakai di bawah autocast/AMP).
        layers.append(
            nn.Conv2d(width(n_layers), 1, KERNEL, stride=1, padding=PADDING, bias=True)
        )

        self.net = nn.Sequential(*layers)
        self.apply(partial(init_weights, std=init_std))

    def forward(self, cond: torch.Tensor, img: torch.Tensor) -> torch.Tensor:
        # ----- Validasi: cond dan img harus sejajar piksel-per-piksel -----
        if cond.dim() != 4 or img.dim() != 4:
            raise ValueError("cond dan img harus tensor 4D [B, C, H, W]")
        if cond.shape[0] != img.shape[0] or cond.shape[2:] != img.shape[2:]:
            raise ValueError(
                f"cond {list(cond.shape)} dan img {list(img.shape)} harus punya "
                "batch dan ukuran spasial yang sama"
            )
        if cond.shape[1] + img.shape[1] != self.in_ch:
            raise ValueError(
                f"Jumlah channel cond + img = {cond.shape[1] + img.shape[1]}, "
                f"tidak sama dengan in_ch = {self.in_ch}"
            )
        h, w = img.shape[2:]
        if min(patch_output_size(h, self.n_layers), patch_output_size(w, self.n_layers)) < 1:
            raise ValueError(f"Input {h}x{w} terlalu kecil untuk n_layers={self.n_layers}")
''', baru=False)


In [ ]:
#@title models/discriminator.py · 6/6: output_size
tulis('models/discriminator.py', r'''
        # ----- Conditioning: concat di dimensi channel -----
        # Urutan tetap [cond, img] agar D selalu melihat pasangan dengan susunan
        # yang sama, baik untuk pasangan asli maupun palsu.
        x = torch.cat([cond, img], dim=1)  # [B, in_ch, H, W]
        return self.net(x)                 # [B, 1, h, w] logit

    def output_size(self, img_size: int) -> int:
        """Ukuran sisi peta logit untuk input img_size (pakai fungsi murni)."""
        return patch_output_size(img_size, self.n_layers)

    def receptive_field(self) -> int:
        return receptive_field(self.n_layers)

    def num_parameters(self, trainable_only: bool = True) -> int:
        return sum(p.numel() for p in self.parameters()
                   if p.requires_grad or not trainable_only)
''', baru=False)


### `dataset.py`

Dataset berpasangan (versi lama dari sesi kita). 12 sel, 410 baris. Isi sel → 1: `import dan konstanta` · 2: `to_tensor` · 3: `scan_pairs` · 4: `_pair_key` · 5: `load_or_create_manifest` · 6: `resolve_split` · 7: `PairedImageDataset` · 8: `__init__` · 9: `_load_raw_pair` · 10: `_crop_flip` · 11: `_file_md5` · 12: `get_dataloader`

In [ ]:
#@title dataset.py · 1/12: import dan konstanta
tulis('dataset.py', r'''"""
dataset.py

Dataset berpasangan untuk Pix2Pix (target: CMP Facades, label -> foto).

Dua format file didukung dan dideteksi otomatis per folder:

1. "berdampingan" (combined): satu file berisi dua gambar kiri|kanan,
   lebar = 2 x tinggi. Mentahnya: A_raw = setengah KIRI, B_raw = setengah KANAN.
2. "terpisah" (separate): dua file dengan nama dasar sama, label .png dan
   foto .jpg/.jpeg. Mentahnya: A_raw = label (.png), B_raw = foto (.jpg).

`direction="AtoB"` memakai urutan mentah apa adanya, `"BtoA"` menukarnya.
Sisi mana yang label pada format berdampingan TIDAK diasumsikan di kode:
cek dengan scripts/preview_pairs.py lalu pilih direction yang membuat
A = label dan B = foto.

Split:
- Jika root punya subfolder train/ val/ test/ -> pakai split bawaan.
- Jika tidak (semua file langsung di root) -> split deterministik
  (default seed 42, 80/10/10) yang disimpan ke file manifest JSON. Run
  berikutnya (dan anggota kelompok lain) membaca manifest yang sama.

Semua tensor keluaran: float32, CPU, rentang [-1, 1], shape [3, H, W].
"""

from __future__ import annotations

import hashlib
import json
import os
import random
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}
SPLITS = ("train", "val", "test")
DIRECTIONS = ("AtoB", "BtoA")
# Format terpisah: label disimpan sebagai .png, foto sebagai .jpg/.jpeg
LABEL_EXT = ".png"
PHOTO_EXTS = (".jpg", ".jpeg")

# Pillow >= 9.1 menyediakan enum ini (nama lama Image.BICUBIC sudah usang).
BICUBIC = Image.Resampling.BICUBIC
FLIP_LR = Image.Transpose.FLIP_LEFT_RIGHT
''', baru=True)


In [ ]:
#@title dataset.py · 2/12: to_tensor
tulis('dataset.py', r'''

# ===========================================================================
# Konversi gambar <-> tensor
# ===========================================================================
def to_tensor(img: Image.Image) -> torch.Tensor:
    """PIL RGB uint8 [H, W, 3] -> tensor float32 [3, H, W] di [-1, 1].

    x / 127.5 - 1 memetakan 0 -> -1 dan 255 -> +1, cocok dengan Tanh generator.
    """
    arr = np.asarray(img, dtype=np.float32)          # [H, W, 3], nilai 0..255
    t = torch.from_numpy(arr).permute(2, 0, 1)       # HWC -> CHW
    return (t / 127.5 - 1.0).contiguous()


def denormalize(x: torch.Tensor) -> torch.Tensor:
    """Tensor [-1, 1] -> uint8 [0, 255], shape tetap ([C,H,W] atau [B,C,H,W]).

    clamp dulu karena output model bisa sedikit keluar rentang; round agar
    konversi bolak-balik uint8 -> [-1,1] -> uint8 tepat sama.
    """
    x = x.detach().to("cpu", torch.float32).clamp(-1.0, 1.0)
    return ((x + 1.0) * 127.5).round().to(torch.uint8)


def _open_rgb(path: str) -> Image.Image:
    # `with` menutup file segera setelah dibaca. convert() membuat salinan di
    # memori, sehingga tidak ada file yang terkunci (penting di Windows).
    with Image.open(path) as im:
        return im.convert("RGB")
''', baru=False)


In [ ]:
#@title dataset.py · 3/12: scan_pairs
tulis('dataset.py', r'''

# ===========================================================================
# Menemukan pasangan file
# ===========================================================================
def scan_pairs(folder: Path) -> list:
    """Kembalikan daftar pasangan terurut dari satu folder.

    Setiap pasangan berupa tuple (path_a, path_b):
    - format berdampingan: (file, None)
    - format terpisah    : (label.png, foto.jpg)
    File non-gambar (misalnya .xml anotasi) diabaikan.
    """
    folder = Path(folder)
    files = sorted(p for p in folder.iterdir()
                   if p.is_file() and p.suffix.lower() in IMG_EXTS)

    groups: dict = {}
    for p in files:
        groups.setdefault(p.stem, []).append(p)

    pairs = []
    for stem in sorted(groups):
        group = groups[stem]
        if len(group) == 1:
            pairs.append((str(group[0]), None))
            continue
        labels = [p for p in group if p.suffix.lower() == LABEL_EXT]
        photos = [p for p in group if p.suffix.lower() in PHOTO_EXTS]
        if len(group) == 2 and len(labels) == 1 and len(photos) == 1:
            pairs.append((str(labels[0]), str(photos[0])))
        else:
            raise ValueError(
                f"Nama dasar '{stem}' di {folder} punya file yang tidak bisa "
                f"dipasangkan: {[p.name for p in group]}"
            )

    kinds = {b is None for _, b in pairs}
    if len(kinds) > 1:
        raise ValueError(
            f"Folder {folder} mencampur format berdampingan dan terpisah; "
            "pisahkan dulu agar pasangan tidak salah."
        )
    return pairs
''', baru=False)


In [ ]:
#@title dataset.py · 4/12: _pair_key
tulis('dataset.py', r'''

def _pair_key(root: Path, pair: tuple) -> str:
    """ID stabil sebuah pasangan untuk manifest: path relatif file pertama,
    pakai '/' agar manifest sama di Windows dan Linux."""
    return Path(pair[0]).relative_to(root).as_posix()


def _make_split(keys: list, seed: int, ratios: tuple) -> dict:
    """Acak urutan dengan seed tetap, lalu potong sesuai rasio."""
    keys = sorted(keys)                 # urutan awal tidak bergantung OS
    random.Random(seed).shuffle(keys)   # Random(seed) lokal: tidak mengganggu RNG global
    n = len(keys)
    n_train = int(round(ratios[0] * n))
    n_val = int(round(ratios[1] * n))
    return {
        "train": keys[:n_train],
        "val": keys[n_train:n_train + n_val],
        "test": keys[n_train + n_val:],
    }
''', baru=False)


In [ ]:
#@title dataset.py · 5/12: load_or_create_manifest
tulis('dataset.py', r'''

def load_or_create_manifest(root: Path, manifest_path: Path, seed: int,
                            ratios: tuple) -> dict:
    """Baca manifest jika sudah ada; jika belum, buat dan simpan."""
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        if manifest.get("seed") != seed or tuple(manifest.get("ratios", ())) != tuple(ratios):
            raise ValueError(
                f"Manifest {manifest_path} dibuat dengan seed/rasio berbeda "
                f"({manifest.get('seed')}, {manifest.get('ratios')}). Hapus file itu "
                "atau samakan config agar split tidak berubah diam-diam."
            )
        return manifest

    keys = [_pair_key(root, p) for p in scan_pairs(root)]
    if not keys:
        raise FileNotFoundError(f"Tidak ada gambar di {root}")
    manifest = {
        "seed": seed,
        "ratios": list(ratios),
        "n_total": len(keys),
        "splits": _make_split(keys, seed, ratios),
    }
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    # Tulis ke file sementara lalu ganti nama: file manifest tidak pernah
    # setengah jadi walaupun proses terhenti di tengah jalan.
    tmp = manifest_path.with_suffix(manifest_path.suffix + ".tmp")
    tmp.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    os.replace(tmp, manifest_path)
    return manifest
''', baru=False)


In [ ]:
#@title dataset.py · 6/12: resolve_split
tulis('dataset.py', r'''

def resolve_split(root, split: str, manifest_path=None, split_seed: int = 42,
                  split_ratios: tuple = (0.8, 0.1, 0.1)) -> tuple:
    """Kembalikan (daftar pasangan, sumber split: "bawaan" atau "manifest")."""
    root = Path(root)
    if not root.is_dir():
        raise FileNotFoundError(f"Folder dataset tidak ditemukan: {root}")

    builtin = [s for s in SPLITS if (root / s).is_dir()]
    if builtin:
        if split not in builtin:
            raise ValueError(
                f"Dataset di {root} punya split bawaan {builtin}, tetapi tidak ada "
                f"'{split}'. Pakai split yang tersedia."
            )
        return scan_pairs(root / split), "bawaan"

    if abs(sum(split_ratios) - 1.0) > 1e-6 or len(split_ratios) != 3:
        raise ValueError(f"split_ratios harus 3 angka berjumlah 1, didapat {split_ratios}")
    if manifest_path is None:
        manifest_path = root / f"split_manifest_seed{split_seed}.json"
    manifest = load_or_create_manifest(root, Path(manifest_path), split_seed,
                                       tuple(split_ratios))

    lookup = {_pair_key(root, p): p for p in scan_pairs(root)}
    keys = manifest["splits"][split]
    missing = [k for k in keys if k not in lookup]
    if missing:
        raise FileNotFoundError(
            f"{len(missing)} file di manifest tidak ditemukan (contoh: {missing[:3]}). "
            "Isi folder berubah sejak manifest dibuat."
        )
    return [lookup[k] for k in keys], "manifest"
''', baru=False)


In [ ]:
#@title dataset.py · 7/12: PairedImageDataset
tulis('dataset.py', r'''

# ===========================================================================
# Dataset
# ===========================================================================
class PairedImageDataset(Dataset):
    """Dataset pasangan (A = kondisi, B = target) untuk Pix2Pix.

    Args:
        root:      folder dataset (lihat docstring modul untuk tata letaknya).
        split:     "train", "val", atau "test".
        direction: "AtoB" (urutan mentah) atau "BtoA" (ditukar).
        img_size:  ukuran sisi gambar keluaran (crop akhir).
        load_size: ukuran resize sebelum random crop (hanya saat augment).
        augment:   None -> otomatis True untuk "train", False untuk lainnya.

    Keyword tambahan:
        subset_size:   ambil N pasangan pertama saja dan MATIKAN augmentasi
                       (untuk overfit test: input harus identik tiap epoch).
        manifest_path: lokasi file manifest jika split dibuat sendiri.
        split_seed, split_ratios: pengaturan split deterministik.

    __getitem__ -> {"A": [3,img,img] float32, "B": [3,img,img] float32, "path": str}
    """
''', baru=False)


In [ ]:
#@title dataset.py · 8/12: __init__
tulis('dataset.py', r'''
    def __init__(self, root, split: str, direction: str = "AtoB",
                 img_size: int = 256, load_size: int = 286, augment=None, *,
                 subset_size=None, manifest_path=None, split_seed: int = 42,
                 split_ratios: tuple = (0.8, 0.1, 0.1)):
        super().__init__()
        if split not in SPLITS:
            raise ValueError(f"split harus salah satu dari {SPLITS}, didapat {split!r}")
        if direction not in DIRECTIONS:
            raise ValueError(f"direction harus salah satu dari {DIRECTIONS}, didapat {direction!r}")
        if img_size < 1 or load_size < img_size:
            raise ValueError(f"Butuh 1 <= img_size <= load_size, didapat {img_size}, {load_size}")

        pairs, self.split_source = resolve_split(root, split, manifest_path,
                                                 split_seed, split_ratios)
        if not pairs:
            raise FileNotFoundError(f"Split '{split}' kosong di {root}")

        if subset_size is not None:
            if subset_size < 1:
                raise ValueError(f"subset_size harus >= 1, didapat {subset_size}")
            pairs = pairs[:subset_size]
            augment = False  # overfit test: data harus sama persis tiap epoch

        # Hanya string dan angka yang disimpan sebagai atribut, sehingga
        # objek dataset mudah di-pickle ke worker DataLoader (wajib di Windows).
        self.pairs = pairs
        self.split = split
        self.direction = direction
        self.img_size = img_size
        self.load_size = load_size
        self.augment = (split == "train") if augment is None else bool(augment)

    def __len__(self) -> int:
        return len(self.pairs)
''', baru=False)


In [ ]:
#@title dataset.py · 9/12: _load_raw_pair
tulis('dataset.py', r'''
    # ----- membaca pasangan mentah -----
    def _load_raw_pair(self, idx: int) -> tuple:
        path_a, path_b = self.pairs[idx]
        if path_b is None:
            # Format berdampingan: belah dua tepat di tengah.
            img = _open_rgb(path_a)
            w, h = img.size
            if w != 2 * h:
                raise ValueError(
                    f"{path_a}: ukuran {w}x{h}, bukan 2:1. Bukan file berdampingan?"
                )
            raw_a = img.crop((0, 0, w // 2, h))   # kiri
            raw_b = img.crop((w // 2, 0, w, h))   # kanan
        else:
            raw_a, raw_b = _open_rgb(path_a), _open_rgb(path_b)
            if raw_a.size != raw_b.size:
                raise ValueError(
                    f"Ukuran label {raw_a.size} != foto {raw_b.size} untuk {path_a}; "
                    "pasangan mungkin tidak sejajar."
                )
        if self.direction == "BtoA":
            raw_a, raw_b = raw_b, raw_a
        return raw_a, raw_b

    # ----- augmentasi -----
    def _sample_aug_params(self) -> tuple:
        """Ambil SATU set parameter acak (top, left, flip) untuk satu pasangan.

        Memakai RNG torch karena DataLoader memberi seed berbeda ke RNG torch
        di setiap worker. (RNG NumPy tidak di-seed per worker, sehingga worker
        bisa menghasilkan augmentasi kembar.)
        """
        max_off = self.load_size - self.img_size
        top = int(torch.randint(0, max_off + 1, (1,)))
        left = int(torch.randint(0, max_off + 1, (1,)))
        flip = bool(torch.rand(1) < 0.5)
        return top, left, flip
''', baru=False)


In [ ]:
#@title dataset.py · 10/12: _crop_flip
tulis('dataset.py', r'''
    def _crop_flip(self, img: Image.Image, top: int, left: int, flip: bool) -> Image.Image:
        s = self.img_size
        img = img.crop((left, top, left + s, top + s))
        return img.transpose(FLIP_LR) if flip else img

    def _transform_pair(self, raw_a: Image.Image, raw_b: Image.Image) -> tuple:
        if self.augment:
            # Train: resize ke load_size, lalu crop + flip dengan parameter yang
            # SAMA untuk A dan B. Jika parameternya diambil terpisah, piksel
            # label tidak lagi sejajar dengan piksel foto -> model belajar salah.
            size = (self.load_size, self.load_size)
            a = raw_a.resize(size, BICUBIC)
            b = raw_b.resize(size, BICUBIC)
            top, left, flip = self._sample_aug_params()   # dipanggil SEKALI
            a = self._crop_flip(a, top, left, flip)
            b = self._crop_flip(b, top, left, flip)
        else:
            # Val/test (dan subset): resize langsung ke img_size, deterministik.
            size = (self.img_size, self.img_size)
            a = raw_a.resize(size, BICUBIC)
            b = raw_b.resize(size, BICUBIC)
        return to_tensor(a), to_tensor(b)

    def __getitem__(self, idx: int) -> dict:
        raw_a, raw_b = self._load_raw_pair(idx)
        a, b = self._transform_pair(raw_a, raw_b)
        # "path": file berdampingan, atau file label untuk format terpisah
        return {"A": a, "B": b, "path": self.pairs[idx][0]}
''', baru=False)


In [ ]:
#@title dataset.py · 11/12: _file_md5
tulis('dataset.py', r'''

# ===========================================================================
# Pemeriksaan tumpang tindih antar split
# ===========================================================================
def _file_md5(path: str) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def find_split_overlap(datasets: dict) -> dict:
    """Cari pasangan yang muncul di lebih dari satu split.

    datasets: {"train": PairedImageDataset, "val": ..., "test": ...}
    Mengembalikan {"same_path": [...], "same_content": [...]}; keduanya
    harus kosong. "same_content" menangkap file yang isinya identik byte demi
    byte walaupun nama/foldernya berbeda (tidak menangkap gambar yang mirip
    tetapi dikompres ulang).
    """
    seen_path: dict = {}
    seen_hash: dict = {}
    same_path, same_content = [], []
    for split, ds in datasets.items():
        for pair in ds.pairs:
            for path in (p for p in pair if p is not None):
                key = str(Path(path).resolve())
                if key in seen_path and seen_path[key] != split:
                    same_path.append((path, seen_path[key], split))
                seen_path.setdefault(key, split)
                digest = _file_md5(path)
                if digest in seen_hash and seen_hash[digest][1] != split:
                    same_content.append((seen_hash[digest][0], path))
                seen_hash.setdefault(digest, (path, split))
    return {"same_path": same_path, "same_content": same_content}
''', baru=False)


In [ ]:
#@title dataset.py · 12/12: get_dataloader
tulis('dataset.py', r'''

# ===========================================================================
# DataLoader
# ===========================================================================
def get_dataloader(cfg, split: str) -> DataLoader:
    """Buat DataLoader dari config.

    Field cfg yang dibaca (nanti didefinisikan di config.py):
        data_root, direction, img_size, load_size, batch_size, num_workers,
        pin_memory, seed, subset_size, manifest_path, split_seed, split_ratios

    Catatan Windows: num_workers > 0 memakai proses "spawn", yang meng-import
    ulang skrip utama di setiap worker. Karena itu kode training WAJIB berada
    di dalam `if __name__ == "__main__":`. Dataset ini sendiri aman di-pickle.
    """
    dataset = PairedImageDataset(
        cfg.data_root, split, cfg.direction, cfg.img_size, cfg.load_size,
        augment=None,
        subset_size=cfg.subset_size,
        manifest_path=cfg.manifest_path,
        split_seed=cfg.split_seed,
        split_ratios=cfg.split_ratios,
    )
    # Generator ber-seed: urutan shuffle dan seed RNG tiap worker (yang
    # menentukan crop/flip) dapat direproduksi.
    gen = torch.Generator()
    gen.manual_seed(cfg.seed)
    return DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=(split == "train"),
        num_workers=cfg.num_workers,
        # pin_memory hanya berguna jika ada GPU; tanpa CUDA PyTorch memberi peringatan.
        pin_memory=bool(cfg.pin_memory) and torch.cuda.is_available(),
        generator=gen,
        # Worker tidak dibuat ulang tiap epoch: di Windows, spawn proses lambat.
        persistent_workers=cfg.num_workers > 0,
    )
''', baru=False)


### `config.py`

Semua hyperparameter, seed, device, simpan/muat JSON, preset ablation. 10 sel, 303 baris. Isi sel → 1: `import dan konstanta` · 2: `Config` · 3: `__post_init__` · 4: `_check_types` · 5: `data` · 6: `training` · 7: `set_seed` · 8: `get_device` · 9: `save_config` · 10: `get_preset`

In [ ]:
#@title config.py · 1/10: import dan konstanta
tulis('config.py', r'''"""
config.py

Satu-satunya tempat hyperparameter proyek Pix2Pix.

- `Config` adalah dataclass FROZEN: nilainya tidak bisa diubah setelah dibuat.
  Perubahan dilakukan dengan `dataclasses.replace(cfg, batch_size=4)`, yang
  membuat objek baru dan menjalankan validasi `__post_init__` lagi. Jadi tidak
  ada jalan untuk memasukkan nilai yang tidak valid tanpa ketahuan.
- Import modul ini tidak membaca file, tidak membuat folder, dan tidak membaca
  environment variable. Semua efek samping hanya terjadi saat fungsi dipanggil.

Status default (lihat komentar per field):
- direction, img_size, ngf, batch_size, use_amp: ditetapkan dari bukti, yaitu
  grid pratinjau CMP base dan scripts/measure_vram.py di Colab T4 (Nov 2026).
  Batas memori RTX 4050 tetap perlu dicek ulang di laptop.
- epochs dan nilai dari paper Pix2Pix (lr, beta, lambda_l1, d_loss_scale):
  belum diuji di eksperimen kita.
"""

from __future__ import annotations

import dataclasses
import json
import random
import typing
from dataclasses import dataclass, fields
from pathlib import Path
from types import MappingProxyType

import numpy as np
import torch

ALLOWED_IMG_SIZES = (128, 256)
ALLOWED_DIRECTIONS = ("AtoB", "BtoA")
ALLOWED_NORMS = ("batch", "instance", "none")
ALLOWED_LABEL_RESIZE = ("auto", "nearest", "bicubic")
ALLOWED_DEVICES = ("auto", "cpu", "cuda")
CONFIG_FILENAME = "config.json"
''', baru=True)


In [ ]:
#@title config.py · 2/10: Config
tulis('config.py', r'''

@dataclass(frozen=True)
class Config:
    # ------------------------------------------------------------------ DATA
    data_root: str = "data/facades"        # folder berisi pasangan gambar
    direction: str = "AtoB"                # grid pratinjau CMP base: A = label, B = foto
    img_size: int = 256                    # T4: semua kombinasi 256 <= 1,34 GB reserved
    load_size: int = 286                   # resize sebelum random crop (train)
    batch_size: int = 2                    # muat sampai 4 di T4; 2 dipertahankan
    num_workers: int = 2
    pin_memory: bool = True
    subset_size: typing.Optional[int] = None   # mis. 8 untuk overfit test
    manifest_path: typing.Optional[str] = None
    split_seed: int = 42
    split_ratios: tuple = (0.8, 0.1, 0.1)
    label_resize_mode: str = "auto"
    # ----------------------------------------------------------------- MODEL
    ngf: int = 64                          # T4: muat; 32 tidak diperlukan
    ndf: int = 64
    n_layers: int = 3                      # 3 -> PatchGAN 70x70
    norm: str = "batch"
    use_skip: bool = True
    use_dropout: bool = True
    n_dropout: int = 3
    dropout_p: float = 0.5
    max_mult: int = 8
    leaky_slope: float = 0.2
    init_std: float = 0.02
''', baru=False)


In [ ]:
#@title config.py · 3/10: __post_init__
tulis('config.py', r'''    # -------------------------------------------------------------- TRAINING
    lr: float = 2e-4
    beta1: float = 0.5
    beta2: float = 0.999
    epochs: int = 100                      # tebakan
    lambda_l1: float = 100.0
    d_loss_scale: float = 0.5
    use_amp: bool = False                  # T4: AMP tidak menghemat memori (didominasi
                                           # bobot+Adam fp32), GradScaler melewati step awal;
                                           # hanya lebih cepat ~1,8x. Hanya aktif di CUDA.
    seed: int = 42
    checkpoint_dir: str = "runs"
    run_name: str = "main"
    log_every: int = 50                    # iterasi
    device: str = "auto"

    def __post_init__(self):
        _check_types(self)
        # JSON tidak punya tuple: list dari file diubah kembali menjadi tuple.
        # object.__setattr__ dipakai karena dataclass ini frozen.
        object.__setattr__(self, "split_ratios", tuple(self.split_ratios))
        _validate(self)
''', baru=False)


In [ ]:
#@title config.py · 4/10: _check_types
tulis('config.py', r'''

# ===========================================================================
# Validasi
# ===========================================================================
def _check_types(cfg: Config) -> None:
    """Tolak tipe yang salah, misalnya use_skip="false" dari JSON yang diedit
    tangan (string "false" bernilai True di Python) atau batch_size=2.0."""
    hints = typing.get_type_hints(Config)
    for f in fields(cfg):
        value = getattr(cfg, f.name)
        expected = hints[f.name]
        optional = typing.get_origin(expected) is typing.Union and type(None) in typing.get_args(expected)
        if optional:
            if value is None:
                continue
            expected = next(t for t in typing.get_args(expected) if t is not type(None))

        if expected is bool:
            ok = isinstance(value, bool)
        elif expected is int:
            ok = isinstance(value, int) and not isinstance(value, bool)
        elif expected is float:
            ok = isinstance(value, (int, float)) and not isinstance(value, bool)
        elif expected is str:
            ok = isinstance(value, str)
        elif expected is tuple:
            ok = isinstance(value, (tuple, list))
        else:  # tidak terjadi untuk field yang ada sekarang
            ok = True
        if not ok:
            raise TypeError(
                f"Config.{f.name} harus bertipe {getattr(expected, '__name__', expected)}, "
                f"didapat {value!r} ({type(value).__name__})"
            )


def _validate(cfg: Config) -> None:
    def fail(msg: str):
        raise ValueError(f"Config tidak valid: {msg}")
''', baru=False)


In [ ]:
#@title config.py · 5/10: lanjutan
tulis('config.py', r'''
    # ----- data -----
    if cfg.img_size not in ALLOWED_IMG_SIZES:
        fail(f"img_size harus salah satu dari {ALLOWED_IMG_SIZES}, didapat {cfg.img_size}")
    if cfg.load_size < cfg.img_size:
        fail(f"load_size ({cfg.load_size}) harus >= img_size ({cfg.img_size})")
    r = cfg.split_ratios
    if (len(r) != 3
            or not all(isinstance(x, (int, float)) and not isinstance(x, bool) for x in r)
            or not all(x > 0 for x in r)
            or abs(sum(r) - 1.0) > 1e-6):
        fail(f"split_ratios harus 3 angka positif berjumlah 1, didapat {r}")
    if cfg.direction not in ALLOWED_DIRECTIONS:
        fail(f"direction harus salah satu dari {ALLOWED_DIRECTIONS}, didapat {cfg.direction!r}")
    if cfg.label_resize_mode not in ALLOWED_LABEL_RESIZE:
        fail(f"label_resize_mode harus salah satu dari {ALLOWED_LABEL_RESIZE}, "
             f"didapat {cfg.label_resize_mode!r}")
    if cfg.batch_size < 1:
        fail(f"batch_size harus >= 1, didapat {cfg.batch_size}")
    if cfg.num_workers < 0:
        fail(f"num_workers harus >= 0, didapat {cfg.num_workers}")
    if cfg.subset_size is not None and cfg.subset_size < 1:
        fail(f"subset_size harus None atau >= 1, didapat {cfg.subset_size}")

    # ----- model -----
    if cfg.norm not in ALLOWED_NORMS:
        fail(f"norm harus salah satu dari {ALLOWED_NORMS}, didapat {cfg.norm!r}")
    for name in ("ngf", "ndf", "n_layers"):
        if getattr(cfg, name) < 1:
            fail(f"{name} harus >= 1, didapat {getattr(cfg, name)}")
    if not 0.0 <= cfg.dropout_p < 1.0:
        fail(f"dropout_p harus di [0, 1), didapat {cfg.dropout_p}")
''', baru=False)


In [ ]:
#@title config.py · 6/10: lanjutan
tulis('config.py', r'''
    # ----- training -----
    if cfg.epochs < 1:
        fail(f"epochs harus >= 1, didapat {cfg.epochs}")
    if cfg.lambda_l1 < 0:
        fail(f"lambda_l1 harus >= 0, didapat {cfg.lambda_l1}")
    if cfg.lr <= 0:
        fail(f"lr harus > 0, didapat {cfg.lr}")
    if cfg.device not in ALLOWED_DEVICES:
        fail(f"device harus salah satu dari {ALLOWED_DEVICES}, didapat {cfg.device!r}")
    # Tambahan di luar daftar spesifikasi (lihat penjelasan):
    if cfg.log_every < 1:
        fail(f"log_every harus >= 1, didapat {cfg.log_every}")
    if not cfg.run_name or any(c in cfg.run_name for c in "/\\"):
        fail(f"run_name harus nama folder tunggal yang tidak kosong, didapat {cfg.run_name!r}")
''', baru=False)


In [ ]:
#@title config.py · 7/10: set_seed
tulis('config.py', r'''

# ===========================================================================
# Seed dan device
# ===========================================================================
def set_seed(seed: int, deterministic: bool = False) -> None:
    """Seed semua sumber angka acak: random, numpy, torch (CPU) dan CUDA.

    Dengan deterministic=True: cudnn.deterministic = True dan
    cudnn.benchmark = False (cuDNN tidak lagi memilih algoritma tercepat secara
    otomatis, yang bisa berbeda antar-run). Training bisa sedikit lebih lambat.

    PENTING untuk dataset.py: dengan num_workers=0, crop/flip acak diambil
    dari RNG torch global di proses utama. Crop itu hanya reprodusibel jika
    set_seed dipanggil SEBELUM loader dibuat, dan urutan pemakaian RNG
    sesudahnya juga sama. Contoh: inisialisasi bobot model di CPU setelah
    loader dibuat ikut memakai RNG yang sama, sehingga mengubah ngf mengubah
    urutan crop. Dengan num_workers > 0, setiap worker mendapat seed dari
    generator DataLoader sehingga crop tidak bergantung pada RNG global.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)              # menurut dokumentasi PyTorch: CPU dan semua device
    torch.cuda.manual_seed_all(seed)     # eksplisit; aman dipanggil walau tanpa GPU
    torch.backends.cudnn.deterministic = deterministic
    if deterministic:
        torch.backends.cudnn.benchmark = False
''', baru=False)


In [ ]:
#@title config.py · 8/10: get_device
tulis('config.py', r'''

def get_device(cfg_or_name) -> torch.device:
    """Kembalikan torch.device dari Config (atau dari string "auto"/"cpu"/"cuda").

    "auto" -> cuda jika tersedia, selain itu cpu.
    "cuda" tanpa GPU -> error, supaya training tidak diam-diam jalan di CPU.
    """
    name = cfg_or_name.device if isinstance(cfg_or_name, Config) else cfg_or_name
    if name not in ALLOWED_DEVICES:
        raise ValueError(f"device harus salah satu dari {ALLOWED_DEVICES}, didapat {name!r}")
    if name == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if name == "cuda" and not torch.cuda.is_available():
        raise RuntimeError(
            "device='cuda' diminta tetapi CUDA tidak tersedia. Cek driver/instalasi "
            "PyTorch versi CUDA, atau pakai device='auto' / 'cpu'."
        )
    return torch.device(name)


def amp_enabled(cfg: Config, device: torch.device) -> bool:
    """AMP (float16) hanya dipakai di CUDA, dan hanya jika use_amp=True."""
    return bool(cfg.use_amp) and device.type == "cuda"


def run_dir(cfg: Config) -> Path:
    """Folder run: <checkpoint_dir>/<run_name>. Tidak membuat folder."""
    return Path(cfg.checkpoint_dir) / cfg.run_name
''', baru=False)


In [ ]:
#@title config.py · 9/10: save_config
tulis('config.py', r'''

# ===========================================================================
# Simpan / muat JSON
# ===========================================================================
def save_config(cfg: Config, folder) -> Path:
    """Tulis cfg ke <folder>/config.json (folder dibuat jika belum ada)."""
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / CONFIG_FILENAME
    path.write_text(json.dumps(dataclasses.asdict(cfg), indent=2), encoding="utf-8")
    return path


def load_config(path, allow_missing: bool = False) -> Config:
    """Baca Config dari JSON.

    - Key yang tidak dikenal -> error (biasanya salah ketik, misalnya "lamda_l1").
    - Key yang hilang -> error secara default, karena config sebuah run harus
      lengkap. allow_missing=True mengisi key yang hilang dengan default
      (untuk memuat run lama setelah field baru ditambahkan).
    - split_ratios kembali menjadi tuple lewat __post_init__.
    """
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise ValueError(f"{path}: isi JSON harus objek, didapat {type(data).__name__}")
    known = {f.name for f in fields(Config)}
    unknown = sorted(set(data) - known)
    if unknown:
        raise ValueError(f"{path}: key tidak dikenal {unknown}")
    missing = sorted(known - set(data))
    if missing and not allow_missing:
        raise ValueError(f"{path}: key hilang {missing} (pakai allow_missing=True untuk run lama)")
    return Config(**data)
''', baru=False)


In [ ]:
#@title config.py · 10/10: get_preset
tulis('config.py', r'''

# ===========================================================================
# Preset ablation
# ===========================================================================
ABLATION_EPOCHS = 30

# Hanya field yang BERBEDA dari baseline. Semua preset juga mendapat
# epochs=ABLATION_EPOCHS dan run_name=<nama preset>; seed tetap sama dengan base.
# n_layers=3, lr=2e-4, dan batch_size=2 tidak punya preset sendiri karena sama
# dengan baseline (lambda100).
#
# Catatan tuning batch_size: dengan jumlah epoch yang sama, batch 4 mendapat
# setengah jumlah update dibanding batch 2 (dan batch 1 dua kalinya). Bandingkan
# hasilnya dengan mencatat juga jumlah iterasi (kolom step di log).
PRESETS = MappingProxyType({
    # ablation arsitektur dan loss
    "lambda10": MappingProxyType({"lambda_l1": 10.0}),
    "lambda100": MappingProxyType({}),            # baseline
    "lambda200": MappingProxyType({"lambda_l1": 200.0}),
    "noskip": MappingProxyType({"use_skip": False}),
    "nlayers1": MappingProxyType({"n_layers": 1}),
    # tuning hyperparameter (panduan proyek: learning rate dan batch size)
    "lr1e-4": MappingProxyType({"lr": 1e-4}),
    "lr4e-4": MappingProxyType({"lr": 4e-4}),
    "batch1": MappingProxyType({"batch_size": 1}),
    "batch4": MappingProxyType({"batch_size": 4}),
})


def get_preset(name: str, base: typing.Optional[Config] = None) -> Config:
    """Config untuk satu run ablation, diturunkan dari `base` (default: Config())."""
    if name not in PRESETS:
        raise ValueError(f"Preset {name!r} tidak dikenal. Pilihan: {sorted(PRESETS)}")
    base = Config() if base is None else base
    return dataclasses.replace(base, epochs=ABLATION_EPOCHS, run_name=name, **PRESETS[name])
''', baru=False)


### `losses.py`

Loss adversarial (BCE non-saturating) dan L1. 4 sel, 107 baris. Isi sel → 1: `AdversarialLoss` · 2: `__init__` · 3: `discriminator_loss` · 4: `generator_loss`

In [ ]:
#@title losses.py · 1/4: AdversarialLoss
tulis('losses.py', r'''"""
losses.py

Fungsi loss Pix2Pix:

    loss_D = d_loss_scale * [ BCE(D(x, y), 1) + BCE(D(x, G(x)), 0) ]
    loss_G = BCE(D(x, G(x)), 1) + lambda_l1 * L1(G(x), y)

x = kondisi (label), y = target (foto), G(x) = foto palsu.
Nilai default lambda_l1 = 100 dan d_loss_scale = 0.5 diatur di config.py.

Pembagian tanggung jawab dengan train.py:
- losses.py hanya MENGHITUNG loss dari logit dan gambar yang diberikan.
- train.py yang memastikan D dipanggil terpisah untuk real dan fake, dan
  bahwa fake di-detach() saat update D. losses.py tidak meng-detach apa pun,
  supaya aliran gradien tetap terlihat jelas di train.py.

Semua loss dihitung dalam float32 (input di-cast dengan .float()). Di bawah
AMP, logit dan gambar dari model bisa berupa float16; menghitung log/sigmoid
dan selisih kecil dalam float16 rawan kehilangan presisi. Dengan cast eksplisit,
hasilnya tidak bergantung pada daftar operasi autocast PyTorch.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class AdversarialLoss(nn.Module):
    """BCE antara logit PatchGAN dan target "real" (1) atau "fake" (0).

    Target dibuat dengan ukuran yang SAMA dengan peta logit (misalnya
    [B, 1, 30, 30] untuk n_layers=3 di 256, atau [B, 1, 126, 126] untuk
    n_layers=1), sehingga setiap patch dinilai terhadap targetnya sendiri.
    Rata-rata diambil atas batch dan semua patch.

    BCEWithLogitsLoss menggabungkan sigmoid + BCE dalam satu rumus yang stabil
    secara numerik, jadi D tidak boleh memakai Sigmoid di output.
    """
''', baru=True)


In [ ]:
#@title losses.py · 2/4: __init__
tulis('losses.py', r'''
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits: torch.Tensor, target_is_real: bool) -> torch.Tensor:
        if not isinstance(target_is_real, bool):
            raise TypeError(f"target_is_real harus bool, didapat {type(target_is_real).__name__}")
        logits = logits.float()
        target = torch.ones_like(logits) if target_is_real else torch.zeros_like(logits)
        return self.bce(logits, target)   # skalar (0-dim)


# Satu instance dipakai bersama: AdversarialLoss tidak punya parameter atau state.
_adv = AdversarialLoss()
''', baru=False)


In [ ]:
#@title losses.py · 3/4: discriminator_loss
tulis('losses.py', r'''

def discriminator_loss(d_real_logits: torch.Tensor, d_fake_logits: torch.Tensor,
                       scale: float = 0.5) -> tuple:
    """Loss D: pasangan asli harus dinilai real, pasangan palsu dinilai fake.

    Args:
        d_real_logits: D(cond, real), shape [B, 1, h, w].
        d_fake_logits: D(cond, fake.detach()), shape sama. detach dilakukan di train.py.
        scale: pengali total (d_loss_scale). 0.5 memperlambat belajar D relatif terhadap G.

    Returns:
        (total, {"d_real", "d_fake"}). Nilai dict berupa tensor 0-dim yang sudah
        di-detach (bukan float): train.py baru memanggil .item() saat logging,
        sehingga GPU tidak perlu sinkron setiap iterasi.
    """
    if d_real_logits.shape != d_fake_logits.shape:
        raise ValueError(f"Shape logit real {tuple(d_real_logits.shape)} != "
                         f"fake {tuple(d_fake_logits.shape)}")
    if scale <= 0:
        raise ValueError(f"scale harus > 0, didapat {scale}")
    loss_real = _adv(d_real_logits, True)
    loss_fake = _adv(d_fake_logits, False)
    total = scale * (loss_real + loss_fake)
    return total, {"d_real": loss_real.detach(), "d_fake": loss_fake.detach()}
''', baru=False)


In [ ]:
#@title losses.py · 4/4: generator_loss
tulis('losses.py', r'''

def generator_loss(d_fake_logits: torch.Tensor, fake: torch.Tensor, real: torch.Tensor,
                   lambda_l1: float) -> tuple:
    """Loss G: menipu D + mendekati target piksel demi piksel.

    Bagian adversarial memakai bentuk NON-SATURATING: fake diberi label "real",
    yaitu meminimalkan -log D(x, G(x)). Bentuk asli minimax, log(1 - D(x, G(x))),
    gradiennya hampir nol justru saat D yakin gambar palsu (awal training), sehingga
    G sulit belajar. Bentuk non-saturating memberi gradien besar di kondisi itu.

    Args:
        d_fake_logits: D(cond, fake) TANPA detach, supaya gradien mengalir ke G.
        fake, real: [B, C, H, W] di rentang [-1, 1].
        lambda_l1: bobot L1 (config.lambda_l1). 0 = hanya loss adversarial.

    Returns:
        (total, {"g_adv", "g_l1"}), nilai dict berupa tensor 0-dim yang di-detach.
        g_l1 adalah L1 SEBELUM dikali lambda, supaya mudah dibandingkan antar-run.
    """
    if fake.shape != real.shape:
        raise ValueError(f"Shape fake {tuple(fake.shape)} != real {tuple(real.shape)}")
    if lambda_l1 < 0:
        raise ValueError(f"lambda_l1 harus >= 0, didapat {lambda_l1}")
    loss_adv = _adv(d_fake_logits, True)            # non-saturating
    loss_l1 = F.l1_loss(fake.float(), real.float())  # rata-rata |fake - real| per piksel
    total = loss_adv + lambda_l1 * loss_l1
    return total, {"g_adv": loss_adv.detach(), "g_l1": loss_l1.detach()}
''', baru=False)


### `train.py`

Loop training: update D lalu G, validasi, grid sampel, checkpoint, resume. 12 sel, 413 baris. Isi sel → 1: `import dan konstanta` · 2: `build_models` · 3: `_backward_and_step` · 4: `Parameter D dibekukan sementara: gradien tetap mengalir LEWAT D ke fake` · 5: `evaluate_val` · 6: `save_checkpoint` · 7: `LossWindow` · 8: `_append_csv` · 9: `train` · 10: `log_window` · 11: `parse_overrides` · 12: `main`

In [ ]:
#@title train.py · 1/12: import dan konstanta
tulis('train.py', r'''"""
train.py

Loop training Pix2Pix. Satu iterasi:

    1. Update D: D(cond, real) dan D(cond, fake.detach()), dipanggil TERPISAH.
    2. Update G: D(cond, fake) TANPA detach, plus L1(fake, real).

Contoh (dari root repo):
    python train.py --set data_root="C:/.../CMP_facade_DB_base/base"
    python train.py --preset lambda10 --set data_root=data/base
    python train.py --config runs/main/config.json --resume
    python train.py --set epochs=5 --set subset_size=8          # overfit test

Isi folder run (runs/<run_name>/):
    config.json        config lengkap run ini
    log_train.csv      rata-rata loss per log_every iterasi (kolom n_iter)
    log_val.csv        L1 validasi per epoch
    samples/epoch_XXX.png   grid tetap A | G(A) | B dari 4 gambar val pertama
    last.pt, best.pt   checkpoint terakhir dan dengan L1 validasi terbaik

Catatan Windows: DataLoader dengan num_workers > 0 memakai proses "spawn", yang
meng-import ulang file ini di setiap worker. Karena itu semua kode yang
menjalankan training berada di dalam main(), dan main() hanya dipanggil dari
`if __name__ == "__main__":`.
"""

from __future__ import annotations

import argparse
import csv
import dataclasses
import json
import os
import time
from pathlib import Path

import numpy as np
import torch
from PIL import Image

from config import (
    Config,
    amp_enabled,
    get_device,
    get_preset,
    load_config,
    run_dir,
    save_config,
    set_seed,
)
from dataset import denormalize, get_dataloader
from losses import discriminator_loss, generator_loss
from models.discriminator import PatchDiscriminator
from models.generator import UNetGenerator

N_SAMPLES = 4   # jumlah pasangan val di grid sampel

# Field yang boleh berbeda saat melanjutkan run (resume). Field lain, misalnya
# arsitektur, lr, atau lambda, harus sama dengan isi checkpoint.
RESUME_MUTABLE = {"epochs", "num_workers", "pin_memory", "log_every", "device",
                  "checkpoint_dir", "data_root", "manifest_path"}
''', baru=True)


In [ ]:
#@title train.py · 2/12: build_models
tulis('train.py', r'''

# ===========================================================================
# Membangun model dan optimizer dari Config
# ===========================================================================
def build_models(cfg: Config, device: torch.device) -> tuple:
    """G dan D dengan argumen yang dipetakan satu per satu dari Config."""
    G = UNetGenerator(
        in_ch=3, out_ch=3, ngf=cfg.ngf, img_size=cfg.img_size,
        use_dropout=cfg.use_dropout, use_skip=cfg.use_skip, norm=cfg.norm,
        n_dropout=cfg.n_dropout, dropout_p=cfg.dropout_p, max_mult=cfg.max_mult,
        leaky_slope=cfg.leaky_slope, init_std=cfg.init_std,
    )
    D = PatchDiscriminator(
        in_ch=6, ndf=cfg.ndf, n_layers=cfg.n_layers, norm=cfg.norm,
        max_mult=cfg.max_mult, leaky_slope=cfg.leaky_slope, init_std=cfg.init_std,
    )
    return G.to(device), D.to(device)


def build_optimizers(cfg: Config, G, D) -> tuple:
    betas = (cfg.beta1, cfg.beta2)
    return (torch.optim.Adam(G.parameters(), lr=cfg.lr, betas=betas),
            torch.optim.Adam(D.parameters(), lr=cfg.lr, betas=betas))


def make_grad_scaler(enabled: bool):
    """GradScaler untuk AMP, atau None jika AMP mati.

    torch.amp.GradScaler("cuda") ada di PyTorch baru; torch.cuda.amp.GradScaler
    (nama lama, sudah usang) dipakai sebagai cadangan untuk versi lama.
    """
    if not enabled:
        return None
    if hasattr(torch.amp, "GradScaler"):
        return torch.amp.GradScaler("cuda")
    return torch.cuda.amp.GradScaler()
''', baru=False)


In [ ]:
#@title train.py · 3/12: _backward_and_step
tulis('train.py', r'''

# ===========================================================================
# Satu langkah training
# ===========================================================================
def _backward_and_step(loss, optimizer, scaler):
    """Tanpa AMP: backward + step. Dengan AMP: loss diskalakan agar gradien fp16
    tidak underflow; step dilewati otomatis jika gradien berisi inf/NaN."""
    if scaler is None:
        loss.backward()
        optimizer.step()
    else:
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()


def train_step(G, D, opt_g, opt_d, scaler_g, scaler_d, cond, real, *,
               amp: bool, lambda_l1: float, d_loss_scale: float) -> dict:
    """Satu iterasi: update D lalu update G. Mengembalikan komponen loss
    (tensor 0-dim yang sudah di-detach; .item() dipanggil hanya saat logging)."""
    device_type = cond.device.type

    # ----- 1. Update D -----
    opt_d.zero_grad(set_to_none=True)
    with torch.autocast(device_type=device_type, dtype=torch.float16, enabled=amp):
        fake = G(cond)                          # graph G disimpan untuk langkah 2
        d_real = D(cond, real)                  # panggilan TERPISAH: real ...
        d_fake = D(cond, fake.detach())         # ... dan fake; detach -> G tidak ikut
        loss_d, parts_d = discriminator_loss(d_real, d_fake, scale=d_loss_scale)
    _backward_and_step(loss_d, opt_d, scaler_d)
''', baru=False)


In [ ]:
#@title train.py · 4/12: Parameter D dibekukan sementara: gradien tetap mengalir LEWAT D ke fake
tulis('train.py', r'''
    # ----- 2. Update G -----
    # Parameter D dibekukan sementara: gradien tetap mengalir LEWAT D ke fake
    # (ke G), tetapi tidak dihitung UNTUK bobot D. Hemat memori dan waktu, dan
    # gradien D dari langkah G tidak tercampur.
    D.requires_grad_(False)
    try:
        opt_g.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device_type, dtype=torch.float16, enabled=amp):
            d_fake_for_g = D(cond, fake)        # TANPA detach
            loss_g, parts_g = generator_loss(d_fake_for_g, fake, real, lambda_l1)
        _backward_and_step(loss_g, opt_g, scaler_g)
    finally:
        D.requires_grad_(True)

    return {"d_loss": loss_d.detach(), **parts_d, "g_loss": loss_g.detach(), **parts_g}
''', baru=False)


In [ ]:
#@title train.py · 5/12: evaluate_val
tulis('train.py', r'''

# ===========================================================================
# Evaluasi dan sampel
# ===========================================================================
@torch.no_grad()
def evaluate_val(G, loader, device) -> float:
    """Rata-rata L1 per piksel antara G(A) dan B di split val.

    G memakai mode eval (BatchNorm memakai running stats, Dropout mati).
    Mode G dikembalikan seperti semula setelahnya.
    """
    was_training = G.training
    G.eval()
    total, count = 0.0, 0
    for batch in loader:
        a, b = batch["A"].to(device), batch["B"].to(device)
        fake = G(a)
        total += (fake.float() - b).abs().sum().item()
        count += b.numel()
    G.train(was_training)
    return total / max(count, 1)


@torch.no_grad()
def save_sample_grid(G, a, b, path: Path) -> Path:
    """Simpan grid: setiap baris A | G(A) | B (uint8 PNG). Mode G dikembalikan."""
    was_training = G.training
    G.eval()
    fake = G(a)
    G.train(was_training)
    rows = [torch.cat([denormalize(x) for x in (a[i], fake[i], b[i])], dim=2)   # sambung lebar
            for i in range(a.shape[0])]
    grid = torch.cat(rows, dim=1).permute(1, 2, 0).numpy()                      # CHW -> HWC
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(np.ascontiguousarray(grid)).save(path)
    return path
''', baru=False)


In [ ]:
#@title train.py · 6/12: save_checkpoint
tulis('train.py', r'''

# ===========================================================================
# Checkpoint
# ===========================================================================
def save_checkpoint(path: Path, *, G, D, opt_g, opt_d, scaler_g, scaler_d,
                    epoch: int, step: int, best_val: float, cfg: Config) -> None:
    state = {
        "G": G.state_dict(), "D": D.state_dict(),
        "opt_g": opt_g.state_dict(), "opt_d": opt_d.state_dict(),
        "scaler_g": scaler_g.state_dict() if scaler_g else None,
        "scaler_d": scaler_d.state_dict() if scaler_d else None,
        "epoch": epoch, "step": step, "best_val": best_val,
        "config": dataclasses.asdict(cfg),
    }
    # Tulis ke file sementara lalu ganti nama: checkpoint tidak pernah setengah
    # jadi walaupun proses terhenti saat menyimpan.
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(state, tmp)
    os.replace(tmp, path)


def load_checkpoint(path: Path, device: torch.device) -> dict:
    # weights_only=True: hanya tensor dan tipe dasar yang dimuat, tanpa
    # menjalankan kode sembarang dari file checkpoint.
    return torch.load(path, map_location=device, weights_only=True)


def check_resume_compatible(saved: dict, cfg: Config) -> None:
    current = dataclasses.asdict(cfg)
    diff = {k: (saved.get(k), current[k]) for k in current
            if k not in RESUME_MUTABLE and _norm(saved.get(k)) != _norm(current[k])}
    if diff:
        raise ValueError(f"Config berbeda dari checkpoint untuk field {diff}. "
                         "Resume hanya boleh mengubah: " + ", ".join(sorted(RESUME_MUTABLE)))


def _norm(v):
    return list(v) if isinstance(v, tuple) else v   # JSON/torch.save: tuple vs list
''', baru=False)


In [ ]:
#@title train.py · 7/12: LossWindow
tulis('train.py', r'''

# ===========================================================================
# Log CSV
# ===========================================================================
TRAIN_FIELDS = ["epoch", "step", "n_iter", "d_loss", "d_real", "d_fake",
                "g_loss", "g_adv", "g_l1", "sec"]
VAL_FIELDS = ["epoch", "step", "val_l1", "epoch_sec"]


class LossWindow:
    """Kumpulkan loss beberapa iterasi lalu catat RATA-RATANYA.

    Satu titik di grafik = rata-rata `n_iter` iterasi (biasanya log_every), bukan
    nilai satu batch. Dengan batch 2, nilai satu batch sangat berisik sehingga
    grafiknya sulit dibaca. Penjumlahan dilakukan pada tensor (tetap di GPU,
    tanpa .item()), jadi GPU hanya perlu sinkron saat baris log ditulis.
    """

    def __init__(self):
        self.reset()

    def reset(self):
        self.sums, self.n, self.t0 = {}, 0, time.perf_counter()

    def add(self, parts: dict):
        for k, v in parts.items():
            self.sums[k] = self.sums.get(k, 0) + v
        self.n += 1

    def pop_row(self, epoch: int, step: int) -> dict:
        row = {"epoch": epoch, "step": step, "n_iter": self.n,
               **{k: round(float(v / self.n), 5) for k, v in self.sums.items()},
               "sec": round(time.perf_counter() - self.t0, 2)}
        self.reset()
        return row
''', baru=False)


In [ ]:
#@title train.py · 8/12: _append_csv
tulis('train.py', r'''

def _append_csv(path: Path, fields: list, row: dict) -> None:
    new = not path.exists()
    with open(path, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        if new:
            w.writeheader()
        w.writerow(row)
''', baru=False)


In [ ]:
#@title train.py · 9/12: train
tulis('train.py', r'''

# ===========================================================================
# Loop training
# ===========================================================================
def train(cfg: Config, *, resume: bool = False, deterministic: bool = False,
          max_steps: int | None = None) -> dict:
    """Jalankan training sesuai cfg. Mengembalikan ringkasan run.

    max_steps: batasi jumlah iterasi total (untuk uji cepat); None = semua epoch.
    """
    set_seed(cfg.seed, deterministic=deterministic)   # sebelum model dan loader dibuat
    device = get_device(cfg)
    amp = amp_enabled(cfg, device)
    out = run_dir(cfg)
    out.mkdir(parents=True, exist_ok=True)

    G, D = build_models(cfg, device)
    opt_g, opt_d = build_optimizers(cfg, G, D)
    scaler_g, scaler_d = make_grad_scaler(amp), make_grad_scaler(amp)
    train_loader = get_dataloader(cfg, "train")
    val_loader = get_dataloader(cfg, "val")

    # Sampel tetap: N_SAMPLES pasangan val pertama (tanpa augmentasi), sama tiap epoch.
    val_ds = val_loader.dataset
    idx = range(min(N_SAMPLES, len(val_ds)))
    sample_a = torch.stack([val_ds[i]["A"] for i in idx]).to(device)
    sample_b = torch.stack([val_ds[i]["B"] for i in idx]).to(device)

    start_epoch, step, best_val = 1, 0, float("inf")
    if resume:
        ckpt_path = out / "last.pt"
        if not ckpt_path.exists():
            raise FileNotFoundError(f"Tidak ada checkpoint untuk resume: {ckpt_path}")
        ckpt = load_checkpoint(ckpt_path, device)
        check_resume_compatible(ckpt["config"], cfg)
        G.load_state_dict(ckpt["G"]); D.load_state_dict(ckpt["D"])
        opt_g.load_state_dict(ckpt["opt_g"]); opt_d.load_state_dict(ckpt["opt_d"])
        if scaler_g and ckpt["scaler_g"]:
            scaler_g.load_state_dict(ckpt["scaler_g"]); scaler_d.load_state_dict(ckpt["scaler_d"])
        start_epoch, step, best_val = ckpt["epoch"] + 1, ckpt["step"], ckpt["best_val"]
        print(f"Resume dari epoch {ckpt['epoch']} (step {step}, val L1 terbaik {best_val:.4f})")
    save_config(cfg, out)

    print(f"Run '{cfg.run_name}' di {out} | device={device} | AMP={amp} | "
          f"train={len(train_loader.dataset)} val={len(val_ds)} | "
          f"G={G.num_parameters():,} D={D.num_parameters():,} parameter")
''', baru=False)


In [ ]:
#@title train.py · 10/12: log_window
tulis('train.py', r'''
    def log_window(epoch_now):
        row = window.pop_row(epoch_now, step)
        _append_csv(out / "log_train.csv", TRAIN_FIELDS, row)
        print(f"  epoch {epoch_now} step {step} (rata-rata {row['n_iter']} iterasi): "
              f"D {row['d_loss']:.3f} | G adv {row['g_adv']:.3f} L1 {row['g_l1']:.3f}")

    window = LossWindow()
    epoch = start_epoch - 1
    for epoch in range(start_epoch, cfg.epochs + 1):
        G.train(); D.train()
        t_epoch = time.perf_counter()
        for batch in train_loader:
            cond = batch["A"].to(device, non_blocking=True)
            real = batch["B"].to(device, non_blocking=True)
            parts = train_step(G, D, opt_g, opt_d, scaler_g, scaler_d, cond, real,
                               amp=amp, lambda_l1=cfg.lambda_l1, d_loss_scale=cfg.d_loss_scale)
            step += 1
            window.add(parts)
            if window.n == cfg.log_every:      # jendela penuh: catat rata-ratanya
                log_window(epoch)
            if max_steps is not None and step >= max_steps:
                break

        val_l1 = evaluate_val(G, val_loader, device)
        epoch_sec = time.perf_counter() - t_epoch
        _append_csv(out / "log_val.csv", VAL_FIELDS,
                    {"epoch": epoch, "step": step, "val_l1": round(val_l1, 5),
                     "epoch_sec": round(epoch_sec, 1)})
        save_sample_grid(G, sample_a, sample_b, out / "samples" / f"epoch_{epoch:03d}.png")

        ckpt_kwargs = dict(G=G, D=D, opt_g=opt_g, opt_d=opt_d, scaler_g=scaler_g,
                           scaler_d=scaler_d, epoch=epoch, step=step, cfg=cfg)
        if val_l1 < best_val:
            best_val = val_l1
            save_checkpoint(out / "best.pt", best_val=best_val, **ckpt_kwargs)
        save_checkpoint(out / "last.pt", best_val=best_val, **ckpt_kwargs)
        print(f"Epoch {epoch}/{cfg.epochs}: val L1 {val_l1:.4f} (terbaik {best_val:.4f}), {epoch_sec:.1f} dtk")

        if max_steps is not None and step >= max_steps:
            break

    if window.n:   # sisa iterasi yang belum genap satu jendela di akhir training
        log_window(epoch)
    return {"run_dir": str(out), "last_epoch": epoch, "steps": step, "best_val_l1": best_val}
''', baru=False)


In [ ]:
#@title train.py · 11/12: parse_overrides
tulis('train.py', r'''

# ===========================================================================
# CLI
# ===========================================================================
def parse_overrides(pairs: list) -> dict:
    """['epochs=5', 'use_amp=false', 'data_root=C:/x'] -> dict bertipe.

    Nilai dibaca sebagai JSON jika bisa (angka, true/false, null, list),
    selain itu dianggap string. Nama field divalidasi oleh Config.
    """
    known = {f.name for f in dataclasses.fields(Config)}
    out = {}
    for pair in pairs:
        if "=" not in pair:
            raise ValueError(f"--set harus berbentuk key=value, didapat {pair!r}")
        key, raw = pair.split("=", 1)
        key = key.strip()
        if key not in known:
            raise ValueError(f"Field config tidak dikenal: {key!r}")
        try:
            value = json.loads(raw)
        except json.JSONDecodeError:
            value = raw
        if key == "split_ratios" and isinstance(value, list):
            value = tuple(value)
        out[key] = value
    return out
''', baru=False)


In [ ]:
#@title train.py · 12/12: main
tulis('train.py', r'''

def main(argv=None) -> dict:
    p = argparse.ArgumentParser(description="Training Pix2Pix.")
    src = p.add_mutually_exclusive_group()
    src.add_argument("--config", help="muat Config dari JSON (misalnya runs/main/config.json)")
    src.add_argument("--preset", help="preset ablation dari config.get_preset")
    p.add_argument("--set", action="append", default=[], metavar="KEY=VALUE",
                   help="ganti satu field config; boleh berulang")
    p.add_argument("--resume", action="store_true", help="lanjutkan dari runs/<run_name>/last.pt")
    p.add_argument("--deterministic", action="store_true", help="cuDNN deterministik (lebih lambat)")
    p.add_argument("--max-steps", type=int, default=None, help="batas iterasi total (uji cepat)")
    args = p.parse_args(argv)

    if args.config:
        cfg = load_config(args.config)
    elif args.preset:
        cfg = get_preset(args.preset)
    else:
        cfg = Config()
    cfg = dataclasses.replace(cfg, **parse_overrides(args.set))
    return train(cfg, resume=args.resume, deterministic=args.deterministic, max_steps=args.max_steps)


if __name__ == "__main__":
    main()
''', baru=False)


### `eval.py`

Evaluasi split test: L1, PSNR, SSIM (ditulis sendiri), grid, perbandingan antar-run. 7 sel, 229 baris. Isi sel → 1: `import dan konstanta` · 2: `to_unit` · 3: `ssim_per_image` · 4: `enable_dropout_only` · 5: `evaluate_run` · 6: `mean_std` · 7: `main`

In [ ]:
#@title eval.py · 1/7: import dan konstanta
tulis('eval.py', r'''"""
eval.py

Evaluasi generator yang sudah dilatih pada split TEST (yang tidak dipakai
saat training maupun pemilihan checkpoint).

Metrik per gambar, dihitung sendiri tanpa jaringan pretrained:
- L1   : rata-rata |G(A) - B| di rentang [-1, 1] (sama dengan val L1 saat training).
- PSNR : 10 * log10(1 / MSE), dengan gambar dipetakan ke [0, 1]. Makin tinggi makin baik.
- SSIM : structural similarity (Wang dkk., 2004), jendela Gaussian 11x11,
         sigma 1.5, C1 = (0.01)^2, C2 = (0.03)^2, dihitung per channel lalu dirata-rata.

Penting: ketiga metrik membandingkan piksel per piksel dengan foto asli. Model
yang menghasilkan gambar buram sering mendapat L1/PSNR bagus walaupun tidak
realistis. Karena itu grid gambar tetap wajib dilihat bersama angka-angka ini.

Contoh:
    python eval.py --runs runs/main
    python eval.py --runs runs/main runs/lambda10 runs/noskip --ckpt best
    python eval.py --runs runs/main --data-root "C:/.../base" --dropout-eval

Output per run (di folder run): eval_<split>_<ckpt>.csv (per gambar) dan
eval_<split>_<ckpt>_grid.png, misalnya eval_test_best.csv dan eval_test_last.csv.

Pilih checkpoint dengan hati-hati: best.pt dipilih dari val L1 terendah, dan
pada GAN model awal yang buram sering punya L1 paling rendah. Bandingkan best
dan last, bersama grid gambarnya.
Output gabungan: --out (default runs/perbandingan_test.csv), satu baris per run.
"""

from __future__ import annotations

import argparse
import csv
import dataclasses
import math
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

from config import Config
import numpy as np
from PIL import Image

from dataset import denormalize, get_dataloader
from train import build_models, load_checkpoint

N_GRID = 6   # jumlah gambar test di grid
''', baru=True)


In [ ]:
#@title eval.py · 2/7: to_unit
tulis('eval.py', r'''

# ===========================================================================
# Metrik
# ===========================================================================
def to_unit(x: torch.Tensor) -> torch.Tensor:
    """[-1, 1] -> [0, 1], float32."""
    return (x.float().clamp(-1, 1) + 1) / 2


def l1_per_image(fake: torch.Tensor, real: torch.Tensor) -> torch.Tensor:
    """[B] rata-rata |fake - real| per gambar, di rentang [-1, 1]."""
    return (fake.float() - real.float()).abs().mean(dim=(1, 2, 3))


def psnr_per_image(fake: torch.Tensor, real: torch.Tensor) -> torch.Tensor:
    """[B] PSNR dalam dB dengan rentang data 1 (gambar di [0, 1]).
    Gambar identik (MSE = 0) menghasilkan +inf."""
    mse = ((to_unit(fake) - to_unit(real)) ** 2).mean(dim=(1, 2, 3))
    return 10 * torch.log10(1.0 / mse)


def _gaussian_window(size: int, sigma: float, channels: int, device) -> torch.Tensor:
    coords = torch.arange(size, dtype=torch.float32, device=device) - (size - 1) / 2
    g = torch.exp(-(coords.double() ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    window_2d = g[:, None] * g[None, :]                       # [size, size], jumlah = 1
    return window_2d.expand(channels, 1, size, size).contiguous()
''', baru=False)


In [ ]:
#@title eval.py · 3/7: ssim_per_image
tulis('eval.py', r'''

def ssim_per_image(fake: torch.Tensor, real: torch.Tensor, window: int = 11,
                   sigma: float = 1.5) -> torch.Tensor:
    """[B] SSIM rata-rata per gambar (gambar dipetakan ke [0, 1]).

    mu = rata-rata lokal berbobot Gaussian; sigma^2 dan kovarians dihitung dari
    E[x^2] - mu^2 dan E[xy] - mu_x mu_y. Konvolusi tanpa padding (hanya posisi
    di mana jendela muat penuh), dilakukan per channel (groups = channel).

    Dihitung dalam float64: E[x^2] - mu^2 adalah pengurangan dua angka yang hampir
    sama, dan di float32 galatnya (~1e-7) tidak kecil dibanding C2 = 9e-4.
    """
    x, y = to_unit(fake).double(), to_unit(real).double()
    c = x.shape[1]
    if min(x.shape[-2:]) < window:
        raise ValueError(f"Gambar {tuple(x.shape[-2:])} lebih kecil dari jendela SSIM {window}")
    w = _gaussian_window(window, sigma, c, x.device)

    def blur(t):
        return F.conv2d(t, w, groups=c)

    mu_x, mu_y = blur(x), blur(y)
    var_x = blur(x * x) - mu_x ** 2
    var_y = blur(y * y) - mu_y ** 2
    cov = blur(x * y) - mu_x * mu_y
    c1, c2 = 0.01 ** 2, 0.03 ** 2
    ssim_map = ((2 * mu_x * mu_y + c1) * (2 * cov + c2)) / \
               ((mu_x ** 2 + mu_y ** 2 + c1) * (var_x + var_y + c2))
    return ssim_map.mean(dim=(1, 2, 3)).float()
''', baru=False)


In [ ]:
#@title eval.py · 4/7: enable_dropout_only
tulis('eval.py', r'''

# ===========================================================================
# Evaluasi satu run
# ===========================================================================
def enable_dropout_only(model: nn.Module) -> None:
    """Aktifkan HANYA modul Dropout (BatchNorm tetap memakai running stats).
    Untuk mencoba perilaku paper Pix2Pix yang memakai dropout juga saat test."""
    for m in model.modules():
        if isinstance(m, nn.Dropout):
            m.train()


def load_run(run: Path, ckpt: str, device, overrides: dict) -> tuple:
    """Muat config dan generator dari checkpoint sebuah run."""
    path = Path(run) / f"{ckpt}.pt"
    if not path.exists():
        raise FileNotFoundError(f"Checkpoint tidak ada: {path}")
    state = load_checkpoint(path, device)
    cfg = dataclasses.replace(Config(**state["config"]), **overrides)
    G, _ = build_models(cfg, device)
    G.load_state_dict(state["G"])
    return cfg, G, state
''', baru=False)


In [ ]:
#@title eval.py · 5/7: evaluate_run
tulis('eval.py', r'''

@torch.no_grad()
def evaluate_run(run: Path, *, split: str = "test", ckpt: str = "best", dropout_eval: bool = False,
                 device: torch.device | None = None, overrides: dict | None = None) -> dict:
    """Hitung metrik per gambar untuk satu run; simpan CSV per gambar dan grid."""
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    overrides = dict(overrides or {})
    overrides.setdefault("subset_size", None)           # evaluasi selalu memakai split penuh
    cfg, G, state = load_run(run, ckpt, device, overrides)
    G.eval()
    if dropout_eval:
        enable_dropout_only(G)
        torch.manual_seed(0)       # dropout acak: seed tetap agar hasil bisa diulang

    loader = get_dataloader(cfg, split)
    rows, grid = [], []
    for batch in loader:
        a, b = batch["A"].to(device), batch["B"].to(device)
        fake = G(a)
        for path, l1, ps, ss in zip(batch["path"], l1_per_image(fake, b),
                                    psnr_per_image(fake, b), ssim_per_image(fake, b)):
            rows.append({"path": Path(path).name, "l1": round(l1.item(), 5),
                         "psnr": round(ps.item(), 3), "ssim": round(ss.item(), 4)})
        for i in range(min(a.shape[0], N_GRID - len(grid))):
            # baris grid dari output yang SAMA dengan yang dinilai metrik
            grid.append(torch.cat([denormalize(t[i]) for t in (a, fake, b)], dim=2))

    run = Path(run)
    with open(run / f"eval_{split}_{ckpt}.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["path", "l1", "psnr", "ssim"])
        w.writeheader()
        w.writerows(rows)
    img = torch.cat(grid, dim=1).permute(1, 2, 0).numpy()      # baris: A | G(A) | B
    Image.fromarray(np.ascontiguousarray(img)).save(run / f"eval_{split}_{ckpt}_grid.png")
''', baru=False)


In [ ]:
#@title eval.py · 6/7: mean_std
tulis('eval.py', r'''
    def mean_std(key):
        vals = [r[key] for r in rows if math.isfinite(r[key])]
        if not vals:
            return float("nan"), float("nan")
        m = sum(vals) / len(vals)
        return m, (sum((v - m) ** 2 for v in vals) / len(vals)) ** 0.5

    summary = {"run": run.name, "ckpt": ckpt, "epoch": state["epoch"], "step": state["step"],
               "split": split, "n": len(rows), "dropout_eval": dropout_eval,
               "lambda_l1": cfg.lambda_l1, "use_skip": cfg.use_skip, "n_layers": cfg.n_layers,
               "lr": cfg.lr, "batch_size": cfg.batch_size}
    for key in ("l1", "psnr", "ssim"):
        m, s = mean_std(key)
        summary[f"{key}_mean"], summary[f"{key}_std"] = round(m, 5), round(s, 5)
    summary["best_val_l1"] = round(state["best_val"], 5)
    return summary


def write_summary(summaries: list, out: Path) -> Path:
    out = Path(out)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(summaries[0]))
        w.writeheader()
        w.writerows(summaries)
    return out
''', baru=False)


In [ ]:
#@title eval.py · 7/7: main
tulis('eval.py', r'''

# ===========================================================================
# CLI
# ===========================================================================
def main(argv=None) -> list:
    p = argparse.ArgumentParser(description="Evaluasi Pix2Pix di split test.")
    p.add_argument("--runs", nargs="+", required=True, help="folder run, misalnya runs/main")
    p.add_argument("--ckpt", default="best", choices=["best", "last"])
    p.add_argument("--split", default="test", choices=["test", "val"])
    p.add_argument("--dropout-eval", action="store_true", help="dropout aktif saat inferensi")
    p.add_argument("--data-root", default=None, help="ganti data_root dari checkpoint (pindah mesin)")
    p.add_argument("--num-workers", type=int, default=0)
    p.add_argument("--device", default=None, choices=["cpu", "cuda"])
    p.add_argument("--out", default="runs/perbandingan_test.csv")
    args = p.parse_args(argv)

    overrides = {"num_workers": args.num_workers, "pin_memory": False}
    if args.data_root:
        overrides["data_root"] = args.data_root
    device = torch.device(args.device) if args.device else None

    summaries = []
    for run in args.runs:
        s = evaluate_run(Path(run), split=args.split, ckpt=args.ckpt, dropout_eval=args.dropout_eval,
                         device=device, overrides=overrides)
        summaries.append(s)
        print(f"{s['run']:>12} (epoch {s['epoch']}, {s['ckpt']}): L1 {s['l1_mean']:.4f} | "
              f"PSNR {s['psnr_mean']:.2f} dB | SSIM {s['ssim_mean']:.4f} | n={s['n']}")
    out = write_summary(summaries, Path(args.out))
    print(f"Ringkasan disimpan ke {out}")
    return summaries


if __name__ == "__main__":
    main()
''', baru=False)


### `scripts/preview_pairs.py`

Skrip grid pratinjau A|B. 3 sel, 113 baris. Isi sel → 1: `import dan konstanta` · 2: `flatness` · 3: `main`

In [ ]:
#@title scripts/preview_pairs.py · 1/3: import dan konstanta
tulis('scripts/preview_pairs.py', r'''"""
scripts/preview_pairs.py

Simpan grid contoh pasangan (kolom kiri = A/kondisi, kolom kanan = B/target)
untuk memastikan secara visual bahwa A memang LABEL dan B memang FOTO.

Contoh (dari root repo):
    python scripts/preview_pairs.py --root data/facades --split train --direction AtoB
    python scripts/preview_pairs.py --root data/facades --split train --direction BtoA --augment

Skrip juga mencetak:
- "flatness" tiap kolom: proporsi piksel yang hampir sama dengan tetangga
  kanannya. Peta label (blok warna rata) biasanya jauh lebih rata daripada
  foto. Ini hanya PETUNJUK heuristik; keputusan akhir tetap dari melihat grid.
- hasil pemeriksaan tumpang tindih antar split yang tersedia.
"""

import argparse
import sys
from pathlib import Path

import torch
from PIL import Image, ImageDraw

# Agar `import dataset` berhasil saat skrip dijalankan dari mana pun.
sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from dataset import (  # noqa: E402
    SPLITS,
    PairedImageDataset,
    denormalize,
    find_split_overlap,
)

HEADER_H = 20  # tinggi baris judul kolom (piksel)
''', baru=True)


In [ ]:
#@title scripts/preview_pairs.py · 2/3: flatness
tulis('scripts/preview_pairs.py', r'''

def flatness(x: torch.Tensor, tol: int = 8) -> float:
    """Proporsi piksel yang selisihnya dengan tetangga kanan <= tol (skala 0..255).
    Toleransi menyerap noise kompresi JPEG pada peta label."""
    u8 = denormalize(x).int()                          # [3, H, W]
    diff = (u8[:, :, 1:] - u8[:, :, :-1]).abs().amax(dim=0)
    return (diff <= tol).float().mean().item()


def build_grid(samples: list, img_size: int) -> Image.Image:
    """samples: list dict {"A","B"}. Hasil: gambar [header + n baris] x 2 kolom."""
    gap = 4
    width = 2 * img_size + gap
    height = HEADER_H + len(samples) * (img_size + gap)
    grid = Image.new("RGB", (width, height), (255, 255, 255))
    draw = ImageDraw.Draw(grid)
    draw.text((4, 4), "A (kondisi)", fill=(0, 0, 0))
    draw.text((img_size + gap + 4, 4), "B (target)", fill=(0, 0, 0))
    for row, s in enumerate(samples):
        y = HEADER_H + row * (img_size + gap)
        for col, key in enumerate(("A", "B")):
            arr = denormalize(s[key]).permute(1, 2, 0).numpy()   # CHW -> HWC
            grid.paste(Image.fromarray(arr), (col * (img_size + gap), y))
    return grid
''', baru=False)


In [ ]:
#@title scripts/preview_pairs.py · 3/3: main
tulis('scripts/preview_pairs.py', r'''

def main(argv=None) -> Path:
    p = argparse.ArgumentParser(description="Simpan grid pasangan A|B untuk dicek visual.")
    p.add_argument("--root", required=True, help="folder dataset")
    p.add_argument("--split", default="train", choices=SPLITS)
    p.add_argument("--direction", default="AtoB", choices=("AtoB", "BtoA"))
    p.add_argument("--n", type=int, default=6, help="jumlah pasangan")
    p.add_argument("--img-size", type=int, default=256)
    p.add_argument("--load-size", type=int, default=286)
    p.add_argument("--augment", action="store_true",
                   help="tampilkan hasil augmentasi train (cek crop/flip sinkron)")
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--out", default="preview_pairs.png")
    args = p.parse_args(argv)

    torch.manual_seed(args.seed)
    ds = PairedImageDataset(args.root, args.split, args.direction, args.img_size,
                            args.load_size, augment=args.augment)
    n = min(args.n, len(ds))
    samples = [ds[i] for i in range(n)]

    out = Path(args.out)
    build_grid(samples, args.img_size).save(out)

    flat_a = sum(flatness(s["A"]) for s in samples) / n
    flat_b = sum(flatness(s["B"]) for s in samples) / n
    print(f"Grid disimpan ke: {out.resolve()}")
    print(f"Split '{args.split}' ({ds.split_source}), {len(ds)} pasangan, direction={args.direction}")
    print(f"Flatness rata-rata  A: {flat_a:.3f}   B: {flat_b:.3f}")
    if flat_a > flat_b:
        print("Petunjuk: A lebih rata -> kemungkinan A = label (sesuai tujuan).")
    else:
        print("Petunjuk: B lebih rata -> kemungkinan B = label. Coba direction sebaliknya.")
    print("Ini hanya heuristik. Pastikan dengan MELIHAT gambar grid.")

    # Pemeriksaan overlap antar split yang tersedia
    available = {}
    for split in SPLITS:
        try:
            available[split] = PairedImageDataset(args.root, split, args.direction,
                                                  args.img_size, args.load_size)
        except (ValueError, FileNotFoundError):
            pass
    if len(available) > 1:
        overlap = find_split_overlap(available)
        print(f"Cek overlap {list(available)}: path sama = {len(overlap['same_path'])}, "
              f"isi file identik = {len(overlap['same_content'])}")
    return out


if __name__ == "__main__":
    main()
''', baru=False)


### `scripts/measure_vram.py`

Skrip pengukur VRAM. 10 sel, 354 baris. Isi sel → 1: `import dan konstanta` · 2: `ComboResult` · 3: `train_step` · 4: `_cleanup` · 5: `_run_combo_inner` · 6: `Tanpa AMP biasanya cukup 1 iterasi. Dengan AMP bisa beberapa iterasi,` · 7: `run_combo` · 8: `write_csv` · 9: `parse_args` · 10: `main`

In [ ]:
#@title scripts/measure_vram.py · 1/10: import dan konstanta
tulis('scripts/measure_vram.py', r'''"""
scripts/measure_vram.py

Ukur puncak memori GPU satu langkah training Pix2Pix (update D lalu update G)
dengan data dummy (torch.randn), untuk setiap kombinasi:

    img_size {256, 128} x batch {1, 2, 4} x AMP {off, on} x ngf {64, 32}

Contoh (dari root repo, di laptop dengan GPU):
    python scripts/measure_vram.py
    python scripts/measure_vram.py --out hasil_vram.csv --iters 3
    python scripts/measure_vram.py --img-sizes 128 --batches 4 --amp off --cudnn-deterministic

Alur per kombinasi:
1. Pemanasan: ulangi langkah training sampai state Adam G dan D terbentuk
   (maks --max-warmup). Dengan AMP, GradScaler bisa melewati beberapa step awal.
2. Iterasi terukur (--iters), dengan state Adam sudah ada seperti saat training.

Yang dicatat per kombinasi:
- peak_alloc_mib    : torch.cuda.max_memory_allocated() sejak model dibuat,
                      termasuk pemanasan (training sungguhan juga harus lolos ini).
- peak_reserved_mib : torch.cuda.max_memory_reserved(), memori yang dipegang
                      caching allocator. Ini lebih dekat ke angka yang terlihat
                      di nvidia-smi / Task Manager, tetapi keduanya BELUM
                      termasuk CUDA context (beberapa ratus MiB, di luar PyTorch).
- steady_alloc_mib  : puncak khusus iterasi terukur. Jika jauh di bawah
                      peak_alloc_mib, lonjakan terjadi sekali di awal saja.
- adam_ready, warmup_iters, g_steps, d_steps: bukti bahwa state Adam sudah ada.

Skrip ini hanya meng-import model; file model tidak diubah.
Langkah training diambil dari train.train_step (yang memakai losses.py), jadi
yang diukur persis langkah yang dijalankan saat training. Di langkah update G,
parameter D dibekukan (requires_grad False), seperti di train.py.
"""

from __future__ import annotations

import argparse
import csv
import gc
import sys
import time
from dataclasses import asdict, dataclass, fields
from pathlib import Path

import torch

# Agar `import models...` berhasil saat skrip dijalankan dari mana pun.
sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from models.discriminator import PatchDiscriminator  # noqa: E402
from models.generator import UNetGenerator  # noqa: E402
from train import make_grad_scaler  # noqa: E402
from train import train_step as _train_step  # noqa: E402

MIB = 2 ** 20
''', baru=True)


In [ ]:
#@title scripts/measure_vram.py · 2/10: ComboResult
tulis('scripts/measure_vram.py', r'''

@dataclass
class ComboResult:
    img_size: int
    batch: int
    amp: bool
    ngf: int
    status: str                       # "ok", "OOM", atau "skipped"
    peak_alloc_mib: float | None      # puncak sejak model dibuat (termasuk pemanasan)
    peak_reserved_mib: float | None
    pct_of_total: float | None        # peak_reserved / total VRAM GPU, dalam %
    steady_alloc_mib: float | None    # puncak khusus iterasi terukur (setelah state Adam ada)
    iter_ms: float | None             # waktu iterasi terukur terakhir
    warmup_iters: int | None          # iterasi pemanasan sampai state Adam G dan D terbentuk
    adam_ready: bool | None           # True jika state Adam G dan D sudah terbentuk
    g_steps: int | None               # langkah Adam G yang benar-benar terjadi
    d_steps: int | None               # langkah Adam D yang benar-benar terjadi
    g_params: int | None
    d_params: int | None
    note: str
''', baru=False)


In [ ]:
#@title scripts/measure_vram.py · 3/10: train_step
tulis('scripts/measure_vram.py', r'''

# ---------------------------------------------------------------------------
# Satu langkah training: SAMA PERSIS dengan train.py (satu sumber kode)
# ---------------------------------------------------------------------------
def train_step(G, D, opt_g, opt_d, scaler_g, scaler_d, cond, real, amp, lambda_l1,
               d_loss_scale=0.5):
    """Memanggil train.train_step, lalu mengembalikan (loss_d, loss_g).

    Fungsi pembungkus ini tetap ada (dengan nama yang sama) supaya tes bisa
    menggantinya dengan versi palsu, misalnya untuk mensimulasikan OOM.
    """
    parts = _train_step(G, D, opt_g, opt_d, scaler_g, scaler_d, cond, real,
                        amp=amp, lambda_l1=lambda_l1, d_loss_scale=d_loss_scale)
    return parts["d_loss"], parts["g_loss"]


def _adam_ready(opt) -> bool:
    """True jika SEMUA parameter optimizer sudah punya state Adam (exp_avg).

    State dibuat pada optimizer.step() pertama yang benar-benar berjalan. Dengan
    AMP, GradScaler melewati step jika gradien berisi inf/NaN (biasanya di iterasi
    awal, saat faktor skala masih terlalu besar), sehingga state belum ada.
    """
    params = [p for group in opt.param_groups for p in group["params"]]
    return all(p in opt.state and "exp_avg" in opt.state[p] for p in params)


def _adam_steps(opt) -> int:
    """Jumlah step Adam yang benar-benar terjadi (minimum di antara parameter)."""
    steps = [float(st["step"]) for st in opt.state.values() if "step" in st]
    return int(min(steps)) if steps else 0


def _is_oom(err: BaseException) -> bool:
    """OOM resmi, plus RuntimeError dari cuDNN/cuBLAS yang sebenarnya kehabisan memori."""
    if isinstance(err, torch.cuda.OutOfMemoryError):
        return True
    msg = str(err)
    return isinstance(err, RuntimeError) and ("out of memory" in msg or "ALLOC_FAILED" in msg)
''', baru=False)


In [ ]:
#@title scripts/measure_vram.py · 4/10: _cleanup
tulis('scripts/measure_vram.py', r'''

def _cleanup(device: torch.device):
    gc.collect()
    if device.type == "cuda":
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
''', baru=False)


In [ ]:
#@title scripts/measure_vram.py · 5/10: _run_combo_inner
tulis('scripts/measure_vram.py', r'''

# ---------------------------------------------------------------------------
# Satu kombinasi
# ---------------------------------------------------------------------------
def _run_combo_inner(args, img_size, batch, amp, ngf, device, result: dict):
    """Semua tensor dan model hidup di dalam fungsi ini, sehingga otomatis
    dibebaskan saat fungsi selesai (termasuk ketika terjadi OOM)."""
    torch.manual_seed(args.seed)
    G = UNetGenerator(ngf=ngf, img_size=img_size, use_dropout=args.use_dropout,
                      use_skip=args.use_skip, norm=args.g_norm).to(device).train()
    D = PatchDiscriminator(in_ch=6, ndf=args.ndf, n_layers=args.n_layers,
                           norm=args.d_norm).to(device).train()
    result["g_params"] = G.num_parameters()
    result["d_params"] = D.num_parameters()

    betas = (args.beta1, args.beta2)
    opt_g = torch.optim.Adam(G.parameters(), lr=args.lr, betas=betas)
    opt_d = torch.optim.Adam(D.parameters(), lr=args.lr, betas=betas)
    # Dua scaler terpisah: setiap optimizer punya siklus scale->step->update sendiri.
    scaler_g = make_grad_scaler(amp)
    scaler_d = make_grad_scaler(amp)

    cond = torch.randn(batch, 3, img_size, img_size, device=device).clamp_(-1, 1)
    real = torch.randn(batch, 3, img_size, img_size, device=device).clamp_(-1, 1)

    def step():
        return train_step(G, D, opt_g, opt_d, scaler_g, scaler_d, cond, real, amp, args.lambda_l1)
''', baru=False)


In [ ]:
#@title scripts/measure_vram.py · 6/10: Tanpa AMP biasanya cukup 1 iterasi. Dengan AMP bisa beberapa iterasi,
tulis('scripts/measure_vram.py', r'''
    # ----- Pemanasan: ulangi sampai state Adam G dan D terbentuk -----
    # Tanpa AMP biasanya cukup 1 iterasi. Dengan AMP bisa beberapa iterasi,
    # karena GradScaler melewati step selama gradien yang diskalakan masih inf.
    warm = 0
    while warm < args.max_warmup and not (_adam_ready(opt_g) and _adam_ready(opt_d)):
        step()
        warm += 1
    result["warmup_iters"] = warm

    if device.type == "cuda":
        torch.cuda.synchronize()
        # Simpan puncak sejauh ini, lalu reset agar puncak iterasi terukur tercatat terpisah.
        result["warm_peak"] = (torch.cuda.max_memory_allocated(), torch.cuda.max_memory_reserved())
        torch.cuda.reset_peak_memory_stats()

    # ----- Iterasi terukur (state Adam sudah ada) -----
    for _ in range(args.iters):
        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        loss_d, loss_g = step()
        if device.type == "cuda":
            torch.cuda.synchronize()
        result["iter_ms"] = (time.perf_counter() - t0) * 1000

    result["adam_ready"] = _adam_ready(opt_g) and _adam_ready(opt_d)
    result["g_steps"] = _adam_steps(opt_g)
    result["d_steps"] = _adam_steps(opt_d)
    notes = []
    if not result["adam_ready"]:
        notes.append(f"state Adam belum terbentuk setelah {warm} iterasi pemanasan; angka meremehkan memori")
    if not (torch.isfinite(loss_d) and torch.isfinite(loss_g)):
        notes.append("loss tidak finite")
    result["note"] = "; ".join(notes)
''', baru=False)


In [ ]:
#@title scripts/measure_vram.py · 7/10: run_combo
tulis('scripts/measure_vram.py', r'''

def run_combo(args, img_size, batch, amp, ngf, device) -> ComboResult:
    result = {"g_params": None, "d_params": None, "iter_ms": None, "warmup_iters": None,
              "adam_ready": None, "g_steps": None, "d_steps": None, "note": ""}
    if amp and device.type != "cuda":
        return ComboResult(img_size, batch, amp, ngf, "skipped", None, None, None, None,
                           None, None, None, None, None, None, None,
                           "AMP fp16 hanya diukur di CUDA")

    _cleanup(device)
    if device.type == "cuda":
        start_alloc = torch.cuda.memory_allocated()
        torch.cuda.reset_peak_memory_stats()   # puncak dihitung ulang dari sini
        if start_alloc > 0:
            result["note"] = f"sisa memori sebelum mulai: {start_alloc / MIB:.1f} MiB"

    status = "ok"
    try:
        _run_combo_inner(args, img_size, batch, amp, ngf, device, result)
    except Exception as err:  # noqa: BLE001 - hanya OOM yang ditelan, lainnya dilempar ulang
        if not _is_oom(err):
            raise
        status = "OOM"
        result["note"] = str(err).splitlines()[0][:120]

    peak_alloc = peak_reserved = pct = steady_alloc = None
    if device.type == "cuda":
        torch.cuda.synchronize()
        # Puncak sejak reset terakhir: iterasi terukur, atau seluruh run jika OOM
        # terjadi saat pemanasan. Puncak total = maksimum dengan puncak pemanasan.
        since_reset = (torch.cuda.max_memory_allocated(), torch.cuda.max_memory_reserved())
        warm_alloc, warm_resv = result.get("warm_peak", (0, 0))
        peak_alloc = max(warm_alloc, since_reset[0]) / MIB
        peak_reserved = max(warm_resv, since_reset[1]) / MIB
        if "warm_peak" in result:
            steady_alloc = since_reset[0] / MIB
        total = torch.cuda.get_device_properties(device).total_memory / MIB
        pct = 100 * peak_reserved / total
    _cleanup(device)

    return ComboResult(img_size, batch, amp, ngf, status,
                       peak_alloc, peak_reserved, pct, steady_alloc, result["iter_ms"],
                       result["warmup_iters"], result["adam_ready"],
                       result["g_steps"], result["d_steps"],
                       result["g_params"], result["d_params"], result["note"])
''', baru=False)


In [ ]:
#@title scripts/measure_vram.py · 8/10: write_csv
tulis('scripts/measure_vram.py', r'''

# ---------------------------------------------------------------------------
# Output
# ---------------------------------------------------------------------------
def write_csv(results: list, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[fl.name for fl in fields(ComboResult)])
        writer.writeheader()
        for r in results:
            row = asdict(r)
            for key in ("peak_alloc_mib", "peak_reserved_mib", "pct_of_total",
                        "steady_alloc_mib", "iter_ms"):
                if row[key] is not None:
                    row[key] = round(row[key], 1)
            writer.writerow(row)


def format_table(results: list) -> str:
    def num(v, fmt):
        return "-" if v is None else format(v, fmt)

    def adam(r):
        if r.adam_ready is None:
            return "-"
        return f"{'ya' if r.adam_ready else 'TIDAK'}/{r.warmup_iters}"

    header = f"{'img':>4} {'batch':>5} {'amp':>4} {'ngf':>4} {'status':>7} " \
             f"{'alloc MiB':>10} {'resv MiB':>9} {'steady':>7} {'% VRAM':>7} " \
             f"{'ms/iter':>8} {'adam/warm':>10}"
    lines = [header, "-" * len(header)]
    for r in results:
        lines.append(
            f"{r.img_size:>4} {r.batch:>5} {'on' if r.amp else 'off':>4} {r.ngf:>4} "
            f"{r.status:>7} {num(r.peak_alloc_mib, '.0f'):>10} "
            f"{num(r.peak_reserved_mib, '.0f'):>9} {num(r.steady_alloc_mib, '.0f'):>7} "
            f"{num(r.pct_of_total, '.0f'):>7} {num(r.iter_ms, '.0f'):>8} {adam(r):>10}"
        )
    return "\n".join(lines)
''', baru=False)


In [ ]:
#@title scripts/measure_vram.py · 9/10: parse_args
tulis('scripts/measure_vram.py', r'''

# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------
def parse_args(argv=None):
    p = argparse.ArgumentParser(description="Ukur puncak VRAM satu langkah training Pix2Pix.")
    # Grid kombinasi
    p.add_argument("--img-sizes", type=int, nargs="+", default=[256, 128])
    p.add_argument("--batches", type=int, nargs="+", default=[1, 2, 4])
    p.add_argument("--amp", nargs="+", choices=["off", "on"], default=["off", "on"])
    p.add_argument("--ngfs", type=int, nargs="+", default=[64, 32])
    # Parameter model (default = yang sudah disepakati)
    p.add_argument("--g-norm", default="batch", choices=["batch", "instance", "none"])
    p.add_argument("--use-skip", action=argparse.BooleanOptionalAction, default=True)
    p.add_argument("--use-dropout", action=argparse.BooleanOptionalAction, default=True)
    p.add_argument("--ndf", type=int, default=64)
    p.add_argument("--n-layers", type=int, default=3)
    p.add_argument("--d-norm", default="batch", choices=["batch", "instance", "none"])
    # Parameter training
    p.add_argument("--lr", type=float, default=2e-4)
    p.add_argument("--beta1", type=float, default=0.5)
    p.add_argument("--beta2", type=float, default=0.999)
    p.add_argument("--lambda-l1", type=float, default=100.0)
    p.add_argument("--iters", type=int, default=3,
                   help="iterasi TERUKUR per kombinasi, setelah state Adam terbentuk")
    p.add_argument("--max-warmup", type=int, default=50,
                   help="batas iterasi pemanasan untuk menunggu state Adam terbentuk")
    p.add_argument("--seed", type=int, default=0)
    p.add_argument("--cudnn-benchmark", action="store_true",
                   help="samakan dengan train.py jika nanti memakai cudnn.benchmark=True")
    p.add_argument("--cudnn-deterministic", action="store_true",
                   help="paksa algoritma cuDNN deterministik (untuk menguji lonjakan memori)")
    p.add_argument("--device", default="cuda", choices=["cuda", "cpu"],
                   help="cpu hanya untuk uji jalan kode; memori tidak diukur")
    p.add_argument("--out", default="vram_results.csv")
    args = p.parse_args(argv)
    if args.iters < 1:
        p.error("--iters harus >= 1")
    if args.max_warmup < 1:
        p.error("--max-warmup harus >= 1")
    return args
''', baru=False)


In [ ]:
#@title scripts/measure_vram.py · 10/10: main
tulis('scripts/measure_vram.py', r'''

def main(argv=None) -> list:
    args = parse_args(argv)
    if args.device == "cuda" and not torch.cuda.is_available():
        raise SystemExit("CUDA tidak tersedia. Jalankan di laptop dengan GPU, "
                         "atau pakai --device cpu hanya untuk uji jalan kode.")
    device = torch.device(args.device)
    torch.backends.cudnn.benchmark = args.cudnn_benchmark
    torch.backends.cudnn.deterministic = args.cudnn_deterministic

    if device.type == "cuda":
        prop = torch.cuda.get_device_properties(device)
        print(f"GPU: {prop.name}, total {prop.total_memory / MIB:.0f} MiB | "
              f"PyTorch {torch.__version__}, CUDA {torch.version.cuda}")
    print(f"G: norm={args.g_norm}, skip={args.use_skip}, dropout={args.use_dropout} | "
          f"D: ndf={args.ndf}, n_layers={args.n_layers}, norm={args.d_norm} | "
          f"iters={args.iters}, max_warmup={args.max_warmup} | "
          f"cudnn benchmark={args.cudnn_benchmark}, deterministic={args.cudnn_deterministic}\n")

    results = []
    for img_size in args.img_sizes:
        for batch in args.batches:
            for amp_flag in args.amp:
                for ngf in args.ngfs:
                    r = run_combo(args, img_size, batch, amp_flag == "on", ngf, device)
                    results.append(r)
                    print(f"  img={img_size} batch={batch} amp={amp_flag} ngf={ngf}: {r.status}"
                          + (f", {r.peak_reserved_mib:.0f} MiB" if r.peak_reserved_mib else ""))

    out = Path(args.out)
    write_csv(results, out)
    print("\n" + format_table(results))
    print(f"\nCSV disimpan ke: {out.resolve()}")
    return results


if __name__ == "__main__":
    main()
''', baru=False)


### `scripts/smoke_real_data.py`

Skrip uji cepat pada data asli (milik kalian, tidak diubah). 2 sel, 74 baris. Isi sel → 1: `import dan konstanta` · 2: `main`

In [ ]:
#@title scripts/smoke_real_data.py · 1/2: import dan konstanta
tulis('scripts/smoke_real_data.py', r'''"""scripts/smoke_real_data.py - uji cepat get_dataloader pada data asli.

Contoh (Windows, dari root repo):
  python scripts/smoke_real_data.py --module dataset --root "D:\\...\\CMP_facade_DB_base\\base" --out cek_lama.png
  python scripts/smoke_real_data.py --module dataset_baru --root "D:\\...\\base" --out cek_baru.png
Manifest ditulis ke folder sementara (tidak mengotori folder data).
"""
import argparse
import dataclasses
import importlib
import sys
import tempfile
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
''', baru=True)


In [ ]:
#@title scripts/smoke_real_data.py · 2/2: main
tulis('scripts/smoke_real_data.py', r'''

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--module", default="dataset")
    ap.add_argument("--root", required=True, help="folder yang LANGSUNG berisi .jpg/.png (subfolder base)")
    ap.add_argument("--direction", default="AtoB", choices=["AtoB", "BtoA"])
    ap.add_argument("--img-size", type=int, default=256)
    ap.add_argument("--load-size", type=int, default=286)
    ap.add_argument("--batches", type=int, default=20)
    ap.add_argument("--out", default=None, help="simpan grid A|B (4 pasangan val) ke file ini")
    args = ap.parse_args()

    import numpy as np
    import torch
    from PIL import Image
    from config import Config

    mod = importlib.import_module(args.module)
    print(f"modul={mod.__name__} torch={torch.__version__} cuda={torch.cuda.is_available()}")

    with tempfile.TemporaryDirectory() as tmp:
        cfg = dataclasses.replace(
            Config(), data_root=args.root, direction=args.direction,
            img_size=args.img_size, load_size=args.load_size,
            num_workers=0, pin_memory=False, manifest_path=str(Path(tmp) / "manifest.json"))
        torch.manual_seed(cfg.seed)  # num_workers=0: crop memakai RNG global

        for split in ("train", "val", "test"):
            ds = mod.get_dataloader(cfg, split).dataset
            print(f"{split}: {len(ds)} pasang, label_mode={getattr(ds, 'label_mode', '(tidak ada)')}")

        loader = mod.get_dataloader(cfg, "train")
        t0, n = time.perf_counter(), 0
        for batch in loader:
            if n == 0:
                a, b = batch["A"], batch["B"]
                print(f"batch pertama: A{tuple(a.shape)} B{tuple(b.shape)} {a.dtype} "
                      f"rentang A[{a.min():.2f},{a.max():.2f}] B[{b.min():.2f},{b.max():.2f}]")
            n += 1
            if n >= args.batches:
                break
        dt = time.perf_counter() - t0
        print(f"{n} batch: {dt:.1f} dtk ({dt / n:.2f} dtk/batch, CPU, num_workers=0)")

        if args.out:
            val = mod.get_dataloader(cfg, "val").dataset
            rows = []
            for i in range(min(4, len(val))):
                s = val[i]
                pair = [mod.denormalize(s[k]).permute(1, 2, 0).numpy() for k in ("A", "B")]
                rows.append(np.concatenate(pair, axis=1))
            Image.fromarray(np.concatenate(rows, axis=0)).save(args.out)
            print(f"grid disimpan: {args.out} (kiri = A, kanan = B)")


if __name__ == "__main__":
    main()
''', baru=False)


### `tests/test_generator.py`

Tes generator. 11 sel, 283 baris. Isi sel → 1: `fixed_seed` · 2: `_conv_params` · 3: `test_output_shape` · 4: `test_param_count_default_literal` · 5: `test_skip_changes_channels_not_output_shape` · 6: `test_skip_actually_carries_information` · 7: `test_forward_backward` · 8: `test_output_range` · 9: `test_first_and_bottleneck_have_no_norm` · 10: `test_eval_mode_is_deterministic` · 11: `test_rejects_unsupported_img_size`

In [ ]:
#@title tests/test_generator.py · 1/11: fixed_seed
tulis('tests/test_generator.py', r'''"""
tests/test_generator.py

Tes untuk models/generator.py. Jalankan dari root repo:
    pytest tests/test_generator.py -v

Semua tes jalan di CPU agar bisa dijalankan siapa saja di kelompok tanpa GPU.
"""

import math

import pytest
import torch
import torch.nn as nn

from models.generator import DownBlock, UNetGenerator, UpBlock


@pytest.fixture(autouse=True)
def fixed_seed():
    # Seed tetap supaya hasil tes (misalnya statistik init) dapat direproduksi.
    torch.manual_seed(0)
''', baru=True)


In [ ]:
#@title tests/test_generator.py · 2/11: _conv_params
tulis('tests/test_generator.py', r'''

# ---------------------------------------------------------------------------
# Hitung jumlah parameter secara analitik (tanpa membuat model), dari spesifikasi.
# Ini cara independen untuk memverifikasi bahwa model dibangun sesuai rancangan.
# ---------------------------------------------------------------------------
def _conv_params(c_in, c_out, block_norm):
    """Parameter satu Conv/ConvT 4x4 + normalisasi sesudahnya."""
    p = c_in * c_out * 4 * 4          # bobot kernel 4x4
    if block_norm == "none":
        p += c_out                    # bias conv (dipakai hanya jika tanpa norm)
    elif block_norm == "batch":
        p += 2 * c_out                # gamma + beta BatchNorm
    # "instance" tanpa affine: 0 parameter tambahan
    return p


def expected_num_params(in_ch=3, out_ch=3, ngf=64, img_size=256,
                        use_skip=True, norm="batch", max_mult=8):
    n = int(math.log2(img_size))
    enc = [ngf * min(2 ** i, max_mult) for i in range(n)]
    total = 0
    # Encoder: blok pertama & bottleneck tanpa norm
    for i in range(n):
        c_in = in_ch if i == 0 else enc[i - 1]
        total += _conv_params(c_in, enc[i], "none" if i in (0, n - 1) else norm)
    # Decoder
    for j in range(1, n):
        c_out = enc[n - 1 - j]
        c_in = enc[-1] if j == 1 else enc[n - j] * (2 if use_skip else 1)
        total += _conv_params(c_in, c_out, norm)
    # Layer output: ConvT + bias, tanpa norm
    total += _conv_params(enc[0] * (2 if use_skip else 1), out_ch, "none")
    return total
''', baru=False)


In [ ]:
#@title tests/test_generator.py · 3/11: test_output_shape
tulis('tests/test_generator.py', r'''

# ---------------------------------------------------------------------------
# 1. Shape: img_size 256 & 128, batch 1 & 2 (mode train, kasus terberat untuk BN)
# ---------------------------------------------------------------------------
@pytest.mark.parametrize("img_size", [256, 128])
@pytest.mark.parametrize("batch", [1, 2])
def test_output_shape(img_size, batch):
    gen = UNetGenerator(img_size=img_size)
    gen.train()  # batch=1 + mode train = kondisi yang dulu bisa crash di bottleneck
    x = torch.randn(batch, 3, img_size, img_size)
    y = gen(x)
    assert y.shape == (batch, 3, img_size, img_size)


@pytest.mark.parametrize("img_size", [256, 128])
def test_intermediate_shapes(img_size):
    """Cek shape tiap blok encoder: spasial terbagi dua, bottleneck 1x1."""
    gen = UNetGenerator(img_size=img_size, ngf=64)
    h = torch.randn(1, 3, img_size, img_size)
    for i, block in enumerate(gen.encoder):
        h = block(h)
        assert h.shape[-1] == img_size // 2 ** (i + 1)
        assert h.shape[1] == gen.enc_channels[i]
    assert h.shape[-2:] == (1, 1)  # bottleneck


def test_custom_channels():
    """in_ch/out_ch berbeda, misalnya grayscale -> RGB."""
    gen = UNetGenerator(in_ch=1, out_ch=3, img_size=128, ngf=32)
    y = gen(torch.randn(2, 1, 128, 128))
    assert y.shape == (2, 3, 128, 128)
''', baru=False)


In [ ]:
#@title tests/test_generator.py · 4/11: test_param_count_default_literal
tulis('tests/test_generator.py', r'''

# ---------------------------------------------------------------------------
# 2. Jumlah parameter
# ---------------------------------------------------------------------------
def test_param_count_default_literal():
    """Angka dihitung manual (lihat penjelasan); berguna sebagai regresi."""
    gen = UNetGenerator()  # ngf=64, 256, batch norm, skip
    assert gen.num_parameters() == 54_414_531


@pytest.mark.parametrize("img_size", [256, 128])
@pytest.mark.parametrize("ngf", [64, 32])
@pytest.mark.parametrize("use_skip", [True, False])
@pytest.mark.parametrize("norm", ["batch", "instance", "none"])
def test_param_count_matches_formula(img_size, ngf, use_skip, norm):
    gen = UNetGenerator(ngf=ngf, img_size=img_size, use_skip=use_skip, norm=norm)
    assert gen.num_parameters() == expected_num_params(
        ngf=ngf, img_size=img_size, use_skip=use_skip, norm=norm)


def test_ngf32_is_about_quarter():
    """Parameter conv skala ~ngf^2, jadi ngf=32 seharusnya ~1/4 dari ngf=64."""
    p64 = UNetGenerator(ngf=64).num_parameters()
    p32 = UNetGenerator(ngf=32).num_parameters()
    assert 0.24 < p32 / p64 < 0.26
''', baru=False)


In [ ]:
#@title tests/test_generator.py · 5/11: test_skip_changes_channels_not_output_shape
tulis('tests/test_generator.py', r'''

# ---------------------------------------------------------------------------
# 3. use_skip True vs False
# ---------------------------------------------------------------------------
def test_skip_changes_channels_not_output_shape():
    g_skip = UNetGenerator(img_size=128, use_skip=True)
    g_noskip = UNetGenerator(img_size=128, use_skip=False)
    x = torch.randn(2, 3, 128, 128)
    assert g_skip(x).shape == g_noskip(x).shape == (2, 3, 128, 128)

    # Channel input decoder: d1 sama, sisanya 2x lebih lebar dengan skip
    in_skip = [b.deconv.in_channels for b in g_skip.decoder]
    in_noskip = [b.deconv.in_channels for b in g_noskip.decoder]
    assert in_skip[0] == in_noskip[0]
    assert all(a == 2 * b for a, b in zip(in_skip[1:], in_noskip[1:]))
    assert g_skip.final[0].in_channels == 2 * g_noskip.final[0].in_channels

    # Tanpa skip -> lebih sedikit parameter (perlu dilaporkan di ablation)
    assert g_noskip.num_parameters() < g_skip.num_parameters()
''', baru=False)


In [ ]:
#@title tests/test_generator.py · 6/11: test_skip_actually_carries_information
tulis('tests/test_generator.py', r'''

def test_skip_actually_carries_information():
    """Dengan skip, mengubah fitur encoder dangkal (e1) harus mengubah output
    walaupun bottleneck dibuat identik. Tanpa skip, output hanya bergantung
    pada bottleneck. Kita uji dengan hook yang mengganti output e1."""
    for use_skip in (True, False):
        gen = UNetGenerator(img_size=128, use_skip=use_skip).eval()
        x = torch.randn(1, 3, 128, 128)

        outputs = []
        for e1_scale in (1.0, 0.0):
            def scale_e1(_mod, _inp, out, s=e1_scale):
                return out * s
            # Kunci bottleneck ke nilai tetap agar hanya jalur skip yang berbeda
            def fix_bottleneck(_mod, _inp, out):
                return torch.ones_like(out)
            h1 = gen.encoder[0].register_forward_hook(scale_e1)
            h2 = gen.encoder[-1].register_forward_hook(fix_bottleneck)
            with torch.no_grad():
                outputs.append(gen(x))
            h1.remove()
            h2.remove()

        differs = not torch.allclose(outputs[0], outputs[1])
        assert differs == use_skip
''', baru=False)


In [ ]:
#@title tests/test_generator.py · 7/11: test_forward_backward
tulis('tests/test_generator.py', r'''

# ---------------------------------------------------------------------------
# 4. Satu forward-backward tanpa error, gradien sampai ke semua parameter
# ---------------------------------------------------------------------------
@pytest.mark.parametrize("use_skip", [True, False])
def test_forward_backward(use_skip):
    gen = UNetGenerator(img_size=128, use_skip=use_skip).train()
    x = torch.randn(2, 3, 128, 128)
    target = torch.rand(2, 3, 128, 128) * 2 - 1  # target acak di [-1, 1]

    loss = nn.functional.l1_loss(gen(x), target)
    loss.backward()

    assert torch.isfinite(loss)
    for name, p in gen.named_parameters():
        assert p.grad is not None, f"{name} tidak mendapat gradien"
        assert torch.isfinite(p.grad).all(), f"gradien {name} tidak finite"


def test_forward_backward_batch1_256():
    """Kasus realistis di GPU 4 GB: batch 1, 256x256, mode train."""
    gen = UNetGenerator(img_size=256).train()
    loss = gen(torch.randn(1, 3, 256, 256)).abs().mean()
    loss.backward()
    assert torch.isfinite(loss)
''', baru=False)


In [ ]:
#@title tests/test_generator.py · 8/11: test_output_range
tulis('tests/test_generator.py', r'''

# ---------------------------------------------------------------------------
# 5. Output berada di [-1, 1]
# ---------------------------------------------------------------------------
@pytest.mark.parametrize("scale", [1.0, 100.0])
def test_output_range(scale):
    gen = UNetGenerator(img_size=128)
    # scale=100 memaksa aktivasi besar untuk memastikan Tanh tetap membatasi
    y = gen(torch.randn(2, 3, 128, 128) * scale)
    assert y.min() >= -1.0 and y.max() <= 1.0
''', baru=False)


In [ ]:
#@title tests/test_generator.py · 9/11: test_first_and_bottleneck_have_no_norm
tulis('tests/test_generator.py', r'''

# ---------------------------------------------------------------------------
# 6. Struktur sesuai spesifikasi
# ---------------------------------------------------------------------------
def test_first_and_bottleneck_have_no_norm():
    gen = UNetGenerator(norm="batch")
    for idx in (0, -1):
        block = gen.encoder[idx]
        assert isinstance(block.norm, nn.Identity)
        assert block.conv.bias is not None  # tanpa norm -> bias dipakai
    for block in gen.encoder[1:-1]:
        assert isinstance(block.norm, nn.BatchNorm2d)
        assert block.conv.bias is None      # dengan norm -> tanpa bias


@pytest.mark.parametrize("img_size", [256, 128])
def test_dropout_on_first_three_decoder_blocks(img_size):
    gen = UNetGenerator(img_size=img_size, use_dropout=True)
    has_dropout = [isinstance(b.dropout, nn.Dropout) for b in gen.decoder]
    assert has_dropout[:3] == [True, True, True]
    assert not any(has_dropout[3:])
    assert all(b.dropout.p == 0.5 for b in gen.decoder[:3])

    gen_nodrop = UNetGenerator(img_size=img_size, use_dropout=False)
    assert not any(isinstance(b.dropout, nn.Dropout) for b in gen_nodrop.decoder)


def test_block_counts():
    assert len(UNetGenerator(img_size=256).encoder) == 8
    assert len(UNetGenerator(img_size=256).decoder) == 7
    assert len(UNetGenerator(img_size=128).encoder) == 7
    assert len(UNetGenerator(img_size=128).decoder) == 6
''', baru=False)


In [ ]:
#@title tests/test_generator.py · 10/11: test_eval_mode_is_deterministic
tulis('tests/test_generator.py', r'''

def test_eval_mode_is_deterministic():
    """Di eval(), Dropout nonaktif dan BN memakai running stats -> output sama."""
    gen = UNetGenerator(img_size=128).eval()
    x = torch.randn(1, 3, 128, 128)
    with torch.no_grad():
        assert torch.equal(gen(x), gen(x))


def test_weight_init_statistics():
    gen = UNetGenerator()
    w = gen.encoder[4].conv.weight  # conv besar (512x512x4x4) -> statistik stabil
    assert abs(w.mean().item()) < 1e-3
    assert abs(w.std().item() - 0.02) < 1e-3
    assert torch.all(gen.final[0].bias == 0)
    bn = gen.encoder[2].norm
    assert abs(bn.weight.mean().item() - 1.0) < 0.01
    assert torch.all(bn.bias == 0)


def test_blocks_halve_and_double_spatial():
    x = torch.randn(1, 8, 32, 32)
    assert DownBlock(8, 16, "batch", 0.2)(x).shape == (1, 16, 16, 16)
    assert UpBlock(8, 4, "batch", 0.0)(x).shape == (1, 4, 64, 64)
''', baru=False)


In [ ]:
#@title tests/test_generator.py · 11/11: test_rejects_unsupported_img_size
tulis('tests/test_generator.py', r'''

# ---------------------------------------------------------------------------
# 7. Validasi argumen: error yang jelas
# ---------------------------------------------------------------------------
@pytest.mark.parametrize("bad_size", [64, 200, 512])
def test_rejects_unsupported_img_size(bad_size):
    with pytest.raises(ValueError, match="img_size"):
        UNetGenerator(img_size=bad_size)


def test_rejects_wrong_input_shape():
    gen = UNetGenerator(img_size=128)
    with pytest.raises(ValueError, match="Input generator"):
        gen(torch.randn(1, 3, 256, 256))   # ukuran salah
    with pytest.raises(ValueError, match="Input generator"):
        gen(torch.randn(1, 1, 128, 128))   # jumlah channel salah


@pytest.mark.parametrize("kwargs", [
    {"norm": "group"},
    {"ngf": 0},
    {"dropout_p": 1.0},
    {"n_dropout": 10},
])
def test_rejects_bad_arguments(kwargs):
    with pytest.raises(ValueError):
        UNetGenerator(img_size=256, **kwargs)
''', baru=False)


### `tests/test_discriminator.py`

Tes discriminator. 8 sel, 233 baris. Isi sel → 1: `fixed_seed` · 2: `test_conv_output_size_formula` · 3: `test_receptive_field_formula` · 4: `test_receptive_field_measured_by_gradient` · 5: `test_no_sigmoid_and_last_layer_is_conv` · 6: `test_concat_order_matters` · 7: `test_first_block_has_no_norm` · 8: `test_rejects_bad_arguments`

In [ ]:
#@title tests/test_discriminator.py · 1/8: fixed_seed
tulis('tests/test_discriminator.py', r'''"""
tests/test_discriminator.py

Tes untuk models/discriminator.py. Jalankan dari root repo:
    pytest tests/test_discriminator.py -v

Ukuran patch dan receptive field dibuktikan dengan dua cara yang independen:
1. dihitung dengan rumus (fungsi murni patch_output_size / receptive_field);
2. diukur dari model sungguhan (shape tensor dan jangkauan gradien).
"""

import pytest
import torch
import torch.nn as nn

from models.discriminator import (
    PatchDiscriminator,
    conv_output_size,
    patch_output_size,
    receptive_field,
)


@pytest.fixture(autouse=True)
def fixed_seed():
    torch.manual_seed(0)


def _pair(batch, size, c=3):
    """Pasangan (cond, img) acak di rentang [-1, 1]."""
    cond = torch.rand(batch, c, size, size) * 2 - 1
    img = torch.rand(batch, c, size, size) * 2 - 1
    return cond, img
''', baru=True)


In [ ]:
#@title tests/test_discriminator.py · 2/8: test_conv_output_size_formula
tulis('tests/test_discriminator.py', r'''

# ---------------------------------------------------------------------------
# 1. Ukuran output patch: hitungan tangan == rumus == shape sebenarnya
# ---------------------------------------------------------------------------
# Hitungan tangan, rumus floor((n + 2*1 - 4) / s) + 1 per layer:
#   256, n_layers=3: 256 -s2-> 128 -s2-> 64 -s2-> 32 -s1-> 31 -s1-> 30
#   256, n_layers=1: 256 -s2-> 128 -s1-> 127 -s1-> 126
#   128, n_layers=3: 128 -s2-> 64 -s2-> 32 -s2-> 16 -s1-> 15 -s1-> 14
#   128, n_layers=1: 128 -s2-> 64 -s1-> 63 -s1-> 62
HAND_COMPUTED_PATCH = {
    (256, 3): 30,
    (256, 1): 126,
    (128, 3): 14,
    (128, 1): 62,
}


def test_conv_output_size_formula():
    assert conv_output_size(256, 4, 2, 1) == 128  # stride 2 membagi dua
    assert conv_output_size(32, 4, 1, 1) == 31    # stride 1 berkurang 1


@pytest.mark.parametrize("img_size,n_layers", list(HAND_COMPUTED_PATCH))
def test_patch_size_formula_matches_hand(img_size, n_layers):
    assert patch_output_size(img_size, n_layers) == HAND_COMPUTED_PATCH[(img_size, n_layers)]


@pytest.mark.parametrize("img_size,n_layers", list(HAND_COMPUTED_PATCH))
@pytest.mark.parametrize("batch", [1, 2])
def test_output_shape_matches_hand(img_size, n_layers, batch):
    disc = PatchDiscriminator(n_layers=n_layers).train()
    out = disc(*_pair(batch, img_size))
    expected = HAND_COMPUTED_PATCH[(img_size, n_layers)]
    assert out.shape == (batch, 1, expected, expected)
    assert disc.output_size(img_size) == expected
''', baru=False)


In [ ]:
#@title tests/test_discriminator.py · 3/8: test_receptive_field_formula
tulis('tests/test_discriminator.py', r'''

# ---------------------------------------------------------------------------
# 2. Receptive field: rumus vs pengukuran gradien
# ---------------------------------------------------------------------------
# Hitungan tangan, mundur dari output dengan rf = (rf - 1) * s + 4:
#   n_layers=3: 1 -s1-> 4 -s1-> 7 -s2-> 16 -s2-> 34 -s2-> 70
#   n_layers=1: 1 -s1-> 4 -s1-> 7 -s2-> 16
HAND_COMPUTED_RF = {1: 16, 3: 70}


@pytest.mark.parametrize("n_layers", [1, 3])
def test_receptive_field_formula(n_layers):
    assert receptive_field(n_layers) == HAND_COMPUTED_RF[n_layers]
    assert PatchDiscriminator(n_layers=n_layers).receptive_field() == HAND_COMPUTED_RF[n_layers]


def _gradient_bbox(disc, img_size, out_row, out_col):
    """Backprop dari SATU elemen output ke input, lalu ukur kotak piksel input
    yang gradiennya tidak nol. Kotak itu adalah receptive field yang nyata."""
    cond, img = _pair(1, img_size)
    cond.requires_grad_(True)
    img.requires_grad_(True)
    out = disc(cond, img)
    out[0, 0, out_row, out_col].backward()
    # Gabungkan gradien kedua input dan semua channel -> peta [H, W]
    g = (cond.grad.abs() + img.grad.abs()).sum(dim=(0, 1))
    rows = torch.nonzero(g.sum(dim=1)).flatten()
    cols = torch.nonzero(g.sum(dim=0)).flatten()
    return rows.min().item(), rows.max().item(), cols.min().item(), cols.max().item()
''', baru=False)


In [ ]:
#@title tests/test_discriminator.py · 4/8: test_receptive_field_measured_by_gradient
tulis('tests/test_discriminator.py', r'''

@pytest.mark.parametrize("n_layers,img_size", [(1, 128), (3, 256), (1, 256), (3, 128)])
def test_receptive_field_measured_by_gradient(n_layers, img_size):
    # eval(): BatchNorm memakai running stats (operasi per piksel). Di mode
    # train, mean/var batch menghubungkan SEMUA piksel sehingga gradien
    # menyebar ke luar RF (lihat tes berikutnya).
    disc = PatchDiscriminator(n_layers=n_layers).eval()
    center = disc.output_size(img_size) // 2  # elemen tengah: RF tidak terpotong tepi
    r0, r1, c0, c1 = _gradient_bbox(disc, img_size, center, center)
    rf = HAND_COMPUTED_RF[n_layers]
    assert r1 - r0 + 1 == rf
    assert c1 - c0 + 1 == rf


@pytest.mark.parametrize("n_layers", [1, 3])
def test_neighbouring_patches_shift_by_total_stride(n_layers):
    """Dua elemen output yang bersebelahan melihat patch yang bergeser
    2**n_layers piksel (perkalian semua stride). Jadi patch saling tumpang tindih."""
    disc = PatchDiscriminator(n_layers=n_layers).eval()
    c = disc.output_size(256) // 2
    r0_a, _, c0_a, _ = _gradient_bbox(disc, 256, c, c)
    r0_b, _, c0_b, _ = _gradient_bbox(disc, 256, c, c + 1)
    assert r0_a == r0_b
    assert c0_b - c0_a == 2 ** n_layers


def test_batchnorm_train_mode_couples_all_pixels():
    """Fakta yang perlu diketahui: di mode train, BN membuat tiap output
    bergantung juga pada piksel di luar RF (lewat statistik batch).
    RF 'murni' hanya berlaku untuk tumpukan conv itu sendiri."""
    disc = PatchDiscriminator(n_layers=1, norm="batch").train()
    center = disc.output_size(128) // 2
    r0, r1, _, _ = _gradient_bbox(disc, 128, center, center)
    assert r1 - r0 + 1 > receptive_field(1)
''', baru=False)


In [ ]:
#@title tests/test_discriminator.py · 5/8: test_no_sigmoid_and_last_layer_is_conv
tulis('tests/test_discriminator.py', r'''

# ---------------------------------------------------------------------------
# 3. Output berupa LOGIT (tanpa Sigmoid)
# ---------------------------------------------------------------------------
def test_no_sigmoid_and_last_layer_is_conv():
    disc = PatchDiscriminator()
    assert not any(isinstance(m, nn.Sigmoid) for m in disc.modules())
    last = disc.net[-1]
    assert isinstance(last, nn.Conv2d) and last.out_channels == 1


def test_output_not_restricted_to_unit_interval():
    """Logit boleh negatif. Kalau ada sigmoid tersembunyi, semua nilai > 0."""
    disc = PatchDiscriminator(n_layers=3).eval()
    with torch.no_grad():
        out = disc(*_pair(2, 128))
    assert (out < 0).any()


def test_works_with_bce_with_logits():
    disc = PatchDiscriminator(n_layers=3).train()
    logits = disc(*_pair(2, 128))
    loss = nn.BCEWithLogitsLoss()(logits, torch.ones_like(logits))
    assert loss.dim() == 0 and torch.isfinite(loss)
''', baru=False)


In [ ]:
#@title tests/test_discriminator.py · 6/8: test_concat_order_matters
tulis('tests/test_discriminator.py', r'''

# ---------------------------------------------------------------------------
# 4. Conditioning dan forward-backward
# ---------------------------------------------------------------------------
def test_concat_order_matters():
    """D(cond, img) != D(img, cond): urutan concat bermakna."""
    disc = PatchDiscriminator().eval()
    a, b = _pair(1, 128)
    with torch.no_grad():
        assert not torch.allclose(disc(a, b), disc(b, a))


@pytest.mark.parametrize("n_layers", [1, 3])
def test_forward_backward_reaches_params_and_img(n_layers):
    disc = PatchDiscriminator(n_layers=n_layers).train()
    cond, img = _pair(2, 128)
    img.requires_grad_(True)  # di train.py, gradien G mengalir lewat input img ini
    logits = disc(cond, img)
    loss = nn.BCEWithLogitsLoss()(logits, torch.zeros_like(logits))
    loss.backward()
    for name, p in disc.named_parameters():
        assert p.grad is not None and torch.isfinite(p.grad).all(), name
    assert img.grad is not None and img.grad.abs().sum() > 0
''', baru=False)


In [ ]:
#@title tests/test_discriminator.py · 7/8: test_first_block_has_no_norm
tulis('tests/test_discriminator.py', r'''

# ---------------------------------------------------------------------------
# 5. Struktur dan jumlah parameter
# ---------------------------------------------------------------------------
def test_first_block_has_no_norm():
    disc = PatchDiscriminator(norm="batch")
    assert isinstance(disc.net[0], nn.Conv2d) and disc.net[0].bias is not None
    assert isinstance(disc.net[1], nn.LeakyReLU)  # langsung aktivasi, tanpa norm


@pytest.mark.parametrize("n_layers", [1, 3])
def test_stride_layout(n_layers):
    convs = [m for m in PatchDiscriminator(n_layers=n_layers).modules()
             if isinstance(m, nn.Conv2d)]
    assert [c.stride[0] for c in convs] == [2] * n_layers + [1, 1]


def test_param_count_default_literal():
    # Hitungan tangan (ndf=64, n_layers=3, batch norm):
    #   c1  6*64*16   + 64 bias        =     6.208
    #   c2  64*128*16 + 2*128 BN       =   131.328
    #   c3  128*256*16 + 2*256 BN      =   524.800
    #   c4  256*512*16 + 2*512 BN      = 2.098.176
    #   out 512*1*16  + 1 bias         =     8.193
    #   total                          = 2.768.705
    assert PatchDiscriminator().num_parameters() == 2_768_705
''', baru=False)


In [ ]:
#@title tests/test_discriminator.py · 8/8: test_rejects_bad_arguments
tulis('tests/test_discriminator.py', r'''

# ---------------------------------------------------------------------------
# 6. Validasi argumen dan input
# ---------------------------------------------------------------------------
@pytest.mark.parametrize("kwargs", [{"norm": "group"}, {"n_layers": 0}, {"ndf": 0}])
def test_rejects_bad_arguments(kwargs):
    with pytest.raises(ValueError):
        PatchDiscriminator(**kwargs)


def test_rejects_mismatched_inputs():
    disc = PatchDiscriminator()
    with pytest.raises(ValueError, match="ukuran spasial"):
        disc(torch.randn(1, 3, 128, 128), torch.randn(1, 3, 64, 64))
    with pytest.raises(ValueError, match="in_ch"):
        disc(torch.randn(1, 3, 128, 128), torch.randn(1, 1, 128, 128))


def test_rejects_too_small_input():
    disc = PatchDiscriminator(n_layers=3)
    with pytest.raises(ValueError, match="terlalu kecil"):
        disc(*_pair(1, 8))
''', baru=False)


### `tests/test_dataset.py`

Tes dataset. 15 sel, 391 baris. Isi sel → 1: `fixed_seed` · 2: `label_like` · 3: `builtin_root` · 4: `test_shape_dtype_range` · 5: `test_direction_swaps_a_and_b` · 6: `recover_params` · 7: `test_crop_and_flip_are_synchronised` · 8: `_keys` · 9: `test_different_seed_gives_different_split` · 10: `test_subset_size` · 11: `test_rejects_non_side_by_side` · 12: `make_cfg` · 13: `test_windows_style_spawn_workers` · 14: `test_denormalize_endpoints_and_dtype` · 15: `test_preview_script_writes_grid`

In [ ]:
#@title tests/test_dataset.py · 1/15: fixed_seed
tulis('tests/test_dataset.py', r'''"""
tests/test_dataset.py

Tes untuk dataset.py. Semua tes memakai gambar SINTETIS yang dibuat di folder
sementara (tmp_path), sehingga bisa dijalankan tanpa mengunduh dataset asli:
- "label" sintetis = blok-blok warna rata (meniru peta label facade)
- "foto" sintetis  = noise acak (meniru tekstur foto; setiap patch unik)

Jalankan: pytest tests/test_dataset.py -v
"""

import json
import pickle
import shutil
from types import SimpleNamespace

import numpy as np
import pytest
import torch
from PIL import Image
from torch.utils.data import DataLoader, RandomSampler, SequentialSampler

from dataset import (
    BICUBIC,
    PairedImageDataset,
    denormalize,
    find_split_overlap,
    get_dataloader,
    to_tensor,
)

HALF = 64  # sisi setiap setengah gambar sintetis


@pytest.fixture(autouse=True)
def fixed_seed():
    torch.manual_seed(0)
''', baru=True)


In [ ]:
#@title tests/test_dataset.py · 2/15: label_like
tulis('tests/test_dataset.py', r'''

# ---------------------------------------------------------------------------
# Pembuat data sintetis
# ---------------------------------------------------------------------------
def label_like(rng, size=HALF, blocks=4):
    """Blok warna rata: blocks x blocks petak, tiap petak satu warna."""
    colors = rng.integers(0, 256, (blocks, blocks, 3), dtype=np.uint8)
    return np.kron(colors, np.ones((size // blocks, size // blocks, 1), dtype=np.uint8))


def photo_like(rng, size=HALF):
    return rng.integers(0, 256, (size, size, 3), dtype=np.uint8)


def write_combined(path, left, right):
    Image.fromarray(np.concatenate([left, right], axis=1)).save(path)


def make_flat_combined(root, n=50, seed=0, left_fn=label_like, right_fn=photo_like):
    """n file berdampingan langsung di root (tanpa split bawaan). PNG = lossless."""
    rng = np.random.default_rng(seed)
    root.mkdir(parents=True, exist_ok=True)
    for i in range(n):
        write_combined(root / f"{i:03d}.png", left_fn(rng), right_fn(rng))
    return root


@pytest.fixture
def flat_root(tmp_path):
    return make_flat_combined(tmp_path / "flat")
''', baru=False)


In [ ]:
#@title tests/test_dataset.py · 3/15: builtin_root
tulis('tests/test_dataset.py', r'''

@pytest.fixture
def builtin_root(tmp_path):
    root = tmp_path / "builtin"
    rng = np.random.default_rng(1)
    for split, n in (("train", 10), ("val", 3), ("test", 3)):
        (root / split).mkdir(parents=True)
        for i in range(n):
            write_combined(root / split / f"{split}_{i}.png", label_like(rng), photo_like(rng))
    return root


@pytest.fixture
def separate_root(tmp_path):
    """Format terpisah ala CMP asli: label .png + foto .jpg + file .xml (diabaikan)."""
    root = tmp_path / "separate"
    root.mkdir()
    rng = np.random.default_rng(2)
    for i in range(20):
        Image.fromarray(label_like(rng)).save(root / f"cmp_{i:04d}.png")
        Image.fromarray(photo_like(rng)).save(root / f"cmp_{i:04d}.jpg", quality=95)
        (root / f"cmp_{i:04d}.xml").write_text("<anotasi/>")
    return root


def halves(path):
    """Belah file berdampingan secara manual (independen dari dataset.py)."""
    with Image.open(path) as im:
        im = im.convert("RGB")
    w, h = im.size
    return im.crop((0, 0, w // 2, h)), im.crop((w // 2, 0, w, h))
''', baru=False)


In [ ]:
#@title tests/test_dataset.py · 4/15: test_shape_dtype_range
tulis('tests/test_dataset.py', r'''

# ---------------------------------------------------------------------------
# 1. Shape, dtype, device, rentang
# ---------------------------------------------------------------------------
@pytest.mark.parametrize("split", ["train", "val", "test"])
def test_shape_dtype_range(flat_root, split):
    ds = PairedImageDataset(flat_root, split, img_size=256, load_size=286)
    item = ds[0]
    assert set(item) == {"A", "B", "path"}
    assert isinstance(item["path"], str)
    for key in ("A", "B"):
        x = item[key]
        assert x.shape == (3, 256, 256)
        assert x.dtype == torch.float32 and x.device.type == "cpu"
        assert x.min() >= -1.0 and x.max() <= 1.0


def test_augment_default_per_split(flat_root):
    assert PairedImageDataset(flat_root, "train").augment is True
    assert PairedImageDataset(flat_root, "val").augment is False
    assert PairedImageDataset(flat_root, "test").augment is False


def test_val_is_deterministic_plain_resize(flat_root):
    """Val: resize bicubic langsung ke img_size, tanpa unsur acak."""
    ds = PairedImageDataset(flat_root, "val", img_size=128, load_size=143)
    i1, i2 = ds[0], ds[0]
    assert torch.equal(i1["A"], i2["A"]) and torch.equal(i1["B"], i2["B"])
    left, right = halves(i1["path"])
    assert torch.equal(i1["A"], to_tensor(left.resize((128, 128), BICUBIC)))
    assert torch.equal(i1["B"], to_tensor(right.resize((128, 128), BICUBIC)))
''', baru=False)


In [ ]:
#@title tests/test_dataset.py · 5/15: test_direction_swaps_a_and_b
tulis('tests/test_dataset.py', r'''

# ---------------------------------------------------------------------------
# 2. direction
# ---------------------------------------------------------------------------
def test_direction_swaps_a_and_b(flat_root):
    ab = PairedImageDataset(flat_root, "val", "AtoB")[0]
    ba = PairedImageDataset(flat_root, "val", "BtoA")[0]
    assert torch.equal(ab["A"], ba["B"]) and torch.equal(ab["B"], ba["A"])


def test_separate_format_label_is_a(separate_root):
    """Format terpisah: AtoB -> A = label (.png), B = foto (.jpg)."""
    from scripts.preview_pairs import flatness
    ds = PairedImageDataset(separate_root, "val", "AtoB", img_size=64, load_size=72)
    item = ds[0]
    assert item["path"].endswith(".png")
    assert flatness(item["A"]) > flatness(item["B"])
''', baru=False)


In [ ]:
#@title tests/test_dataset.py · 6/15: recover_params
tulis('tests/test_dataset.py', r'''

# ---------------------------------------------------------------------------
# 3. BUKTI crop/flip sinkron antara A dan B
# ---------------------------------------------------------------------------
def recover_params(out, ref, size):
    """Cari (top, left, flip) yang membuat `out` sama persis dengan potongan
    `ref` (gambar setelah resize ke load_size). Pencarian brute-force, tidak
    memakai kode augmentasi di dataset.py, jadi independen."""
    n_off = ref.shape[-1] - size + 1
    found = []
    for flip in (False, True):
        x = out.flip(-1) if flip else out           # batalkan flip
        for top in range(n_off):
            for left in range(n_off):
                crop = ref[:, top:top + size, left:left + size]
                if torch.equal(x[:, :4, :4], crop[:, :4, :4]) and torch.equal(x, crop):
                    found.append((top, left, flip))
    assert len(found) == 1, f"parameter tidak unik/tidak ditemukan: {found}"
    return found[0]
''', baru=False)


In [ ]:
#@title tests/test_dataset.py · 7/15: test_crop_and_flip_are_synchronised
tulis('tests/test_dataset.py', r'''

def test_crop_and_flip_are_synchronised(tmp_path):
    # Kedua sisi berupa noise INDEPENDEN, sehingga A dan B berbeda isinya dan
    # setiap potongan hanya cocok di satu posisi.
    root = make_flat_combined(tmp_path / "sync", n=10, left_fn=photo_like)
    size, load = 32, 40
    ds = PairedImageDataset(root, "train", img_size=size, load_size=load)
    assert ds.augment

    params_seen = []
    for draw in range(30):
        idx = draw % len(ds)
        item = ds[idx]
        left, right = halves(item["path"])
        ref_a = to_tensor(left.resize((load, load), BICUBIC))
        ref_b = to_tensor(right.resize((load, load), BICUBIC))
        pa = recover_params(item["A"], ref_a, size)
        pb = recover_params(item["B"], ref_b, size)
        assert pa == pb, f"A dan B tidak sinkron: A={pa}, B={pb}"
        params_seen.append(pa)

    # Augmentasi benar-benar acak: kedua kondisi flip muncul, offset bervariasi.
    assert {p[2] for p in params_seen} == {False, True}
    assert len({p[:2] for p in params_seen}) > 5


def test_random_params_sampled_once_per_item(flat_root, monkeypatch):
    ds = PairedImageDataset(flat_root, "train", img_size=32, load_size=40)
    calls = []
    original = ds._sample_aug_params
    monkeypatch.setattr(ds, "_sample_aug_params", lambda: calls.append(1) or original())
    ds[0]
    assert len(calls) == 1
''', baru=False)


In [ ]:
#@title tests/test_dataset.py · 8/15: _keys
tulis('tests/test_dataset.py', r'''

# ---------------------------------------------------------------------------
# 4. Split: tidak ada tumpang tindih
# ---------------------------------------------------------------------------
def _keys(ds):
    return {p[0] for p in ds.pairs}


def test_manifest_split_disjoint_and_complete(flat_root):
    dss = {s: PairedImageDataset(flat_root, s) for s in ("train", "val", "test")}
    assert [len(dss[s]) for s in ("train", "val", "test")] == [40, 5, 5]
    k = {s: _keys(d) for s, d in dss.items()}
    assert not (k["train"] & k["val"]) and not (k["train"] & k["test"]) and not (k["val"] & k["test"])
    assert len(k["train"] | k["val"] | k["test"]) == 50
    assert find_split_overlap(dss) == {"same_path": [], "same_content": []}
    assert all(d.split_source == "manifest" for d in dss.values())


def test_manifest_saved_and_deterministic(flat_root):
    PairedImageDataset(flat_root, "train")
    manifest_file = flat_root / "split_manifest_seed42.json"
    assert manifest_file.exists()
    first = json.loads(manifest_file.read_text())
    assert first["seed"] == 42 and first["n_total"] == 50

    manifest_file.unlink()                     # buat ulang dari nol
    PairedImageDataset(flat_root, "train")
    assert json.loads(manifest_file.read_text())["splits"] == first["splits"]


def test_manifest_custom_path_and_seed_mismatch(flat_root, tmp_path):
    path = tmp_path / "manifests" / "facades.json"
    PairedImageDataset(flat_root, "train", manifest_path=path, split_seed=42)
    assert path.exists()
    with pytest.raises(ValueError, match="seed/rasio berbeda"):
        PairedImageDataset(flat_root, "train", manifest_path=path, split_seed=7)
''', baru=False)


In [ ]:
#@title tests/test_dataset.py · 9/15: test_different_seed_gives_different_split
tulis('tests/test_dataset.py', r'''

def test_different_seed_gives_different_split(flat_root, tmp_path):
    a = PairedImageDataset(flat_root, "val", manifest_path=tmp_path / "s42.json", split_seed=42)
    b = PairedImageDataset(flat_root, "val", manifest_path=tmp_path / "s7.json", split_seed=7)
    assert _keys(a) != _keys(b)


def test_builtin_split_used_and_disjoint(builtin_root):
    dss = {s: PairedImageDataset(builtin_root, s) for s in ("train", "val", "test")}
    assert [len(dss[s]) for s in ("train", "val", "test")] == [10, 3, 3]
    assert all(d.split_source == "bawaan" for d in dss.values())
    assert find_split_overlap(dss) == {"same_path": [], "same_content": []}
    assert not (builtin_root / "split_manifest_seed42.json").exists()


def test_overlap_checker_detects_duplicate_file(builtin_root):
    """Kontrol negatif: salin satu file train ke val -> harus terdeteksi."""
    src = sorted((builtin_root / "train").iterdir())[0]
    shutil.copy(src, builtin_root / "val" / "duplikat.png")
    dss = {s: PairedImageDataset(builtin_root, s) for s in ("train", "val")}
    assert len(find_split_overlap(dss)["same_content"]) == 1


def test_missing_builtin_split_raises(tmp_path):
    root = tmp_path / "only_train"
    make_flat_combined(root / "train", n=3)
    with pytest.raises(ValueError, match="split bawaan"):
        PairedImageDataset(root, "val")
''', baru=False)


In [ ]:
#@title tests/test_dataset.py · 10/15: test_subset_size
tulis('tests/test_dataset.py', r'''

# ---------------------------------------------------------------------------
# 5. subset_size untuk overfit test
# ---------------------------------------------------------------------------
def test_subset_size(flat_root):
    full = PairedImageDataset(flat_root, "train")
    sub = PairedImageDataset(flat_root, "train", subset_size=8)
    assert len(sub) == 8
    assert sub.augment is False                  # walaupun split train
    assert sub.pairs == full.pairs[:8]
    i1, i2 = sub[3], sub[3]
    assert torch.equal(i1["A"], i2["A"])         # identik tiap kali dibaca
''', baru=False)


In [ ]:
#@title tests/test_dataset.py · 11/15: test_rejects_non_side_by_side
tulis('tests/test_dataset.py', r'''

# ---------------------------------------------------------------------------
# 6. Validasi input
# ---------------------------------------------------------------------------
def test_rejects_non_side_by_side(tmp_path):
    root = tmp_path / "bad"
    root.mkdir()
    Image.fromarray(np.zeros((64, 64, 3), np.uint8)).save(root / "persegi.png")
    ds = PairedImageDataset(root, "train", split_ratios=(1.0, 0.0, 0.0))
    with pytest.raises(ValueError, match="bukan 2:1"):
        ds[0]


def test_rejects_mixed_formats(flat_root):
    Image.fromarray(np.zeros((8, 8, 3), np.uint8)).save(flat_root / "000.jpg")
    with pytest.raises(ValueError, match="mencampur format|tidak bisa"):
        PairedImageDataset(flat_root, "train", manifest_path=flat_root / "m.json")


@pytest.mark.parametrize("kwargs", [
    {"split": "dev"}, {"direction": "AB"}, {"img_size": 300, "load_size": 286},
])
def test_rejects_bad_arguments(flat_root, kwargs):
    args = {"split": "train", **kwargs}
    with pytest.raises(ValueError):
        PairedImageDataset(flat_root, **args)
''', baru=False)


In [ ]:
#@title tests/test_dataset.py · 12/15: make_cfg
tulis('tests/test_dataset.py', r'''

# ---------------------------------------------------------------------------
# 7. DataLoader
# ---------------------------------------------------------------------------
def make_cfg(root, **over):
    cfg = dict(data_root=str(root), direction="AtoB", img_size=64, load_size=72,
               batch_size=4, num_workers=2, pin_memory=True, seed=123,
               subset_size=None, manifest_path=None, split_seed=42,
               split_ratios=(0.8, 0.1, 0.1))
    cfg.update(over)
    return SimpleNamespace(**cfg)


def test_dataloader_shuffle_only_train(flat_root):
    cfg = make_cfg(flat_root, num_workers=0)
    assert isinstance(get_dataloader(cfg, "train").sampler, RandomSampler)
    assert isinstance(get_dataloader(cfg, "val").sampler, SequentialSampler)
    assert isinstance(get_dataloader(cfg, "test").sampler, SequentialSampler)


def test_dataloader_batch_shapes_with_workers(flat_root):
    loader = get_dataloader(make_cfg(flat_root), "train")
    assert loader.num_workers == 2
    batch = next(iter(loader))
    assert batch["A"].shape == batch["B"].shape == (4, 3, 64, 64)
    assert len(batch["path"]) == 4


def test_dataloader_reproducible_with_seed(flat_root):
    b1 = next(iter(get_dataloader(make_cfg(flat_root), "train")))
    b2 = next(iter(get_dataloader(make_cfg(flat_root), "train")))
    assert b1["path"] == b2["path"]                # urutan shuffle sama
    assert torch.equal(b1["A"], b2["A"])           # crop/flip sama
''', baru=False)


In [ ]:
#@title tests/test_dataset.py · 13/15: test_windows_style_spawn_workers
tulis('tests/test_dataset.py', r'''

def test_windows_style_spawn_workers(flat_root):
    """Windows memakai 'spawn': dataset harus bisa di-pickle dan dibaca ulang
    di proses baru. Kita paksa spawn agar perilaku Windows teruji di Linux."""
    ds = PairedImageDataset(flat_root, "train", img_size=64, load_size=72)
    pickle.loads(pickle.dumps(ds))
    loader = DataLoader(ds, batch_size=2, num_workers=2, multiprocessing_context="spawn")
    batch = next(iter(loader))
    assert batch["A"].shape == (2, 3, 64, 64)


def test_dataloader_subset(flat_root):
    loader = get_dataloader(make_cfg(flat_root, subset_size=8, num_workers=0), "train")
    assert len(loader.dataset) == 8 and loader.dataset.augment is False
''', baru=False)


In [ ]:
#@title tests/test_dataset.py · 14/15: test_denormalize_endpoints_and_dtype
tulis('tests/test_dataset.py', r'''

# ---------------------------------------------------------------------------
# 8. denormalize
# ---------------------------------------------------------------------------
def test_denormalize_endpoints_and_dtype():
    x = torch.tensor([-1.0, 0.0, 1.0, -5.0, 5.0]).view(1, 1, 5)
    out = denormalize(x)
    assert out.dtype == torch.uint8
    assert out.flatten().tolist() == [0, 128, 255, 0, 255]  # di luar rentang di-clamp


@pytest.mark.parametrize("shape", [(3, 16, 16), (2, 3, 16, 16)])
def test_denormalize_roundtrip_exact(shape):
    u8 = torch.randint(0, 256, shape, dtype=torch.uint8)
    normalized = u8.float() / 127.5 - 1.0          # sama dengan to_tensor
    assert torch.equal(denormalize(normalized), u8)
''', baru=False)


In [ ]:
#@title tests/test_dataset.py · 15/15: test_preview_script_writes_grid
tulis('tests/test_dataset.py', r'''

# ---------------------------------------------------------------------------
# 9. Skrip grid pratinjau (smoke test)
# ---------------------------------------------------------------------------
def test_preview_script_writes_grid(flat_root, tmp_path, capsys):
    from scripts.preview_pairs import HEADER_H, main
    out = main(["--root", str(flat_root), "--split", "train", "--n", "3",
                "--img-size", "64", "--load-size", "72", "--augment",
                "--out", str(tmp_path / "grid.png")])
    with Image.open(out) as grid:
        assert grid.size == (2 * 64 + 4, HEADER_H + 3 * (64 + 4))
    printed = capsys.readouterr().out
    assert "A lebih rata" in printed            # label sintetis ada di kiri
    assert "isi file identik = 0" in printed
''', baru=False)


### `tests/test_config.py`

Tes config. 11 sel, 299 baris. Isi sel → 1: `import dan konstanta` · 2: `test_config_has_all_dataloader_fields` · 3: `test_integration_get_dataloader_with_config` · 4: `test_json_roundtrip_is_identical` · 5: `test_load_validates_values_and_types` · 6: `_draw` · 7: `test_invalid_values_raise` · 8: `test_wrong_types_raise` · 9: `_diff` · 10: `test_get_device_cuda_without_gpu_raises` · 11: `test_import_has_no_side_effects`

In [ ]:
#@title tests/test_config.py · 1/11: import dan konstanta
tulis('tests/test_config.py', r'''"""
tests/test_config.py

Tes untuk config.py. Jalankan dari root repo:
    pytest tests/test_config.py -v
"""

import dataclasses
import json
import os
import random
import subprocess
import sys
from pathlib import Path

import numpy as np
import pytest
import torch
from PIL import Image

import config as config_module
from config import (
    ABLATION_EPOCHS,
    PRESETS,
    Config,
    amp_enabled,
    get_device,
    get_preset,
    load_config,
    run_dir,
    save_config,
    set_seed,
)

REPO_ROOT = Path(__file__).resolve().parents[1]

# Field yang dibaca get_dataloader di dataset.py (label_resize_mode opsional di sana).
DATALOADER_FIELDS = (
    "data_root", "direction", "img_size", "load_size", "batch_size", "num_workers",
    "pin_memory", "seed", "subset_size", "manifest_path", "split_seed",
    "split_ratios", "label_resize_mode",
)
''', baru=True)


In [ ]:
#@title tests/test_config.py · 2/11: test_config_has_all_dataloader_fields
tulis('tests/test_config.py', r'''

# ---------------------------------------------------------------------------
# 1. Kecocokan dengan dataset.py
# ---------------------------------------------------------------------------
def test_config_has_all_dataloader_fields():
    names = {f.name for f in dataclasses.fields(Config)}
    missing = [n for n in DATALOADER_FIELDS if n not in names]
    assert not missing, f"Field yang dibutuhkan get_dataloader tidak ada: {missing}"


def _make_png_jpg_pairs(root: Path, n: int = 12, size: int = 64):
    """n pasangan format terpisah, meniru CMP base: label .png mode palet
    (indeks kelas + peta warna) dan foto .jpg RGB dengan nama dasar sama."""
    rng = np.random.default_rng(0)
    root.mkdir(parents=True)
    palette = [0, 0, 0] + [c for k in range(1, 13) for c in (k * 20, 255 - k * 20, 128)]
    for i in range(n):
        idx = np.kron(rng.integers(1, 13, (4, 4)), np.ones((size // 4, size // 4))).astype(np.uint8)
        label = Image.fromarray(idx, mode="P")
        label.putpalette(palette)
        label.save(root / f"cmp_b{i:04d}.png")
        photo = rng.integers(0, 256, (size, size, 3), dtype=np.uint8)
        Image.fromarray(photo).save(root / f"cmp_b{i:04d}.jpg", quality=95)
''', baru=False)


In [ ]:
#@title tests/test_config.py · 3/11: test_integration_get_dataloader_with_config
tulis('tests/test_config.py', r'''

@pytest.mark.parametrize("num_workers", [0, 2])
def test_integration_get_dataloader_with_config(tmp_path, num_workers):
    from dataset import get_dataloader

    data = tmp_path / "facades"
    _make_png_jpg_pairs(data)
    # img_size harus 128/256 (validasi), jadi 128/144 dipakai, bukan 32/36.
    cfg = Config(data_root=str(data), img_size=128, load_size=144, batch_size=2,
                 num_workers=num_workers, manifest_path=str(tmp_path / "manifest.json"))
    set_seed(cfg.seed)
    batch = next(iter(get_dataloader(cfg, "train")))
    assert batch["A"].shape == batch["B"].shape == (2, 3, 128, 128)
    assert batch["A"].dtype == torch.float32
    assert batch["A"].min() >= -1 and batch["A"].max() <= 1
    assert all(p.endswith(".png") for p in batch["path"])   # A = label untuk AtoB
''', baru=False)


In [ ]:
#@title tests/test_config.py · 4/11: test_json_roundtrip_is_identical
tulis('tests/test_config.py', r'''

# ---------------------------------------------------------------------------
# 2. Simpan / muat JSON
# ---------------------------------------------------------------------------
def test_json_roundtrip_is_identical(tmp_path):
    cfg = Config(split_ratios=(0.7, 0.2, 0.1), subset_size=8, use_skip=False,
                 manifest_path="splits/m.json", lambda_l1=10.0, run_name="coba")
    path = save_config(cfg, tmp_path / "run")
    assert path == tmp_path / "run" / "config.json"
    assert json.loads(path.read_text())["split_ratios"] == [0.7, 0.2, 0.1]  # JSON: list

    loaded = load_config(path)
    assert loaded == cfg
    assert isinstance(loaded.split_ratios, tuple)


def test_load_rejects_unknown_key(tmp_path):
    path = save_config(Config(), tmp_path)
    data = json.loads(path.read_text())
    data["lamda_l1"] = 50          # salah ketik
    path.write_text(json.dumps(data))
    with pytest.raises(ValueError, match="tidak dikenal"):
        load_config(path)


def test_load_missing_key_rejected_unless_allowed(tmp_path):
    path = save_config(Config(), tmp_path)
    data = json.loads(path.read_text())
    del data["log_every"]
    path.write_text(json.dumps(data))
    with pytest.raises(ValueError, match="hilang"):
        load_config(path)
    assert load_config(path, allow_missing=True).log_every == Config().log_every
''', baru=False)


In [ ]:
#@title tests/test_config.py · 5/11: test_load_validates_values_and_types
tulis('tests/test_config.py', r'''

def test_load_validates_values_and_types(tmp_path):
    path = save_config(Config(), tmp_path)
    data = json.loads(path.read_text())
    data["use_skip"] = "false"     # string dari edit tangan
    path.write_text(json.dumps(data))
    with pytest.raises(TypeError, match="use_skip"):
        load_config(path)
''', baru=False)


In [ ]:
#@title tests/test_config.py · 6/11: _draw
tulis('tests/test_config.py', r'''

# ---------------------------------------------------------------------------
# 3. set_seed
# ---------------------------------------------------------------------------
def _draw():
    return torch.randn(5), random.random(), np.random.rand(5)


def test_set_seed_makes_all_rngs_identical():
    set_seed(123)
    t1, r1, n1 = _draw()
    set_seed(123)
    t2, r2, n2 = _draw()
    assert torch.equal(t1, t2) and r1 == r2 and np.array_equal(n1, n2)

    set_seed(124)
    t3, r3, n3 = _draw()
    assert not torch.equal(t1, t3) and r1 != r3 and not np.array_equal(n1, n3)


def test_set_seed_deterministic_flags():
    old = (torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark)
    try:
        torch.backends.cudnn.benchmark = True
        set_seed(0, deterministic=True)
        assert torch.backends.cudnn.deterministic is True
        assert torch.backends.cudnn.benchmark is False
        set_seed(0, deterministic=False)
        assert torch.backends.cudnn.deterministic is False
    finally:
        torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = old
''', baru=False)


In [ ]:
#@title tests/test_config.py · 7/11: test_invalid_values_raise
tulis('tests/test_config.py', r'''

# ---------------------------------------------------------------------------
# 4. Validasi: setiap nilai tidak valid memicu error
# ---------------------------------------------------------------------------
INVALID = [
    {"img_size": 64}, {"img_size": 512},
    {"img_size": 256, "load_size": 255},
    {"split_ratios": (0.8, 0.2)}, {"split_ratios": (0.8, 0.1, 0.2)},
    {"split_ratios": (0.9, 0.1, 0.0)}, {"split_ratios": (1.1, -0.05, -0.05)},
    {"direction": "A2B"},
    {"norm": "group"},
    {"label_resize_mode": "lanczos"},
    {"batch_size": 0},
    {"num_workers": -1},
    {"subset_size": 0},
    {"ngf": 0}, {"ndf": 0}, {"n_layers": 0}, {"epochs": 0},
    {"dropout_p": 1.0}, {"dropout_p": -0.1},
    {"lambda_l1": -1.0},
    {"lr": 0.0}, {"lr": -1e-4},
    {"device": "mps"},
    {"log_every": 0},
    {"run_name": ""}, {"run_name": "a/b"},
]


@pytest.mark.parametrize("bad", INVALID, ids=lambda d: ",".join(f"{k}={v}" for k, v in d.items()))
def test_invalid_values_raise(bad):
    with pytest.raises(ValueError, match="Config tidak valid"):
        Config(**bad)


@pytest.mark.parametrize("bad", INVALID[:6], ids=lambda d: ",".join(d))
def test_replace_is_validated_too(bad):
    with pytest.raises(ValueError):
        dataclasses.replace(Config(), **bad)


@pytest.mark.parametrize("bad", [{"batch_size": 2.0}, {"use_skip": 1}, {"seed": "42"},
                                 {"lr": True}, {"subset_size": 8.5}])
''', baru=False)


In [ ]:
#@title tests/test_config.py · 8/11: test_wrong_types_raise
tulis('tests/test_config.py', r'''def test_wrong_types_raise(bad):
    with pytest.raises(TypeError):
        Config(**bad)


def test_valid_edge_values_accepted():
    Config(img_size=128, load_size=128, dropout_p=0.0, lambda_l1=0.0,
           num_workers=0, subset_size=1, device="cpu", lr=1)


def test_config_is_frozen():
    with pytest.raises(dataclasses.FrozenInstanceError):
        Config().batch_size = 4
''', baru=False)


In [ ]:
#@title tests/test_config.py · 9/11: _diff
tulis('tests/test_config.py', r'''

# ---------------------------------------------------------------------------
# 5. Preset ablation hanya mengubah field yang dimaksud
# ---------------------------------------------------------------------------
def _diff(a: Config, b: Config) -> dict:
    return {f.name: getattr(b, f.name) for f in dataclasses.fields(Config)
            if getattr(a, f.name) != getattr(b, f.name)}


@pytest.mark.parametrize("name,expected_extra", [
    ("lambda10", {"lambda_l1": 10.0}),
    ("lambda100", {}),
    ("lambda200", {"lambda_l1": 200.0}),
    ("noskip", {"use_skip": False}),
    ("nlayers1", {"n_layers": 1}),
    ("lr1e-4", {"lr": 1e-4}),
    ("lr4e-4", {"lr": 4e-4}),
    ("batch1", {"batch_size": 1}),
    ("batch4", {"batch_size": 4}),
])
def test_preset_changes_only_intended_fields(name, expected_extra):
    base = Config()
    preset = get_preset(name)
    assert _diff(base, preset) == {"epochs": ABLATION_EPOCHS, "run_name": name, **expected_extra}
    assert preset.seed == base.seed


def test_preset_uses_given_base_and_keeps_its_seed():
    base = Config(seed=7, batch_size=1, ngf=32)
    preset = get_preset("noskip", base)
    assert _diff(base, preset) == {"epochs": ABLATION_EPOCHS, "run_name": "noskip", "use_skip": False}


def test_all_presets_listed_and_unknown_rejected():
    assert set(PRESETS) == {"lambda10", "lambda100", "lambda200", "noskip", "nlayers1",
                            "lr1e-4", "lr4e-4", "batch1", "batch4"}
    with pytest.raises(ValueError, match="tidak dikenal"):
        get_preset("nlayers3")
''', baru=False)


In [ ]:
#@title tests/test_config.py · 10/11: test_get_device_cuda_without_gpu_raises
tulis('tests/test_config.py', r'''

# ---------------------------------------------------------------------------
# 6. Device, AMP, run_dir
# ---------------------------------------------------------------------------
@pytest.mark.skipif(torch.cuda.is_available(), reason="hanya relevan di mesin tanpa GPU")
def test_get_device_cuda_without_gpu_raises():
    with pytest.raises(RuntimeError, match="CUDA tidak tersedia"):
        get_device("cuda")
    with pytest.raises(RuntimeError):
        get_device(Config(device="cuda"))


def test_get_device_auto(monkeypatch):
    monkeypatch.setattr(torch.cuda, "is_available", lambda: False)
    assert get_device(Config()).type == "cpu"
    monkeypatch.setattr(torch.cuda, "is_available", lambda: True)
    assert get_device("auto").type == "cuda"
    assert get_device("cpu").type == "cpu"


def test_amp_only_on_cuda():
    cfg = Config(use_amp=True)
    assert amp_enabled(cfg, torch.device("cpu")) is False
    assert amp_enabled(cfg, torch.device("cuda")) is True
    assert amp_enabled(Config(use_amp=False), torch.device("cuda")) is False


def test_run_dir_does_not_create_folder(tmp_path):
    cfg = Config(checkpoint_dir=str(tmp_path / "runs"), run_name="lambda10")
    assert run_dir(cfg) == tmp_path / "runs" / "lambda10"
    assert not (tmp_path / "runs").exists()
''', baru=False)


In [ ]:
#@title tests/test_config.py · 11/11: test_import_has_no_side_effects
tulis('tests/test_config.py', r'''

# ---------------------------------------------------------------------------
# 7. Tanpa efek samping saat import dan tanpa environment variable
# ---------------------------------------------------------------------------
def test_import_has_no_side_effects(tmp_path):
    env = {**os.environ, "PYTHONPATH": str(REPO_ROOT)}
    subprocess.run([sys.executable, "-c", "import config; config.Config()"],
                   cwd=tmp_path, env=env, check=True)
    assert list(tmp_path.iterdir()) == []      # tidak ada file/folder yang dibuat


def test_source_does_not_read_environment():
    src = Path(config_module.__file__).read_text(encoding="utf-8")
    # Pola pemakaian nyata, bukan sekadar kata "environment" di docstring.
    for pattern in ("os.environ", "getenv(", "import os"):
        assert pattern not in src, f"config.py memakai {pattern!r}"
''', baru=False)


### `tests/test_measure_vram.py`

Tes skrip VRAM. 5 sel, 143 baris. Isi sel → 1: `_tiny_cpu_args` · 2: `test_main_runs_on_cpu_and_writes_csv` · 3: `never_steps` · 4: `fake_oom` · 5: `test_cuda_required_by_default`

In [ ]:
#@title tests/test_measure_vram.py · 1/5: _tiny_cpu_args
tulis('tests/test_measure_vram.py', r'''"""
tests/test_measure_vram.py

Tes jalan-kode untuk scripts/measure_vram.py. Tes ini jalan di CPU dengan
model kecil (ngf=8), jadi TIDAK mengukur memori. Angka VRAM yang sebenarnya
hanya bisa didapat dengan menjalankan skripnya di GPU kalian.
"""

import csv

import pytest
import torch

from models.discriminator import PatchDiscriminator
from models.generator import UNetGenerator
from scripts import measure_vram as mv


def _tiny_cpu_args(tmp_path, *extra):
    return ["--device", "cpu", "--img-sizes", "128", "--batches", "1", "2",
            "--ngfs", "8", "--ndf", "8", "--out", str(tmp_path / "vram.csv"), *extra]
''', baru=True)


In [ ]:
#@title tests/test_measure_vram.py · 2/5: test_main_runs_on_cpu_and_writes_csv
tulis('tests/test_measure_vram.py', r'''

def test_main_runs_on_cpu_and_writes_csv(tmp_path):
    results = mv.main(_tiny_cpu_args(tmp_path))
    # 1 img_size x 2 batch x 2 AMP x 1 ngf = 4 kombinasi
    assert len(results) == 4
    statuses = sorted(r.status for r in results)
    assert statuses == ["ok", "ok", "skipped", "skipped"]  # AMP dilewati di CPU
    for r in results:
        if r.status == "ok":
            assert r.g_params > 0 and r.d_params > 0 and r.iter_ms > 0
            assert r.peak_alloc_mib is None    # memori tidak diukur di CPU
            # Tanpa AMP, 1 iterasi pemanasan cukup untuk membentuk state Adam;
            # lalu 3 iterasi terukur (default) -> 4 step Adam untuk G dan D.
            assert r.warmup_iters == 1 and r.adam_ready is True
            assert r.g_steps == 4 and r.d_steps == 4
            assert r.note == ""

    with open(tmp_path / "vram.csv", newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    assert len(rows) == 4
    assert {"img_size", "batch", "amp", "ngf", "status", "peak_alloc_mib", "peak_reserved_mib",
            "steady_alloc_mib", "warmup_iters", "adam_ready", "g_steps", "d_steps"} <= set(rows[0])


def test_adam_ready_and_steps_track_real_optimizer_steps():
    model = torch.nn.Linear(3, 2)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    assert mv._adam_ready(opt) is False and mv._adam_steps(opt) == 0
    for _ in range(2):
        model(torch.randn(4, 3)).sum().backward()
        opt.step()
    assert mv._adam_ready(opt) is True and mv._adam_steps(opt) == 2


def test_skipped_steps_are_reported_not_hidden(tmp_path, monkeypatch):
    """Simulasi GradScaler yang selalu melewati step: state Adam tidak pernah
    terbentuk. Pemanasan harus berhenti di --max-warmup dan hasilnya ditandai."""
    calls = []
''', baru=False)


In [ ]:
#@title tests/test_measure_vram.py · 3/5: never_steps
tulis('tests/test_measure_vram.py', r'''
    def never_steps(*args, **kwargs):
        calls.append(1)
        return torch.tensor(0.0), torch.tensor(0.0)

    monkeypatch.setattr(mv, "train_step", never_steps)
    results = mv.main(_tiny_cpu_args(tmp_path, "--batches", "1", "--max-warmup", "3", "--iters", "2"))
    r = next(r for r in results if r.status == "ok")
    assert len(calls) == 3 + 2                   # 3 pemanasan (batas) + 2 terukur
    assert r.warmup_iters == 3 and r.adam_ready is False
    assert r.g_steps == 0 and "state Adam belum terbentuk" in r.note


def test_train_step_updates_both_models():
    torch.manual_seed(0)
    G = UNetGenerator(ngf=8, img_size=128).train()
    D = PatchDiscriminator(ndf=8).train()
    g_before = [p.detach().clone() for p in G.parameters()]
    d_before = [p.detach().clone() for p in D.parameters()]
    opt_g = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
    cond, real = torch.randn(2, 3, 128, 128), torch.randn(2, 3, 128, 128)

    loss_d, loss_g = mv.train_step(G, D, opt_g, opt_d, None, None, cond, real,
                                   amp=False, lambda_l1=100.0)
    assert torch.isfinite(loss_d) and torch.isfinite(loss_g)
    assert any(not torch.equal(a, b) for a, b in zip(g_before, G.parameters()))
    assert any(not torch.equal(a, b) for a, b in zip(d_before, D.parameters()))


def test_uses_the_same_training_step_as_train_py():
    """Satu sumber kode: yang diukur adalah langkah training yang sesungguhnya."""
    import train as tr
    assert mv._train_step is tr.train_step
    assert mv.make_grad_scaler is tr.make_grad_scaler


def test_oom_is_recorded_and_sweep_continues(tmp_path, monkeypatch):
    calls = []
''', baru=False)


In [ ]:
#@title tests/test_measure_vram.py · 4/5: fake_oom
tulis('tests/test_measure_vram.py', r'''
    def fake_oom(*args, **kwargs):
        calls.append(1)
        raise torch.cuda.OutOfMemoryError("CUDA out of memory. Tried to allocate 2.00 GiB")

    monkeypatch.setattr(mv, "train_step", fake_oom)
    results = mv.main(_tiny_cpu_args(tmp_path))
    ok_combos = [r for r in results if r.status != "skipped"]
    assert len(calls) == 2                       # sweep tidak berhenti di OOM pertama
    assert all(r.status == "OOM" for r in ok_combos)
    assert all("out of memory" in r.note for r in ok_combos)


def test_non_oom_error_is_not_swallowed(tmp_path, monkeypatch):
    def bug(*args, **kwargs):
        raise RuntimeError("mat1 and mat2 shapes cannot be multiplied")

    monkeypatch.setattr(mv, "train_step", bug)
    with pytest.raises(RuntimeError, match="shapes"):
        mv.main(_tiny_cpu_args(tmp_path))


@pytest.mark.parametrize("err,expected", [
    (torch.cuda.OutOfMemoryError("CUDA out of memory"), True),
    (RuntimeError("CUDA error: out of memory"), True),
    (RuntimeError("CUBLAS_STATUS_ALLOC_FAILED when calling cublasCreate"), True),
    (RuntimeError("size mismatch"), False),
    (ValueError("out of memory"), False),
])
def test_is_oom(err, expected):
    assert mv._is_oom(err) is expected


def test_table_shows_dash_for_missing_values():
    r = mv.ComboResult(256, 4, True, 64, "OOM", None, None, None, None, None,
                       None, None, None, None, None, None, "")
    table = mv.format_table([r])
    assert "OOM" in table and " - " in table
''', baru=False)


In [ ]:
#@title tests/test_measure_vram.py · 5/5: test_cuda_required_by_default
tulis('tests/test_measure_vram.py', r'''

def test_cuda_required_by_default(monkeypatch):
    monkeypatch.setattr(torch.cuda, "is_available", lambda: False)
    with pytest.raises(SystemExit, match="CUDA tidak tersedia"):
        mv.main(["--out", "unused.csv"])
''', baru=False)


### `tests/test_losses.py`

Tes loss. 7 sel, 190 baris. Isi sel → 1: `fixed_seed` · 2: `test_adversarial_loss_any_patch_size` · 3: `test_zero_logits_give_ln2` · 4: `test_non_saturating_gradient_is_large_when_d_rejects_fake` · 5: `test_detach_controls_whether_g_receives_gradient_from_d_loss` · 6: `test_half_precision_inputs_give_float32_loss` · 7: `test_rejects_invalid_inputs`

In [ ]:
#@title tests/test_losses.py · 1/7: fixed_seed
tulis('tests/test_losses.py', r'''"""
tests/test_losses.py

Tes untuk losses.py. Jalankan dari root repo:
    pytest tests/test_losses.py -v
"""

import math

import pytest
import torch
import torch.nn.functional as F

from losses import AdversarialLoss, discriminator_loss, generator_loss
from models.discriminator import PatchDiscriminator
from models.generator import UNetGenerator

LN2 = math.log(2.0)


@pytest.fixture(autouse=True)
def fixed_seed():
    torch.manual_seed(0)
''', baru=True)


In [ ]:
#@title tests/test_losses.py · 2/7: test_adversarial_loss_any_patch_size
tulis('tests/test_losses.py', r'''

# ---------------------------------------------------------------------------
# 1. Shape: target mengikuti peta patch, output skalar
# ---------------------------------------------------------------------------
@pytest.mark.parametrize("patch", [30, 126, 14, 62])   # 256/128 x n_layers 3/1
@pytest.mark.parametrize("target_is_real", [True, False])
def test_adversarial_loss_any_patch_size(patch, target_is_real):
    logits = torch.randn(2, 1, patch, patch)
    loss = AdversarialLoss()(logits, target_is_real)
    assert loss.dim() == 0 and torch.isfinite(loss)


@pytest.mark.parametrize("n_layers,expected", [(3, 30), (1, 126)])
def test_losses_with_real_discriminator_output(n_layers, expected):
    D = PatchDiscriminator(ndf=8, n_layers=n_layers)
    cond, real, fake = (torch.rand(2, 3, 256, 256) * 2 - 1 for _ in range(3))
    d_real, d_fake = D(cond, real), D(cond, fake)
    assert d_real.shape == (2, 1, expected, expected)
    total_d, parts_d = discriminator_loss(d_real, d_fake)
    total_g, parts_g = generator_loss(d_fake, fake, real, lambda_l1=100.0)
    for t in (total_d, total_g, *parts_d.values(), *parts_g.values()):
        assert t.dim() == 0 and torch.isfinite(t)
''', baru=False)


In [ ]:
#@title tests/test_losses.py · 3/7: test_zero_logits_give_ln2
tulis('tests/test_losses.py', r'''

# ---------------------------------------------------------------------------
# 2. Nilai dibandingkan dengan hitungan manual
# ---------------------------------------------------------------------------
def test_zero_logits_give_ln2():
    """Logit 0 -> sigmoid 0.5 -> BCE = -ln(0.5) = ln 2, untuk target 1 maupun 0."""
    z = torch.zeros(2, 1, 30, 30)
    total, parts = discriminator_loss(z, z, scale=0.5)
    assert parts["d_real"].item() == pytest.approx(LN2)
    assert parts["d_fake"].item() == pytest.approx(LN2)
    assert total.item() == pytest.approx(0.5 * (LN2 + LN2))
    total1, _ = discriminator_loss(z, z, scale=1.0)
    assert total1.item() == pytest.approx(2 * LN2)


def test_generator_loss_manual_value():
    z = torch.zeros(2, 1, 30, 30)
    fake = torch.zeros(2, 3, 8, 8)
    real = torch.full((2, 3, 8, 8), 0.5)        # |fake - real| = 0.5 di setiap piksel
    total, parts = generator_loss(z, fake, real, lambda_l1=100.0)
    assert parts["g_adv"].item() == pytest.approx(LN2)
    assert parts["g_l1"].item() == pytest.approx(0.5)
    assert total.item() == pytest.approx(LN2 + 100.0 * 0.5)


def test_confident_correct_logits_give_near_zero_loss():
    big = torch.full((1, 1, 30, 30), 20.0)
    assert AdversarialLoss()(big, True).item() < 1e-6     # yakin real, memang real
    assert AdversarialLoss()(-big, False).item() < 1e-6   # yakin fake, memang fake
    assert AdversarialLoss()(big, False).item() == pytest.approx(20.0, rel=1e-3)  # yakin tapi salah


def test_lambda_zero_and_l1_reported_before_weighting():
    z = torch.zeros(1, 1, 30, 30)
    fake, real = torch.zeros(1, 3, 8, 8), torch.ones(1, 3, 8, 8)
    total0, parts0 = generator_loss(z, fake, real, lambda_l1=0.0)
    assert total0.item() == pytest.approx(parts0["g_adv"].item())
    _, parts200 = generator_loss(z, fake, real, lambda_l1=200.0)
    assert parts200["g_l1"].item() == pytest.approx(1.0)   # g_l1 tidak dikali lambda
''', baru=False)


In [ ]:
#@title tests/test_losses.py · 4/7: test_non_saturating_gradient_is_large_when_d_rejects_fake
tulis('tests/test_losses.py', r'''

# ---------------------------------------------------------------------------
# 3. Non-saturating: gradien tetap besar saat D yakin gambar palsu
# ---------------------------------------------------------------------------
def test_non_saturating_gradient_is_large_when_d_rejects_fake():
    logits = torch.full((1, 1, 30, 30), -10.0, requires_grad=True)   # D: "jelas palsu"
    n = logits.numel()

    # Bentuk yang dipakai: -log D(G(x)), lewat generator_loss
    total, _ = generator_loss(logits, torch.zeros(1, 3, 8, 8), torch.zeros(1, 3, 8, 8), lambda_l1=0.0)
    grad_ns = torch.autograd.grad(total, logits)[0]
    # Bentuk minimax asli: minimalkan log(1 - D(G(x))) = -BCE(logit, 0)
    saturating = -F.binary_cross_entropy_with_logits(logits, torch.zeros_like(logits))
    grad_sat = torch.autograd.grad(saturating, logits)[0]

    assert abs(grad_ns.sum().item()) > 0.99            # turunan per logit ~ -1/n, totalnya ~1
    assert abs(grad_sat.sum().item()) < 1e-3           # hampir nol: G tidak belajar
    assert grad_ns.sum().item() < 0                    # arah: menaikkan logit (lebih "real")
''', baru=False)


In [ ]:
#@title tests/test_losses.py · 5/7: test_detach_controls_whether_g_receives_gradient_from_d_loss
tulis('tests/test_losses.py', r'''

# ---------------------------------------------------------------------------
# 4. Detach adalah tanggung jawab train.py (losses.py tidak meng-detach)
# ---------------------------------------------------------------------------
def test_detach_controls_whether_g_receives_gradient_from_d_loss():
    G = UNetGenerator(ngf=8, img_size=128)
    D = PatchDiscriminator(ndf=8)
    cond, real = torch.randn(1, 3, 128, 128), torch.randn(1, 3, 128, 128)
    fake = G(cond)

    loss_d, _ = discriminator_loss(D(cond, real), D(cond, fake.detach()))
    loss_d.backward()
    assert all(p.grad is None for p in G.parameters())           # benar: G tidak tersentuh
    assert any(p.grad is not None for p in D.parameters())

    D.zero_grad(set_to_none=True)
    loss_wrong, _ = discriminator_loss(D(cond, real), D(cond, fake))   # LUPA detach
    loss_wrong.backward()
    assert any(p.grad is not None for p in G.parameters())       # G ikut menerima gradien D


def test_generator_loss_sends_gradient_to_g():
    G = UNetGenerator(ngf=8, img_size=128)
    D = PatchDiscriminator(ndf=8)
    cond, real = torch.randn(1, 3, 128, 128), torch.randn(1, 3, 128, 128)
    fake = G(cond)
    total, parts = generator_loss(D(cond, fake), fake, real, lambda_l1=100.0)
    total.backward()
    assert all(p.grad is not None for p in G.parameters())
    assert total.requires_grad and not parts["g_adv"].requires_grad   # dict = nilai untuk log saja
''', baru=False)


In [ ]:
#@title tests/test_losses.py · 6/7: test_half_precision_inputs_give_float32_loss
tulis('tests/test_losses.py', r'''

# ---------------------------------------------------------------------------
# 5. Presisi: selalu float32, juga dengan input float16 / autocast
# ---------------------------------------------------------------------------
def test_half_precision_inputs_give_float32_loss():
    logits = torch.randn(2, 1, 30, 30).half()
    fake = (torch.rand(2, 3, 16, 16) * 2 - 1).half()
    real = torch.rand(2, 3, 16, 16) * 2 - 1
    total_g, _ = generator_loss(logits, fake, real, 100.0)
    total_d, _ = discriminator_loss(logits, logits)
    assert total_g.dtype == torch.float32 and total_d.dtype == torch.float32


def test_under_cpu_autocast():
    G = UNetGenerator(ngf=8, img_size=128)
    D = PatchDiscriminator(ndf=8)
    cond, real = torch.randn(1, 3, 128, 128), torch.randn(1, 3, 128, 128)
    with torch.autocast(device_type="cpu", dtype=torch.bfloat16):
        fake = G(cond)
        total, _ = generator_loss(D(cond, fake), fake, real, 100.0)
    assert total.dtype == torch.float32 and torch.isfinite(total)
    total.backward()


@pytest.mark.skipif(not torch.cuda.is_available(), reason="butuh GPU")
def test_under_cuda_autocast_fp16():
    G = UNetGenerator(ngf=8, img_size=128).cuda()
    D = PatchDiscriminator(ndf=8).cuda()
    cond, real = torch.randn(2, 3, 128, 128, device="cuda"), torch.randn(2, 3, 128, 128, device="cuda")
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        fake = G(cond)
        loss_d, _ = discriminator_loss(D(cond, real), D(cond, fake.detach()))
        loss_g, _ = generator_loss(D(cond, fake), fake, real, 100.0)
    for loss in (loss_d, loss_g):
        assert loss.dtype == torch.float32 and torch.isfinite(loss)
    loss_g.backward()
''', baru=False)


In [ ]:
#@title tests/test_losses.py · 7/7: test_rejects_invalid_inputs
tulis('tests/test_losses.py', r'''

# ---------------------------------------------------------------------------
# 6. Validasi input
# ---------------------------------------------------------------------------
def test_rejects_invalid_inputs():
    z = torch.zeros(1, 1, 30, 30)
    with pytest.raises(ValueError, match="Shape logit"):
        discriminator_loss(z, torch.zeros(1, 1, 14, 14))
    with pytest.raises(ValueError, match="scale"):
        discriminator_loss(z, z, scale=0.0)
    with pytest.raises(ValueError, match="Shape fake"):
        generator_loss(z, torch.zeros(1, 3, 8, 8), torch.zeros(1, 3, 16, 16), 100.0)
    with pytest.raises(ValueError, match="lambda_l1"):
        generator_loss(z, torch.zeros(1, 3, 8, 8), torch.zeros(1, 3, 8, 8), -1.0)
    with pytest.raises(TypeError, match="bool"):
        AdversarialLoss()(z, 1)
''', baru=False)


### `tests/test_train.py`

Tes training: pemetaan config, D terpisah, detach, mode, resume, overfit. 9 sel, 267 baris. Isi sel → 1: `fixed_seed` · 2: `tiny_cfg` · 3: `test_build_models_maps_every_field` · 4: `test_d_called_separately_and_detach_only_for_d_update` · 5: `test_d_step_does_not_touch_g` · 6: `test_evaluate_and_samples_restore_train_mode` · 7: `test_full_tiny_run_writes_everything` · 8: `test_resume_continues_epochs` · 9: `test_parse_overrides_types`

In [ ]:
#@title tests/test_train.py · 1/9: fixed_seed
tulis('tests/test_train.py', r'''"""
tests/test_train.py

Tes untuk train.py. Semua jalan di CPU dengan model kecil (ngf/ndf = 8, 128x128)
dan data sintetis format CMP (label .png palet + foto .jpg).
Jalankan: pytest tests/test_train.py -v
"""

import csv
import dataclasses

import numpy as np
import pytest
import torch
import torch.nn as nn
from PIL import Image

import train as tr
from config import Config
from tests.helpers import make_pairs


@pytest.fixture(autouse=True)
def fixed_seed():
    torch.manual_seed(0)
''', baru=True)


In [ ]:
#@title tests/test_train.py · 2/9: tiny_cfg
tulis('tests/test_train.py', r'''

# ---------------------------------------------------------------------------
# Data dan config kecil
# ---------------------------------------------------------------------------
@pytest.fixture
def tiny_cfg(tmp_path):
    make_pairs(tmp_path / "data")
    return Config(data_root=str(tmp_path / "data"), img_size=128, load_size=144, batch_size=2,
                  num_workers=0, pin_memory=False, manifest_path=str(tmp_path / "manifest.json"),
                  ngf=8, ndf=8, epochs=2, log_every=1, device="cpu",
                  checkpoint_dir=str(tmp_path / "runs"), run_name="tes")


def read_csv(path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))
''', baru=False)


In [ ]:
#@title tests/test_train.py · 3/9: test_build_models_maps_every_field
tulis('tests/test_train.py', r'''

# ---------------------------------------------------------------------------
# 1. Config dipetakan benar ke konstruktor (nama dan nilai)
# ---------------------------------------------------------------------------
@pytest.mark.parametrize("over", [
    {},
    {"use_dropout": False},
    {"n_dropout": 1, "dropout_p": 0.3},
    {"norm": "instance"},
    {"norm": "none", "use_skip": False},
    {"n_layers": 1, "max_mult": 4},
])
def test_build_models_maps_every_field(tiny_cfg, over):
    cfg = dataclasses.replace(tiny_cfg, **over)
    G, D = tr.build_models(cfg, torch.device("cpu"))

    dropouts = [b.dropout for b in G.decoder if isinstance(b.dropout, nn.Dropout)]
    assert len(dropouts) == (cfg.n_dropout if cfg.use_dropout else 0)
    assert all(d.p == cfg.dropout_p for d in dropouts)
    assert G.use_skip == cfg.use_skip and G.img_size == cfg.img_size
    assert G.enc_channels[0] == cfg.ngf and max(G.enc_channels) == cfg.ngf * min(cfg.max_mult, 2 ** 6)

    norm_type = {"batch": nn.BatchNorm2d, "instance": nn.InstanceNorm2d, "none": nn.Identity}[cfg.norm]
    assert isinstance(G.encoder[1].norm, norm_type)
    assert D.n_layers == cfg.n_layers
    assert D.net[0].out_channels == cfg.ndf
    assert any(isinstance(m, norm_type) for m in D.net) or cfg.norm == "none"
    leaky = [m for m in G.modules() if isinstance(m, nn.LeakyReLU)]
    assert all(m.negative_slope == cfg.leaky_slope for m in leaky)


def test_optimizers_use_config_values(tiny_cfg):
    cfg = dataclasses.replace(tiny_cfg, lr=1e-3, beta1=0.4, beta2=0.99)
    G, D = tr.build_models(cfg, torch.device("cpu"))
    for opt in tr.build_optimizers(cfg, G, D):
        g = opt.param_groups[0]
        assert g["lr"] == 1e-3 and g["betas"] == (0.4, 0.99)
''', baru=False)


In [ ]:
#@title tests/test_train.py · 4/9: test_d_called_separately_and_detach_only_for_d_update
tulis('tests/test_train.py', r'''

# ---------------------------------------------------------------------------
# 2. D dipanggil TERPISAH; fake di-detach HANYA saat update D
# ---------------------------------------------------------------------------
def test_d_called_separately_and_detach_only_for_d_update(tiny_cfg):
    G, D = tr.build_models(tiny_cfg, torch.device("cpu"))
    opt_g, opt_d = tr.build_optimizers(tiny_cfg, G, D)
    cond = torch.randn(2, 3, 128, 128)
    real = torch.randn(2, 3, 128, 128)

    calls = []
    def spy(module, inputs):
        _, img = inputs
        calls.append({"batch": img.shape[0], "img_requires_grad": img.requires_grad,
                      "d_params_trainable": next(module.parameters()).requires_grad})
    handle = D.register_forward_pre_hook(spy)
    tr.train_step(G, D, opt_g, opt_d, None, None, cond, real, amp=False, lambda_l1=100.0, d_loss_scale=0.5)
    handle.remove()

    assert len(calls) == 3
    assert [c["batch"] for c in calls] == [2, 2, 2]               # tidak pernah real+fake digabung (4)
    assert [c["img_requires_grad"] for c in calls] == [False, False, True]   # real, fake.detach(), fake
    assert [c["d_params_trainable"] for c in calls] == [True, True, False]   # D dibekukan saat update G
    assert all(p.requires_grad for p in D.parameters())          # dipulihkan setelahnya


def test_train_step_updates_both_and_returns_losses(tiny_cfg):
    G, D = tr.build_models(tiny_cfg, torch.device("cpu"))
    opt_g, opt_d = tr.build_optimizers(tiny_cfg, G, D)
    g0 = [p.detach().clone() for p in G.parameters()]
    d0 = [p.detach().clone() for p in D.parameters()]
    parts = tr.train_step(G, D, opt_g, opt_d, None, None, torch.randn(2, 3, 128, 128),
                          torch.randn(2, 3, 128, 128), amp=False, lambda_l1=100.0, d_loss_scale=0.5)
    assert set(parts) == {"d_loss", "d_real", "d_fake", "g_loss", "g_adv", "g_l1"}
    assert all(v.dim() == 0 and torch.isfinite(v) and not v.requires_grad for v in parts.values())
    assert any(not torch.equal(a, b) for a, b in zip(g0, G.parameters()))
    assert any(not torch.equal(a, b) for a, b in zip(d0, D.parameters()))
''', baru=False)


In [ ]:
#@title tests/test_train.py · 5/9: test_d_step_does_not_touch_g
tulis('tests/test_train.py', r'''

def test_d_step_does_not_touch_g(tiny_cfg):
    """Isolasi langkah 1: setelah update D, gradien G harus tetap kosong."""
    G, D = tr.build_models(tiny_cfg, torch.device("cpu"))
    _, opt_d = tr.build_optimizers(tiny_cfg, G, D)
    calls = {"n": 0}
    real_step = tr._backward_and_step
    def stop_after_d(loss, opt, scaler):
        real_step(loss, opt, scaler)
        calls["n"] += 1
        if calls["n"] == 1:
            assert all(p.grad is None for p in G.parameters())
            raise StopIteration
    tr._backward_and_step, saved = stop_after_d, tr._backward_and_step
    try:
        with pytest.raises(StopIteration):
            tr.train_step(G, D, None, opt_d, None, None, torch.randn(1, 3, 128, 128),
                          torch.randn(1, 3, 128, 128), amp=False, lambda_l1=100.0, d_loss_scale=0.5)
    finally:
        tr._backward_and_step = saved
    assert all(p.requires_grad for p in D.parameters())          # finally memulihkan D
''', baru=False)


In [ ]:
#@title tests/test_train.py · 6/9: test_evaluate_and_samples_restore_train_mode
tulis('tests/test_train.py', r'''

# ---------------------------------------------------------------------------
# 3. Mode train/eval, shape, rentang
# ---------------------------------------------------------------------------
def test_evaluate_and_samples_restore_train_mode(tiny_cfg, tmp_path):
    from dataset import get_dataloader
    G, _ = tr.build_models(tiny_cfg, torch.device("cpu"))
    G.train()
    val_l1 = tr.evaluate_val(G, get_dataloader(tiny_cfg, "val"), torch.device("cpu"))
    assert G.training and 0.0 <= val_l1 <= 2.0          # |fake - real| <= 2 di rentang [-1, 1]

    a = torch.rand(3, 3, 128, 128) * 2 - 1
    path = tr.save_sample_grid(G, a, a, tmp_path / "grid.png")
    assert G.training
    with Image.open(path) as im:
        assert im.size == (3 * 128, 3 * 128) and im.mode == "RGB"   # 3 kolom A|G(A)|B, 3 baris


def test_generator_output_range_inside_train_step(tiny_cfg):
    G, D = tr.build_models(tiny_cfg, torch.device("cpu"))
    seen = []
    G.register_forward_hook(lambda m, i, o: seen.append(o.detach()))
    opt_g, opt_d = tr.build_optimizers(tiny_cfg, G, D)
    tr.train_step(G, D, opt_g, opt_d, None, None, torch.randn(2, 3, 128, 128) * 5,
                  torch.rand(2, 3, 128, 128) * 2 - 1, amp=False, lambda_l1=100.0, d_loss_scale=0.5)
    assert seen[0].shape == (2, 3, 128, 128)
    assert seen[0].min() >= -1 and seen[0].max() <= 1
''', baru=False)


In [ ]:
#@title tests/test_train.py · 7/9: test_full_tiny_run_writes_everything
tulis('tests/test_train.py', r'''

# ---------------------------------------------------------------------------
# 4. Run kecil end-to-end, file output, resume
# ---------------------------------------------------------------------------
def test_full_tiny_run_writes_everything(tiny_cfg):
    summary = tr.train(tiny_cfg)
    out = tr.run_dir(tiny_cfg)
    # 12 pasang -> train 10 (round 9.6), batch 2 -> 5 iterasi/epoch, 2 epoch
    assert summary["last_epoch"] == 2 and summary["steps"] == 10
    for name in ("config.json", "last.pt", "best.pt", "log_train.csv", "log_val.csv",
                 "samples/epoch_001.png", "samples/epoch_002.png"):
        assert (out / name).exists(), name
    train_rows, val_rows = read_csv(out / "log_train.csv"), read_csv(out / "log_val.csv")
    assert len(train_rows) == 10 and [r["epoch"] for r in val_rows] == ["1", "2"]
    assert all(np.isfinite(float(r["g_l1"])) for r in train_rows)
    ckpt = tr.load_checkpoint(out / "last.pt", torch.device("cpu"))
    assert ckpt["epoch"] == 2 and ckpt["step"] == 10
    assert ckpt["config"]["ngf"] == 8


def test_log_rows_are_window_averages(tiny_cfg, monkeypatch):
    """Setiap baris log = rata-rata log_every iterasi; sisa di akhir ikut dicatat."""
    counter = {"i": 0}

    def fake_step(*args, **kwargs):
        counter["i"] += 1
        v = torch.tensor(float(counter["i"]))      # nilai loss = nomor iterasi
        return {k: v for k in ("d_loss", "d_real", "d_fake", "g_loss", "g_adv", "g_l1")}

    monkeypatch.setattr(tr, "train_step", fake_step)
    tr.train(dataclasses.replace(tiny_cfg, log_every=3))       # 10 iterasi total
    rows = read_csv(tr.run_dir(tiny_cfg) / "log_train.csv")
    assert [int(r["step"]) for r in rows] == [3, 6, 9, 10]
    assert [int(r["n_iter"]) for r in rows] == [3, 3, 3, 1]
    assert [float(r["g_l1"]) for r in rows] == [2.0, 5.0, 8.0, 10.0]   # (1+2+3)/3, (4+5+6)/3, ...
    assert [r["epoch"] for r in rows] == ["1", "2", "2", "2"]          # jendela boleh melintasi epoch
''', baru=False)


In [ ]:
#@title tests/test_train.py · 8/9: test_resume_continues_epochs
tulis('tests/test_train.py', r'''

def test_resume_continues_epochs(tiny_cfg):
    tr.train(dataclasses.replace(tiny_cfg, epochs=1))
    summary = tr.train(dataclasses.replace(tiny_cfg, epochs=2), resume=True)
    assert summary["last_epoch"] == 2 and summary["steps"] == 10
    assert [r["epoch"] for r in read_csv(tr.run_dir(tiny_cfg) / "log_val.csv")] == ["1", "2"]


def test_resume_rejects_changed_architecture(tiny_cfg):
    tr.train(dataclasses.replace(tiny_cfg, epochs=1))
    with pytest.raises(ValueError, match="ngf"):
        tr.train(dataclasses.replace(tiny_cfg, epochs=2, ngf=16), resume=True)


def test_resume_without_checkpoint_fails(tiny_cfg):
    with pytest.raises(FileNotFoundError):
        tr.train(tiny_cfg, resume=True)


def test_overfit_tiny_subset_reduces_l1(tiny_cfg):
    """Sanity check: 2 pasangan tanpa augmentasi, L1 harus turun jelas."""
    # 150 iterasi: rasio akhir/awal sekitar 0,41 di seed 0, 1, 2 (diukur), jadi
    # ambang 0,6 punya margin lebar dan tidak rapuh terhadap seed.
    cfg = dataclasses.replace(tiny_cfg, subset_size=2, epochs=150, batch_size=2, lr=1e-3,
                              use_dropout=False, log_every=1)
    tr.train(cfg)
    rows = read_csv(tr.run_dir(cfg) / "log_train.csv")
    first = np.mean([float(r["g_l1"]) for r in rows[:5]])
    last = np.mean([float(r["g_l1"]) for r in rows[-5:]])
    assert last < 0.6 * first, (first, last)
''', baru=False)


In [ ]:
#@title tests/test_train.py · 9/9: test_parse_overrides_types
tulis('tests/test_train.py', r'''

# ---------------------------------------------------------------------------
# 5. CLI
# ---------------------------------------------------------------------------
def test_parse_overrides_types():
    got = tr.parse_overrides(["epochs=5", "use_amp=false", "lambda_l1=10.5",
                              "data_root=C:/Users/x y/base", "subset_size=null",
                              "split_ratios=[0.7, 0.2, 0.1]"])
    assert got == {"epochs": 5, "use_amp": False, "lambda_l1": 10.5,
                   "data_root": "C:/Users/x y/base", "subset_size": None,
                   "split_ratios": (0.7, 0.2, 0.1)}
    with pytest.raises(ValueError, match="tidak dikenal"):
        tr.parse_overrides(["lamda_l1=5"])
    with pytest.raises(ValueError, match="key=value"):
        tr.parse_overrides(["epochs"])


def test_main_cli_preset_and_max_steps(tiny_cfg):
    sets = [f"{k}={v}" for k, v in {
        "data_root": tiny_cfg.data_root, "img_size": 128, "load_size": 144, "num_workers": 0,
        "pin_memory": "false", "manifest_path": tiny_cfg.manifest_path, "ngf": 8, "ndf": 8,
        "device": "cpu", "checkpoint_dir": tiny_cfg.checkpoint_dir}.items()]
    argv = ["--preset", "noskip", "--max-steps", "3"]
    for s in sets:
        argv += ["--set", s]
    summary = tr.main(argv)
    assert summary["steps"] == 3 and summary["run_dir"].endswith("noskip")
    saved = tr.load_checkpoint(tr.run_dir(dataclasses.replace(tiny_cfg, run_name="noskip")) / "last.pt",
                               torch.device("cpu"))
    assert saved["config"]["use_skip"] is False and saved["config"]["epochs"] == 30
''', baru=False)


### `tests/test_eval.py`

Tes evaluasi: metrik vs hitungan tangan, dropout saat eval, perbandingan run. 6 sel, 179 baris. Isi sel → 1: `fixed_seed` · 2: `test_l1_manual` · 3: `test_ssim_rejects_tiny_image` · 4: `trained_runs` · 5: `test_main_compares_runs_and_reports_config` · 6: `test_data_root_override_after_moving_data`

In [ ]:
#@title tests/test_eval.py · 1/6: fixed_seed
tulis('tests/test_eval.py', r'''"""
tests/test_eval.py

Tes untuk eval.py: metrik dibandingkan dengan hitungan tangan, lalu evaluasi
end-to-end pada run kecil di CPU.
Jalankan: pytest tests/test_eval.py -v
"""

import csv
import dataclasses
import math
import shutil

import pytest
import torch
import torch.nn as nn

import eval as ev
import train as tr
from config import Config
from tests.helpers import make_pairs


@pytest.fixture(autouse=True)
def fixed_seed():
    torch.manual_seed(0)


def const(v, size=32):
    """Gambar [1, 3, size, size] bernilai konstan v (rentang [-1, 1])."""
    return torch.full((1, 3, size, size), float(v))
''', baru=True)


In [ ]:
#@title tests/test_eval.py · 2/6: test_l1_manual
tulis('tests/test_eval.py', r'''

# ---------------------------------------------------------------------------
# 1. Metrik vs hitungan tangan
# ---------------------------------------------------------------------------
def test_l1_manual():
    assert ev.l1_per_image(const(0.2), const(0.0)).item() == pytest.approx(0.2)


def test_psnr_manual_and_identical():
    # 0.2 dan 0.0 di [-1, 1] -> 0.6 dan 0.5 di [0, 1] -> MSE 0.01 -> 10*log10(1/0.01) = 20 dB
    assert ev.psnr_per_image(const(0.2), const(0.0)).item() == pytest.approx(20.0, abs=1e-4)
    x = torch.rand(2, 3, 32, 32) * 2 - 1
    assert torch.isinf(ev.psnr_per_image(x, x)).all()


def test_ssim_identical_is_one():
    x = torch.rand(2, 3, 32, 32) * 2 - 1
    assert torch.allclose(ev.ssim_per_image(x, x), torch.ones(2), atol=1e-5)


@pytest.mark.parametrize("va,vb", [(0.2, 0.0), (-0.6, 0.4), (1.0, -1.0)])
def test_ssim_constant_images_closed_form(va, vb):
    """Dua gambar konstan: varians dan kovarians 0, jadi
    SSIM = (2ab + C1) / (a^2 + b^2 + C1), dengan a, b di [0, 1]."""
    a, b = (va + 1) / 2, (vb + 1) / 2
    c1 = 0.01 ** 2
    expected = (2 * a * b + c1) / (a * a + b * b + c1)
    assert ev.ssim_per_image(const(va), const(vb)).item() == pytest.approx(expected, rel=1e-4)


def test_ssim_symmetric_bounded_and_monotone_in_noise():
    x = torch.rand(1, 3, 64, 64) * 2 - 1
    y = torch.rand(1, 3, 64, 64) * 2 - 1
    assert ev.ssim_per_image(x, y).item() == pytest.approx(ev.ssim_per_image(y, x).item(), abs=1e-6)
    assert -1.0 <= ev.ssim_per_image(x, y).item() < 0.2           # dua noise acak: hampir tak mirip
    scores = [ev.ssim_per_image((x + s * torch.randn_like(x)).clamp(-1, 1), x).item()
              for s in (0.01, 0.1, 0.5)]
    assert scores[0] > scores[1] > scores[2]                      # makin berisik makin rendah
''', baru=False)


In [ ]:
#@title tests/test_eval.py · 3/6: test_ssim_rejects_tiny_image
tulis('tests/test_eval.py', r'''

def test_ssim_rejects_tiny_image():
    with pytest.raises(ValueError, match="jendela"):
        ev.ssim_per_image(torch.zeros(1, 3, 8, 8), torch.zeros(1, 3, 8, 8))


# ---------------------------------------------------------------------------
# 2. Mode dropout saat evaluasi
# ---------------------------------------------------------------------------
def test_enable_dropout_only_keeps_batchnorm_in_eval():
    G, _ = tr.build_models(Config(ngf=8, img_size=128, device="cpu"), torch.device("cpu"))
    G.eval()
    ev.enable_dropout_only(G)
    assert all(m.training for m in G.modules() if isinstance(m, nn.Dropout))
    assert not any(m.training for m in G.modules() if isinstance(m, nn.BatchNorm2d))
''', baru=False)


In [ ]:
#@title tests/test_eval.py · 4/6: trained_runs
tulis('tests/test_eval.py', r'''

# ---------------------------------------------------------------------------
# 3. End-to-end pada run kecil
# ---------------------------------------------------------------------------
@pytest.fixture
def trained_runs(tmp_path):
    make_pairs(tmp_path / "data", n=20)           # split 16 / 2 / 2
    base = Config(data_root=str(tmp_path / "data"), img_size=128, load_size=144, batch_size=2,
                  num_workers=0, pin_memory=False, manifest_path=str(tmp_path / "manifest.json"),
                  ngf=8, ndf=8, epochs=1, log_every=4, device="cpu",
                  checkpoint_dir=str(tmp_path / "runs"))
    runs = []
    for name, over in (("main", {}), ("noskip", {"use_skip": False})):
        cfg = dataclasses.replace(base, run_name=name, **over)
        tr.train(cfg)
        runs.append(tr.run_dir(cfg))
    return tmp_path, runs


def test_evaluate_run_outputs(trained_runs):
    _, runs = trained_runs
    s = ev.evaluate_run(runs[0], device=torch.device("cpu"))
    assert s["n"] == 2 and s["split"] == "test" and s["epoch"] == 1
    assert 0.0 <= s["l1_mean"] <= 2.0 and math.isfinite(s["psnr_mean"]) and -1 <= s["ssim_mean"] <= 1
    with open(runs[0] / "eval_test_best.csv", newline="") as f:
        assert len(list(csv.DictReader(f))) == 2
    from PIL import Image
    with Image.open(runs[0] / "eval_test_best_grid.png") as im:
        assert im.size == (3 * 128, 2 * 128)          # 2 gambar test, 3 kolom


def test_best_and_last_written_separately(trained_runs):
    _, runs = trained_runs
    ev.evaluate_run(runs[0], ckpt="best", device=torch.device("cpu"))
    ev.evaluate_run(runs[0], ckpt="last", device=torch.device("cpu"))
    for name in ("eval_test_best.csv", "eval_test_last.csv",
                 "eval_test_best_grid.png", "eval_test_last_grid.png"):
        assert (runs[0] / name).exists(), name
''', baru=False)


In [ ]:
#@title tests/test_eval.py · 5/6: test_main_compares_runs_and_reports_config
tulis('tests/test_eval.py', r'''

def test_main_compares_runs_and_reports_config(trained_runs):
    tmp_path, runs = trained_runs
    out = tmp_path / "perbandingan.csv"
    summaries = ev.main(["--runs", *map(str, runs), "--device", "cpu", "--out", str(out)])
    with open(out, newline="") as f:
        rows = list(csv.DictReader(f))
    assert [r["run"] for r in rows] == ["main", "noskip"]
    assert [r["use_skip"] for r in rows] == ["True", "False"]
    assert summaries[0]["l1_mean"] != summaries[1]["l1_mean"]


def test_dropout_eval_is_active_and_reproducible(trained_runs, monkeypatch):
    """Catatan: di model mini yang baru dilatih beberapa iterasi, efek dropout ke
    output hanya ~1e-7 (jalur bottleneck nyaris nol, skip connection dominan).
    Karena itu yang diuji adalah modul Dropout-nya sendiri, bukan selisih L1."""
    _, runs = trained_runs
    seen = []
    original = nn.Dropout.forward

    def spy(self, x):
        out = original(self, x)
        seen.append((self.training, (out == 0).float().mean().item()))
        return out

    monkeypatch.setattr(nn.Dropout, "forward", spy)
    a = ev.evaluate_run(runs[0], dropout_eval=True, device=torch.device("cpu"))
    active = list(seen)
    seen.clear()
    ev.evaluate_run(runs[0], dropout_eval=False, device=torch.device("cpu"))
    inactive = list(seen)
    b = ev.evaluate_run(runs[0], dropout_eval=True, device=torch.device("cpu"))

    assert active and all(training for training, _ in active)
    assert sum(z for _, z in active) / len(active) > 0.3          # sekitar p = 0.5 nilai dinolkan
    assert inactive and not any(training for training, _ in inactive)
    assert a["l1_mean"] == b["l1_mean"]                           # seed tetap: bisa diulang
''', baru=False)


In [ ]:
#@title tests/test_eval.py · 6/6: test_data_root_override_after_moving_data
tulis('tests/test_eval.py', r'''

def test_data_root_override_after_moving_data(trained_runs):
    """Checkpoint dari Colab menyimpan path Colab; --data-root menggantinya."""
    tmp_path, runs = trained_runs
    shutil.move(str(tmp_path / "data"), str(tmp_path / "data_pindah"))
    with pytest.raises(FileNotFoundError):
        ev.evaluate_run(runs[0], device=torch.device("cpu"))
    s = ev.evaluate_run(runs[0], device=torch.device("cpu"),
                        overrides={"data_root": str(tmp_path / "data_pindah")})
    assert s["n"] == 2


def test_missing_checkpoint(tmp_path):
    with pytest.raises(FileNotFoundError, match="Checkpoint"):
        ev.evaluate_run(tmp_path / "tidak_ada", device=torch.device("cpu"))
''', baru=False)


### Verifikasi isi file

In [ ]:
import hashlib
EXPECTED = {
    "pytest.ini": "350a8b5dfbacea0ec61149282e2c2a5b21e973b0e0d07b1196de0bb579837a59",
    "tests/__init__.py": "f3f3d5fab2a5a4a57e00262cd8885eaf72dd8a7a3012792a35f3d22adefe0a75",
    "tests/helpers.py": "cc1babc92d5f565e6e710d1fe6c7e669b134e152dc6fbd504ff5ecb4079b42f8",
    "scripts/__init__.py": "c623bdb75f22e2e9d996acb9ae125c233a5cc99c04497289f770561765eb1971",
    "models/__init__.py": "15a962d8051c021ac7d1df7b2186c35b44e8918189126b598c4e54e8c893cd17",
    "models/generator.py": "7cee1a0b6a4f25f5e6dd0b4c228313df71e922aeb4b68192ea159f6ccb862937",
    "models/discriminator.py": "0a81f46ec70805c7aa0e9b9b92bca654bccf163703e26a296878ef12017cf8cb",
    "dataset.py": "659b15a5fe21fb462fe1a579a7e2107d7b9069efa17b607d2c6dd7a1bc55c2d9",
    "config.py": "08229ca0ac160a50dee482bcd3996f3676d964df9b875f0a536878ac0300d01b",
    "losses.py": "bc5991e97e11536189d64e74fdd9f430517254fbcf191f87b1f373fad3b2bd8f",
    "train.py": "b81190fde870fac3d300c043e62074d7861d1e60f38f0ddeea760338cd22bef5",
    "eval.py": "72c42daf731627e007323305dd996f375b74dca757cdef4f6448c7427e803b40",
    "scripts/preview_pairs.py": "869509ccce55e5ec116489f9f8109f6447961df5c2378b1e968b0ec64920204d",
    "scripts/measure_vram.py": "f0211c932434109ed76352c0c4ec0bc540998c2d97feaf7c12d51a0c0e3ea506",
    "scripts/smoke_real_data.py": "e69477b995afb863bbdb400cdfbaeab5fc914d910579e3474b91aa9300ecb7cb",
    "tests/test_generator.py": "7ffbe5f3120e7e1d999bdf91afa0c21d1b49712aaeb9cb98a13238e23b0f0de6",
    "tests/test_discriminator.py": "cdcd5cc270698f633528f56afde9cf30ff532901ea05c05a03e277e759f4f591",
    "tests/test_dataset.py": "729d18ec8cfb9e16bb411ce39e6e6f89c3a62bb423d2a693ae99c106c5b8e7a7",
    "tests/test_config.py": "8a552577247f94a667509cae5772b5b110a08eb573d925f6308f4e2b362d52d4",
    "tests/test_measure_vram.py": "7789b90dae2b9d101da0bd31cacff46586688b446a7051a0385a46ef2f125851",
    "tests/test_losses.py": "9dc5165e65bd8cb272035141bfc65b52d8fe8f82d64f178d77ee5f2ddd1645fb",
    "tests/test_train.py": "c1729d258be0fb4f8837d1cc155f3318975ac3e69f1ca952894b0ffb361d103a",
    "tests/test_eval.py": "89e6c8b4229be153ab4a3bf154f6a4e6dcd2965251aa5b771fcd632b08ef8233"
}




problems = []
for rel, h in EXPECTED.items():
    text = (REPO / rel).read_text(encoding="utf-8")
    if hashlib.sha256(text.encode("utf-8")).hexdigest() == h:
        status = "identik"
    else:
        status = "BEDA ISI"
        problems.append(rel)
    print(f"{status:10s} {rel}")
if problems:
    raise RuntimeError(f"Isi file tidak sama dengan repo: {problems}. Jalankan ulang bagian 1 dari awal.")
print("Semua file identik dengan repo.")


## 2. Semua tes

Di workspace saya (CPU): 257 lulus, 1 dilewati (tes autocast CUDA), juga ketika ada paket lain bernama `tests`/`scripts` di path seperti di Colab. Di T4 yang diharapkan: 257 lulus, 1 dilewati (tes khusus mesin tanpa GPU).

In [ ]:
if JALANKAN_TES:
    rc = sh(f"{sys.executable} -m pytest -q", log="pytest_semua.txt")
    if rc != 0:
        print("\n" + "!" * 70 + "\nPERINGATAN: ada tes yang GAGAL atau ERROR. Baca pesan di atas sebelum lanjut.\n" + "!" * 70)
else:
    print("Tes DILEWATI (JALANKAN_TES=False di Pengaturan).")


## 3. Dataset CMP Facades (base)

Diunduh langsung dari [halaman resmi CMP Facade Database](https://cmp.felk.cvut.cz/~tylecr1/facade/): base dataset, 378 gambar. Lisensi: Creative Commons **Attribution-ShareAlike** (badge di halaman: versi 3.0). Sitasi: R. Tyleček dan R. Šára, *Spatial Pattern Templates for Recognition of Objects with Regular Structure*, Proc. GCPR 2013.

In [ ]:
import urllib.request, zipfile

BASE_URL = "https://cmp.felk.cvut.cz/~tylecr1/facade/CMP_facade_DB_base.zip"
zip_path = Path("/content/data/CMP_facade_DB_base.zip")
zip_path.parent.mkdir(parents=True, exist_ok=True)

for attempt in (1, 2, 3):
    try:
        if not zip_path.exists():
            print(f"mengunduh (percobaan {attempt}):", BASE_URL)
            urllib.request.urlretrieve(BASE_URL, zip_path)
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall("/content/data/CMP_facade_DB_base")
        break
    except Exception as e:
        print("gagal:", type(e).__name__, e)
        zip_path.unlink(missing_ok=True)
else:
    raise RuntimeError("Unduhan dataset gagal 3 kali. Cek apakah situs CMP bisa dibuka, lalu jalankan ulang sel ini.")
print(f"zip {zip_path.stat().st_size / 2**20:.1f} MiB, diekstrak ke /content/data/CMP_facade_DB_base")


In [ ]:
folders = sorted(p for p in Path("/content/data").rglob("*") if p.is_dir() and any(p.glob("cmp_b*.jpg")))
assert folders, "Folder berisi cmp_b*.jpg tidak ditemukan setelah ekstrak."
DATA_ROOT = str(folders[0])
jpg, png, xml = (len(list(Path(DATA_ROOT).glob(f"*.{e}"))) for e in ("jpg", "png", "xml"))
print("DATA_ROOT =", DATA_ROOT)
print(f"jpg={jpg} png={png} xml={xml}   (data di laptop kalian: 378 / 378 / 378)")
if (jpg, png) != (378, 378):
    print("PERINGATAN: jumlah berbeda dari data di laptop; split tidak akan sama persis.")


## 4. Grid pratinjau A|B (dataset lama)

Kolom kiri harus **peta label berwarna**, kolom kanan **foto**. Pada 100 pasangan pertama di workspace saya, `AtoB` sudah benar. Manifest split ditulis ke folder data di Colab, bukan ke laptop kalian.

In [ ]:
sh(f'{sys.executable} scripts/preview_pairs.py --root "{DATA_ROOT}" --split val --direction AtoB --out cek_AtoB.png', log="preview_val.txt")
sh(f'{sys.executable} scripts/preview_pairs.py --root "{DATA_ROOT}" --split train --direction AtoB --augment --out cek_AtoB_augment.png', log="preview_train.txt")
show(REPO / "cek_AtoB.png", REPO / "cek_AtoB_augment.png")


## 5. Smoke test dataset lama

Yang diharapkan: split **302 / 38 / 38**, batch pertama `(2, 3, 256, 256)` di [-1, 1]. Waktu = CPU Colab, bukan laptop.

In [ ]:
sh(f'{sys.executable} scripts/smoke_real_data.py --module dataset --root "{DATA_ROOT}" --out cek_lama.png', log="smoke_lama.txt")
show(REPO / "cek_lama.png")


## 6. Perbandingan dengan `dataset_baru.py`

Dijalankan hanya jika `/content/dataset_baru.py` dan `/content/test_dataset_baru.py` ada. `dataset.py` dan `config.py` tidak diubah. Di `tests/test_dataset_baru.py` hanya baris import `dataset` yang diganti menjadi `dataset_baru`.

In [ ]:
import shutil
SRC_BARU = Path("/content/dataset_baru.py")
SRC_TES_BARU = Path("/content/test_dataset_baru.py")
ADA_BARU = SRC_BARU.exists() and SRC_TES_BARU.exists()
if ADA_BARU:
    shutil.copy(SRC_BARU, REPO / "dataset_baru.py")
    shutil.copy(SRC_TES_BARU, REPO / "tests" / "test_dataset_baru.py")
    print("dataset_baru.py dan test_dataset_baru.py disalin ke repo.")
else:
    print("Bagian 6 DILEWATI: file tidak lengkap di /content",
          {"dataset_baru.py": SRC_BARU.exists(), "test_dataset_baru.py": SRC_TES_BARU.exists()})


In [ ]:
# 6.2 Ganti HANYA baris import dataset -> dataset_baru (setiap perubahan dicetak)
import re
if ADA_BARU:
    p = REPO / "tests" / "test_dataset_baru.py"
    lines = p.read_text(encoding="utf-8").splitlines(keepends=True)
    changed = []
    for i, line in enumerate(lines):
        new = re.sub(r"^(\s*)from dataset import\b", r"\1from dataset_baru import", line)
        new = re.sub(r"^(\s*)import dataset(\s|$)", r"\1import dataset_baru\2", new)
        if new != line:
            changed.append(f"baris {i + 1}: {line.rstrip()!r} -> {new.rstrip()!r}")
            lines[i] = new
    p.write_text("".join(lines), encoding="utf-8")
    print("\n".join(changed) if changed else "Tidak ada baris import yang perlu diganti.")
    (HASIL / "ganti_import.txt").write_text("\n".join(changed), encoding="utf-8")


In [ ]:
# 6.3 Tes dataset lama dan tes dataset baru
if ADA_BARU:
    print("=== tests/test_dataset.py (lama) ===")
    sh(f"{sys.executable} -m pytest -q tests/test_dataset.py", log="pytest_dataset_lama.txt")
    print("=== tests/test_dataset_baru.py (baru) ===")
    sh(f"{sys.executable} -m pytest -q tests/test_dataset_baru.py", log="pytest_dataset_baru.txt")


In [ ]:
# 6.4 Tes integrasi test_config terhadap dataset_baru, di SALINAN repo
if ADA_BARU:
    tmp = Path("/tmp/repo_baru")
    shutil.rmtree(tmp, ignore_errors=True)
    shutil.copytree(REPO, tmp)
    shutil.copy(REPO / "dataset_baru.py", tmp / "dataset.py")   # hanya di salinan
    sh(f"{sys.executable} -m pytest -q tests/test_config.py -k integration", log="pytest_integrasi_baru.txt", cwd=tmp)


In [ ]:
# 6.5 Smoke test dataset baru pada data asli
if ADA_BARU:
    sh(f'{sys.executable} scripts/smoke_real_data.py --module dataset_baru --root "{DATA_ROOT}" --out cek_baru.png', log="smoke_baru.txt")
    show(REPO / "cek_lama.png", REPO / "cek_baru.png")


In [ ]:
# 6.6 Fakta otomatis untuk tabel perbedaan: signature, persistent_workers, format, pickle, spawn
import dataclasses, importlib, inspect, pickle, tempfile
import numpy as np, pandas as pd
from PIL import Image as PILImage
from IPython.display import display

def make_combined(root, n=10):
    root.mkdir(parents=True, exist_ok=True); rng = np.random.default_rng(0)
    for i in range(n):
        PILImage.fromarray(rng.integers(0, 256, (64, 128, 3), dtype=np.uint8)).save(root / f"{i:03d}.png")

def make_separate(root, n=10):
    root.mkdir(parents=True, exist_ok=True); rng = np.random.default_rng(1)
    palette = [0, 0, 0] + [c for k in range(1, 13) for c in (k * 20, 255 - k * 20, 128)]
    for i in range(n):
        idx = np.kron(rng.integers(1, 13, (4, 4)), np.ones((16, 16))).astype(np.uint8)
        lab = PILImage.fromarray(idx, mode="P"); lab.putpalette(palette); lab.save(root / f"cmp_b{i:04d}.png")
        PILImage.fromarray(rng.integers(0, 256, (64, 64, 3), dtype=np.uint8)).save(root / f"cmp_b{i:04d}.jpg")


In [ ]:
def probe(name):
    mod = importlib.reload(importlib.import_module(name))
    src = inspect.getsource(mod)
    sig = lambda n: str(inspect.signature(getattr(mod, n))) if hasattr(mod, n) else "(tidak ada)"
    info = {"PairedImageDataset(...)": sig("PairedImageDataset"), "get_dataloader(...)": sig("get_dataloader"),
            "find_split_overlap(...)": sig("find_split_overlap"),
            "baris persistent_workers": "; ".join(l.strip() for l in src.splitlines() if "persistent_workers" in l) or "(tidak ada)",
            "menyebut NEAREST": "NEAREST" in src}
    for fmt, maker in (("format berdampingan", make_combined), ("format terpisah png+jpg", make_separate)):
        with tempfile.TemporaryDirectory() as d:
            root = Path(d) / "data"; maker(root)
            try:
                cfg = dataclasses.replace(Config(), data_root=str(root), img_size=128, load_size=144,
                                          num_workers=0, pin_memory=False, manifest_path=str(Path(d) / "m.json"))
                ds = mod.get_dataloader(cfg, "train").dataset
                info[fmt] = f"OK, {len(ds)} train, A{tuple(ds[0]['A'].shape)}, label_mode={getattr(ds, 'label_mode', '-')}"
                if fmt.startswith("format terpisah"):
                    pickle.loads(pickle.dumps(ds)); info["pickle"] = "OK"
                    dl = torch.utils.data.DataLoader(ds, batch_size=2, num_workers=2, multiprocessing_context="spawn")
                    info["DataLoader spawn, 2 worker"] = f"OK, A{tuple(next(iter(dl))['A'].shape)}"
            except Exception as e:
                info[fmt] = f"GAGAL: {type(e).__name__}: {e}"[:200]
    return info

if ADA_BARU:
    sys.path.insert(0, str(REPO))
    from config import Config
    table = pd.DataFrame({name: probe(name) for name in ("dataset", "dataset_baru")})
    table.to_csv(HASIL / "perbandingan_otomatis.csv")
    pd.set_option("display.max_colwidth", None); display(table)


## 7. Pengukuran VRAM

Skrip ini sekarang memakai **langkah training yang sama persis dengan `train.py`** (termasuk D dibekukan saat update G), dan menjalankan **pemanasan sampai state Adam G dan D terbentuk** sebelum iterasi terukur. Pengukuran sebelumnya meremehkan baris AMP karena state Adam belum ada. Kolom `adam_ready` harus `True` di semua baris; `warmup_iters` > 1 pada baris AMP berarti GradScaler sempat melewati step.

`steady_alloc_mib` = puncak khusus iterasi terukur. Jika jauh di bawah `peak_alloc_mib`, lonjakan memori hanya terjadi sekali di awal.

GPU Colab (T4, 16 GB) **bukan** RTX 4050 (sekitar 4 GB). `peak_alloc_mib` kurang lebih sebanding; `peak_reserved`, waktu, dan OOM tidak. Kolom `perkiraan_muat_4GB` memakai ambang **tebakan** 80% × 4096 MiB dan hanya bermakna jika `adam_ready` True.

In [ ]:
# 7.1 Sweep lengkap: img {256,128} x batch {1,2,4} x AMP {off,on} x ngf {64,32}
if UKUR_VRAM and torch.cuda.is_available():
    sh(f"{sys.executable} scripts/measure_vram.py --out {HASIL / 'vram_colab.csv'}", log="vram_colab.txt")
else:
    print("Bagian 7 DILEWATI:", "UKUR_VRAM=False" if not UKUR_VRAM else "tidak ada GPU")


In [ ]:
# 7.2 Uji ulang lonjakan 128 x batch 4 x AMP off: default cuDNN vs cuDNN deterministik
if UKUR_VRAM and torch.cuda.is_available():
    for name, flag in (("default", ""), ("deterministic", "--cudnn-deterministic")):
        sh(f"{sys.executable} scripts/measure_vram.py --img-sizes 128 --batches 4 --amp off "
           f"{flag} --out {HASIL / f'vram_128x4_{name}.csv'}", log=f"vram_128x4_{name}.txt")


In [ ]:
import pandas as pd
from IPython.display import display
pd.set_option("display.width", 200)
cols = ["img_size", "batch", "amp", "ngf", "status", "peak_alloc_mib", "peak_reserved_mib",
        "steady_alloc_mib", "iter_ms", "warmup_iters", "adam_ready", "g_steps", "perkiraan_muat_4GB"]
csv_path = HASIL / "vram_colab.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path)
    # TEBAKAN, bukan hasil ukur; hanya bermakna jika state Adam sudah terbentuk
    df["perkiraan_muat_4GB"] = (df["peak_reserved_mib"] < 0.8 * 4096) & df["adam_ready"].astype(bool)
    df.to_csv(csv_path, index=False)
    display(df[cols])
    print("Baris tanpa state Adam:", int((~df["adam_ready"].astype(bool)).sum()))
for name in ("default", "deterministic"):
    p = HASIL / f"vram_128x4_{name}.csv"
    if p.exists():
        print(f"\n128 x batch 4 x AMP off, cuDNN {name}:")
        display(pd.read_csv(p)[[c for c in cols if c != "perkiraan_muat_4GB"]])


## 8.1 Training run utama

Menjalankan `train.py` dengan config default (256, ngf 64, batch 2, tanpa AMP) selama `EPOCHS` epoch dari sel Pengaturan (100 = training penuh, sekitar 30–40 menit di T4). Progres tampil langsung per 50 iterasi dan per epoch.

Dengan `LANJUTKAN_RUN=True`, run yang sudah selesai di runtime ini **dilewati**, dan run yang terhenti **dilanjutkan** dari `last.pt`. Jadi *Run all* boleh diulang tanpa kehilangan hasil.

Yang masuk zip: `log_train.csv`, `log_val.csv`, `config.json`, grafik loss, dan grid sampel per epoch (A | G(A) | B dari 4 gambar val yang sama). **Checkpoint tidak ikut di-zip** karena besar (sekitar 700 MB per file); file itu tetap di `/content/runs/main` sampai runtime berakhir.

In [ ]:
rc = latih("main", f'--set data_root="{DATA_ROOT}" --set checkpoint_dir="{RUNS}" --set num_workers=2',
           EPOCHS, log="train_main.txt")
if rc != 0:
    print("\n" + "!" * 70 + f"\nPERINGATAN: training run utama berhenti dengan kode {rc}. Baca pesan di atas.\n" + "!" * 70)


In [ ]:
# Salin log + sampel ke folder hasil, lalu gambar grafik dari log yang ada
import shutil
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
run = RUNS / "main"
for name in ("log_train.csv", "log_val.csv", "config.json"):
    if (run / name).exists():
        shutil.copy(run / name, HASIL / f"train_{name}")
(HASIL / "samples").mkdir(exist_ok=True)
for png in sorted((run / "samples").glob("*.png")):
    shutil.copy(png, HASIL / "samples" / png.name)

tl = pd.read_csv(run / "log_train.csv") if (run / "log_train.csv").exists() else None
if tl is not None and "n_iter" in tl and len(tl) > 1:
    # jendela sisa di akhir training (n_iter kecil) tidak digambar karena sangat berisik
    tl = tl[tl["n_iter"] >= tl["n_iter"].max() / 2]
vl = pd.read_csv(run / "log_val.csv") if (run / "log_val.csv").exists() else None
if tl is None and vl is None:
    print("Training tidak menghasilkan log; lihat output sel sebelumnya.")
else:
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
    if tl is not None:
        ax[0].plot(tl["step"], tl["d_loss"]); ax[1].plot(tl["step"], tl["g_adv"])
        ax[2].plot(tl["step"], tl["g_l1"], label="train L1")
    else:
        print("log_train.csv belum ada (iterasi < log_every); hanya L1 validasi yang digambar.")
    if vl is not None:
        ax[2].plot(vl["step"], vl["val_l1"], "o-", label="val L1"); display(vl)
    for a, title in zip(ax, ("loss D", "loss G adversarial", "L1")):
        a.set_title(title); a.set_xlabel("step")
    ax[2].legend()
    fig.tight_layout(); fig.savefig(HASIL / "grafik_loss.png", dpi=110); plt.show()


In [ ]:
# Grid sampel epoch pertama dan terakhir (kiri ke kanan: A | G(A) | B)
samples = sorted((RUNS / "main" / "samples").glob("*.png"))
if samples:
    show(samples[0], samples[-1])


## 8.2 Ablation dan tuning hyperparameter

Menjalankan setiap preset di `ABLASI` (sel Pengaturan) selama `EPOCHS_ABLASI` epoch, dengan seed dan split yang sama. Preset `lambda100` = baseline dengan jumlah epoch yang sama, jadi pembanding yang adil untuk preset lain. Catatan batch size: dengan epoch yang sama, `batch4` mendapat setengah jumlah update dibanding batch 2 (kolom `step` di log).

In [ ]:
import shutil, sys
sys.path.insert(0, str(REPO))
from config import PRESETS
ABLASI_LIST = [n.strip() for n in ABLASI.split(",") if n.strip()]
for name in ABLASI_LIST:
    if name not in PRESETS:
        print(f"Preset {name!r} tidak dikenal, dilewati. Pilihan: {sorted(PRESETS)}")
        continue
    print(f"=== {name} ===")
    rc = latih(name, f'--preset {name} --set data_root="{DATA_ROOT}" --set checkpoint_dir="{RUNS}" '
                     f'--set num_workers=2', EPOCHS_ABLASI, log=f"train_{name}.txt")
    if rc != 0:
        print(f"PERINGATAN: preset {name} berhenti dengan kode {rc}; lanjut ke preset berikutnya.")
if not ABLASI_LIST:
    print("Tidak ada ablation (ABLASI kosong di Pengaturan).")


## 8.3 Evaluasi split test dan perbandingan run

`eval.py` menilai checkpoint **`best.pt` (val L1 terbaik) dan `last.pt` (epoch terakhir)** setiap run di `/content/runs` pada **split test** (38 gambar yang tidak dipakai saat training maupun saat memilih checkpoint). L1, PSNR, dan SSIM ditulis sendiri tanpa jaringan pretrained.

Ingat: metrik piksel menguntungkan gambar buram, dan model GAN awal yang buram sering punya val L1 terendah, sehingga `best.pt` bisa jatuh di epoch awal. Karena itu `best` dan `last` dievaluasi keduanya; bandingkan juga grid `eval_test_best_grid.png` dan `eval_test_last_grid.png` secara visual.

In [ ]:
import shutil
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
runs_done = sorted(d for d in RUNS.iterdir() if (d / "best.pt").exists()) if RUNS.exists() else []
if JALANKAN_EVAL and runs_done:
    run_args = " ".join(str(r) for r in runs_done)
    for ck in ("best", "last"):
        print(f"=== checkpoint {ck} ===")
        sh(f'{sys.executable} eval.py --runs {run_args} --ckpt {ck} --num-workers 2 '
           f'--out "{HASIL / f"perbandingan_test_{ck}.csv"}"', log=f"eval_{ck}.txt")
    tabel = pd.concat([pd.read_csv(HASIL / f"perbandingan_test_{ck}.csv") for ck in ("best", "last")
                       if (HASIL / f"perbandingan_test_{ck}.csv").exists()])
    display(tabel[["run", "ckpt", "epoch", "step", "n", "l1_mean", "psnr_mean", "ssim_mean", "best_val_l1",
                   "lambda_l1", "use_skip", "n_layers", "lr", "batch_size"]])
    for d in runs_done:   # simpan log, config, hasil evaluasi, dan sampel epoch terakhir setiap run
        dst = HASIL / "runs" / d.name
        dst.mkdir(parents=True, exist_ok=True)
        for name in ("log_train.csv", "log_val.csv", "config.json", "eval_test_best.csv", "eval_test_last.csv",
                     "eval_test_best_grid.png", "eval_test_last_grid.png"):
            if (d / name).exists():
                shutil.copy(d / name, dst / name)
        samples = sorted((d / "samples").glob("*.png"))
        if samples:
            shutil.copy(samples[-1], dst / f"sampel_{samples[-1].name}")
elif not JALANKAN_EVAL:
    print("Evaluasi DILEWATI (JALANKAN_EVAL=False).")
else:
    print("Belum ada run dengan best.pt di", RUNS)


In [ ]:
# Kurva val L1 semua run dalam satu grafik (sumbu x = epoch)
import matplotlib.pyplot as plt
import pandas as pd
kurva = [(d.name, pd.read_csv(d / "log_val.csv")) for d in runs_done if (d / "log_val.csv").exists()]
if kurva:
    fig, ax = plt.subplots(figsize=(8, 4))
    for name, vl in kurva:
        ax.plot(vl["epoch"], vl["val_l1"], marker="o", markersize=3, label=name)
    ax.set_xlabel("epoch"); ax.set_ylabel("val L1 (lebih kecil lebih baik)"); ax.set_title("Val L1 per run")
    ax.legend(); fig.tight_layout(); fig.savefig(HASIL / "kurva_val_l1.png", dpi=110); plt.show()
for d in runs_done[:3]:
    for ck in ("best", "last"):
        if (d / f"eval_test_{ck}_grid.png").exists():
            show(d / f"eval_test_{ck}_grid.png")


## 9. Ringkasan hasil

Mengumpulkan angka penting dari semua bagian ke satu tabel dan menyimpannya sebagai `ringkasan.md` (bahan laporan dan log interaksi).

In [ ]:
import re as _re
import pandas as pd
from IPython.display import display
def _baca(name):
    p = HASIL / name
    return p.read_text(encoding="utf-8") if p.exists() else ""

def _status_tes():
    teks = _re.sub(r"\x1b\[[0-9;]*m", "", _baca("pytest_semua.txt"))
    if not teks:
        return "tidak dijalankan"
    akhir = [l.strip("= ") for l in teks.splitlines() if _re.search(r"\b(passed|failed|errors?)\b", l)]
    hasil = akhir[-1] if akhir else "tidak terbaca"
    return ("GAGAL/ERROR: " if _re.search(r"\b(failed|errors?)\b", hasil) else "") + hasil

def _cari(pola, teks, default="-"):
    m = _re.search(pola, teks)
    return m.group(0) if m else default

baris = [("GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "tidak ada"),
         ("Tes pytest", _status_tes()),
         ("Dataset", "jpg={}, png={}, xml={} di {}".format(*(len(list(Path(DATA_ROOT).glob(f"*.{e}"))) for e in ("jpg", "png", "xml")), DATA_ROOT)),
         ("Split (smoke test)", "; ".join(l.split(",")[0] for l in _baca("smoke_lama.txt").splitlines() if "pasang" in l) or "-"),
         ("Batch pertama", _cari(r"batch pertama:[^\n]*", _baca("smoke_lama.txt")))]
vram = HASIL / "vram_colab.csv"
if vram.exists():
    v = pd.read_csv(vram)
    v256 = v[v.img_size == 256]
    baris.append(("VRAM 256 (maks reserved)", f"{v256.peak_reserved_mib.max():.0f} MiB, adam_ready semua: {bool(v.adam_ready.all())}"))
vl_path = HASIL / "train_log_val.csv"
if vl_path.exists():
    vl = pd.read_csv(vl_path)
    baris.append(("Training", f"{len(vl)} epoch, val L1 awal {vl.val_l1.iloc[0]:.4f}, akhir {vl.val_l1.iloc[-1]:.4f}, "
                              f"terbaik {vl.val_l1.min():.4f} (epoch {int(vl.loc[vl.val_l1.idxmin(), 'epoch'])}), "
                              f"{vl.epoch_sec.mean():.1f} dtk/epoch"))
for ck in ("best", "last"):
    pt = HASIL / f"perbandingan_test_{ck}.csv"
    if pt.exists():
        for r in pd.read_csv(pt).itertuples():
            baris.append((f"Test {r.run} ({ck})", f"epoch {r.epoch}, L1 {r.l1_mean:.4f}, PSNR {r.psnr_mean:.2f} dB, SSIM {r.ssim_mean:.4f} (n={r.n})"))
ringkasan = pd.DataFrame(baris, columns=["Hal", "Hasil"])
display(ringkasan)
try:
    teks = ringkasan.to_markdown(index=False)      # butuh paket tabulate
except ImportError:
    teks = ringkasan.to_string(index=False)
_ = (HASIL / "ringkasan.md").write_text(teks, encoding="utf-8")
print("disimpan:", HASIL / "ringkasan.md")


## 10. Kumpulkan hasil

Semua log, gambar, dan CSV di `/content/hasil` di-zip lalu diunduh otomatis.

In [ ]:
import shutil
archive = shutil.make_archive("/content/hasil_pix2pix", "zip", HASIL)
print("isi:", sorted(p.name for p in HASIL.iterdir()))
try:
    from google.colab import files
    files.download(archive)
except Exception as e:
    print("Unduh manual dari panel Files:", archive, f"({e})")
